# Neural machine translation (Aplications of Natural Language Processing)

# STUDENT ID: 05988721G

In order to complete the practice, the following pair of models has been chosen:

**Helsinki-NLP/opus-mt-en-es + Helsinki-NLP/opus-mt-es-en**

Moreover, the chosen corpues has been **'News Commentary'**

## Synthetic data generation

I've implemented a synthetic data generation method based on iterative back-translation. The main idea is to use monolingual corpora to create additional synthetic parallel sentence pairs that augment the small authentic bilingual training set. Concretely, the Spanish monolingual corpus is translated with the current Spanish to English model to generate synthetic English sentences, which are then paired with the original Spanish sentences for English to Spanish training; symmetrically, the English monolingual corpus is translated with the current English to Spanish model to generate synthetic Spanish sentences, which are paired with the original English sentences for Spanish to English training. This is implemented through functions that load the monolingual datasets, generate translations in batches from the saved checkpoints, build aligned synthetic sentence pairs, and concatenate them with the authentic parallel corpus before each new fine-tuning iteration. The same procedure is repeated across iterations so that synthetic data are regenerated with progressively updated models.

In greater detail, the synthetic data generation stage is organized as a controlled transformation pipeline that preserves sentence alignment and directional consistency at every step. The code first extracts and validates the monolingual texts from the Hugging Face datasets, rejecting empty inputs or blank segments, and then loads the opposite-direction fine-tuned checkpoint from the previous stage to produce translations in mini-batches under the same length constraints used elsewhere in the project. After generation, each synthetic sentence is paired strictly by position with its original monolingual counterpart, so the resulting pairs keep a one-to-one correspondence without altering the order of the source material. These validated synthetic pairs are then converted into translation datasets with the appropriate language-code structure and used to rebuild the augmented training corpus for the new round. It is important to highlight that, from Iteration 2 onward, the project **does not accumulate synthetic** data from earlier rounds; instead, it regenerates a **fresh synthetic corpus** with the most recent opposite-direction model, which makes the procedure **genuinely iterative** rather than a simple one-time augmentation.

This method is implemented from Iteration 1 in Section 7

## Installation of the required libraries

In [1]:
!pip uninstall -y transformers accelerate tokenizers huggingface_hub datasets evaluate sacrebleu sacremoses gradio diffusers

!pip install --no-cache-dir \
    "huggingface_hub==0.26.2" \
    "transformers==4.46.3" \
    "tokenizers==0.20.3" \
    "datasets==3.1.0" \
    "evaluate==0.4.3" \
    "sacremoses==0.1.1" \
    "sacrebleu==2.4.3" \
    "accelerate==1.6.0" \
    "torch" \
    "sentencepiece"

Found existing installation: transformers 4.46.3
Uninstalling transformers-4.46.3:
  Successfully uninstalled transformers-4.46.3
Found existing installation: accelerate 1.6.0
Uninstalling accelerate-1.6.0:
  Successfully uninstalled accelerate-1.6.0
Found existing installation: tokenizers 0.20.3
Uninstalling tokenizers-0.20.3:
  Successfully uninstalled tokenizers-0.20.3
Found existing installation: huggingface-hub 0.26.2
Uninstalling huggingface-hub-0.26.2:
  Successfully uninstalled huggingface-hub-0.26.2
Found existing installation: datasets 3.1.0
Uninstalling datasets-3.1.0:
  Successfully uninstalled datasets-3.1.0
Found existing installation: evaluate 0.4.3
Uninstalling evaluate-0.4.3:
  Successfully uninstalled evaluate-0.4.3
Found existing installation: sacrebleu 2.4.3
Uninstalling sacrebleu-2.4.3:
  Successfully uninstalled sacrebleu-2.4.3
Found existing installation: sacremoses 0.1.1
Uninstalling sacremoses-0.1.1:
  Successfully uninstalled sacremoses-0.1.1
     ━━━━━━━━━━━━

## SECTION 1

This first section presents corpus preprocessing and cleaning as the foundational stage of the experiment, whose purpose is to ensure that every later dataset is extracted from a reliable and structurally valid bilingual pool. This is implemented by first downloading and extracting the English–Spanish News Commentary corpus into the required Google Drive directory, then defining the raw and cleaned file paths, and finally recreating and executing the provided `preprocess-corpus.sh`script that was included in Moodle. The script performs the core cleaning operations expected in Moses-format parallel data: it removes duplicated sentence pairs, discards pairs with empty segments, normalizes whitespace, and preserves line-by-line alignment between both language files. In addition, this sectiion creates a wrapper of the original script inside a safer Python execution function that validates the inputs before execution, runs the preprocessing locally for efficiency, copies the cleaned files back to Drive, and verifies the output afterwards.


This cell implements the initial acquisition of the bilingual corpus required for preprocessing.

In [2]:
from google.colab import drive
from pathlib import Path
import subprocess

drive.mount("/content/drive", force_remount=True)

mydrive = Path("/content/drive/MyDrive/anlp")
mydrive.mkdir(parents=True, exist_ok=True)

zip_path = mydrive / "en-es.txt.zip"
download_url = "https://object.pouta.csc.fi/OPUS-News-Commentary/v11/moses/en-es.txt.zip"

print("Downloading:", download_url)
subprocess.run(["wget", "-O", str(zip_path), download_url], check=True)

print("\nListing ZIP contents:")
subprocess.run(["unzip", "-l", str(zip_path)], check=True)

print("\nExtracting ZIP into:", mydrive)
subprocess.run(["unzip", "-o", str(zip_path), "-d", str(mydrive)], check=True)

print("\nFiles now present in /content/drive/MyDrive/anlp:")
for p in sorted(mydrive.iterdir()):
    print(" -", p.name)

Mounted at /content/drive
Downloading: https://object.pouta.csc.fi/OPUS-News-Commentary/v11/moses/en-es.txt.zip

Listing ZIP contents:

Extracting ZIP into: /content/drive/MyDrive/anlp

Files now present in /content/drive/MyDrive/anlp:
 - LICENSE
 - News-Commentary.en-es.en
 - News-Commentary.en-es.es
 - News-Commentary.en-es.ids
 - README
 - en-es.txt.zip


This code defines the working paths and the core file-level configuration needed for preprocessing.

In [3]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive", force_remount=True)

mydrive = Path("/content/drive/MyDrive/anlp")
mydrive.mkdir(parents=True, exist_ok=True)

source = "en"
target = "es"

raw_source_filename = "News-Commentary.en-es.en"
raw_target_filename = "News-Commentary.en-es.es"

clean_prefix = "news_commentary_clean"
clean_source_filename = f"{clean_prefix}.{source}"
clean_target_filename = f"{clean_prefix}.{target}"

raw_source_path = mydrive / raw_source_filename
raw_target_path = mydrive / raw_target_filename

clean_source_path = mydrive / clean_source_filename
clean_target_path = mydrive / clean_target_filename

print("Raw source file :", raw_source_path)
print("Raw target file :", raw_target_path)
print("Clean source out:", clean_source_path)
print("Clean target out:", clean_target_path)

print("\nCurrent files in the anlp folder:")
for p in sorted(mydrive.iterdir()):
    print(" -", p.name)

assert raw_source_path.exists(), f"Raw source file not found: {raw_source_path}"
assert raw_target_path.exists(), f"Raw target file not found: {raw_target_path}"

Mounted at /content/drive
Raw source file : /content/drive/MyDrive/anlp/News-Commentary.en-es.en
Raw target file : /content/drive/MyDrive/anlp/News-Commentary.en-es.es
Clean source out: /content/drive/MyDrive/anlp/news_commentary_clean.en
Clean target out: /content/drive/MyDrive/anlp/news_commentary_clean.es

Current files in the anlp folder:
 - LICENSE
 - News-Commentary.en-es.en
 - News-Commentary.en-es.es
 - News-Commentary.en-es.ids
 - README
 - en-es.txt.zip


The following code reproduces the provided preprocess-corpus.sh script inside the project. The implementation writes the full shell script as a string into /content/preprocess-corpus.sh, preserves its original behavior, and grants execution permissions.

In [4]:
from pathlib import Path

script_path = Path("/content/preprocess-corpus.sh")

script_content = r"""#!/bin/bash

if [ $# -ne 4 ]; then
  echo "Error: Wrong number of arguments"
  echo "Usage: $0 <filein.sl> <filein.tl> <fileout.sl> <fileout.tl>"
  exit 1
fi

file_sl="$1"
file_tl="$2"

file_sl_out="$3"
file_tl_out="$4"

if [ ! -f "$file_sl" ]; then
  echo "Error: File '$file_sl' does not exist."
  exit 1
fi

if [ ! -f "$file_tl" ]; then
  echo "Error: File '$file_tl' does not exist."
  exit 1
fi

if [ -f "$file_sl_out" ]; then
  echo "Error: File '$file_sl_out' already exist."
  echo "Please remove it"
  exit 1
fi

if [ -f "$file_tl_out" ]; then
  echo "Error: File '$file_tl_out' already exist."
    echo "Please remove it"
  exit 1
fi

lines_sl=$(wc -l < "$file_sl")
lines_tl=$(wc -l < "$file_tl")

if [ "$lines_sl" -ne "$lines_tl" ]; then
  echo "Error: The files provide do not contain the same number of lines:"
  echo "   $file_sl: $lines_sl lines"
  echo "   $file_tl: $lines_tl lines"
fi

temp=$(mktemp -p "$PWD")

cat $file_sl | sed -re "s/^\s+//g" | sed -re "s/\s+$//g" | sed -re "s/\s+/ /g" > $temp"-sl"
cat $file_tl | sed -re "s/^\s+//g" | sed -re "s/\s+$//g" | sed -re "s/\s+/ /g" > $temp"-tl"

paste $temp"-sl" $temp"-tl" | sort | uniq |\
awk -F$'\t' '{if (($1!="")&&($2!="")) print}' | shuf > $temp-"sltl"

cut -f1 $temp-"sltl" > "$file_sl_out"
cut -f2 $temp-"sltl" > "$file_tl_out"

rm $temp-"sltl" $temp"-sl" $temp"-tl"
"""

script_path.write_text(script_content, encoding="utf-8")
script_path.chmod(0o755)

print(f"Script written to: {script_path}")

Script written to: /content/preprocess-corpus.sh


This cell introduces the Python helper functions that make preprocessing easier to validate. It defines utility functions to count and read lines, approximate the script’s whitespace normalization, validate the cleaned bilingual corpus, and execute the shell preprocessing script locally before copying the results back to Google Drive.

In [5]:
import re
import subprocess
from pathlib import Path

def count_lines(path):
    path = Path(path)
    with path.open("r", encoding="utf-8") as f:
        return sum(1 for _ in f)

def read_lines(path):
    path = Path(path)
    with path.open("r", encoding="utf-8") as f:
        return [line.rstrip("\n") for line in f]

def normalize_script_like_whitespace(text):
    return re.sub(r"\s+", " ", text.strip())

def validate_clean_parallel_corpus(source_path, target_path, verbose=True):
    source_path = Path(source_path)
    target_path = Path(target_path)

    if not source_path.exists():
        raise FileNotFoundError(f"Clean source file not found: {source_path}")
    if not target_path.exists():
        raise FileNotFoundError(f"Clean target file not found: {target_path}")

    source_lines = read_lines(source_path)
    target_lines = read_lines(target_path)

    if len(source_lines) != len(target_lines):
        raise ValueError(
            f"Line count mismatch after preprocessing: "
            f"{source_path} has {len(source_lines)} lines, "
            f"{target_path} has {len(target_lines)} lines."
        )

    if len(source_lines) == 0:
        raise ValueError("The cleaned corpus is empty.")

    blank_source_idx = [i for i, line in enumerate(source_lines) if line.strip() == ""]
    blank_target_idx = [i for i, line in enumerate(target_lines) if line.strip() == ""]

    if blank_source_idx:
        raise ValueError(f"Blank lines found in source file at indices: {blank_source_idx[:10]}")
    if blank_target_idx:
        raise ValueError(f"Blank lines found in target file at indices: {blank_target_idx[:10]}")

    pairs = list(zip(source_lines, target_lines))
    unique_pairs = set(pairs)

    if len(unique_pairs) != len(pairs):
        raise ValueError(
            f"Duplicate parallel pairs detected after preprocessing: "
            f"{len(pairs) - len(unique_pairs)} duplicates still remain."
        )

    suspicious_source_ws = [i for i, line in enumerate(source_lines) if line != normalize_script_like_whitespace(line)]
    suspicious_target_ws = [i for i, line in enumerate(target_lines) if line != normalize_script_like_whitespace(line)]

    summary = {
        "num_pairs": len(pairs),
        "source_path": str(source_path),
        "target_path": str(target_path),
        "same_number_of_lines": True,
        "no_blank_lines": True,
        "no_duplicate_parallel_pairs": True,
        "alignment_is_structurally_consistent": True,
        "whitespace_check_note": (
            "Structural validation passed. Some lines may still contain Unicode spacing "
            "characters not fully normalized by the provided shell script."
            if suspicious_source_ws or suspicious_target_ws
            else "No suspicious spacing patterns detected."
        ),
        "num_suspicious_source_spacing_lines": len(suspicious_source_ws),
        "num_suspicious_target_spacing_lines": len(suspicious_target_ws),
    }

    if verbose:
        print("Validation successful.")
        print(f"Number of cleaned parallel pairs: {summary['num_pairs']}")
        print(f"Clean source file: {summary['source_path']}")
        print(f"Clean target file: {summary['target_path']}")
        print(summary["whitespace_check_note"])
        if suspicious_source_ws:
            print("Example suspicious source indices:", suspicious_source_ws[:10])
        if suspicious_target_ws:
            print("Example suspicious target indices:", suspicious_target_ws[:10])

    return summary

def run_preprocess_script_locally(
    script_path,
    raw_source_path,
    raw_target_path,
    clean_source_path,
    clean_target_path,
    overwrite=False,
    verbose=True,
):
    import shutil
    import time

    script_path = Path(script_path)
    raw_source_path = Path(raw_source_path)
    raw_target_path = Path(raw_target_path)
    clean_source_path = Path(clean_source_path)
    clean_target_path = Path(clean_target_path)

    if not script_path.exists():
        raise FileNotFoundError(f"Script not found: {script_path}")
    if not raw_source_path.exists():
        raise FileNotFoundError(f"Raw source file not found: {raw_source_path}")
    if not raw_target_path.exists():
        raise FileNotFoundError(f"Raw target file not found: {raw_target_path}")

    raw_source_lines = count_lines(raw_source_path)
    raw_target_lines = count_lines(raw_target_path)

    if raw_source_lines != raw_target_lines:
        raise ValueError(
            f"Raw corpus files do not have the same number of lines: "
            f"{raw_source_path} has {raw_source_lines}, "
            f"{raw_target_path} has {raw_target_lines}."
        )

    workdir = Path("/content/anlp_work")
    workdir.mkdir(parents=True, exist_ok=True)

    local_raw_source = workdir / raw_source_path.name
    local_raw_target = workdir / raw_target_path.name
    local_clean_source = workdir / clean_source_path.name
    local_clean_target = workdir / clean_target_path.name

    for p in [local_raw_source, local_raw_target, local_clean_source, local_clean_target]:
        if p.exists():
            p.unlink()

    if overwrite:
        for p in [clean_source_path, clean_target_path]:
            if p.exists():
                p.unlink()
    else:
        if clean_source_path.exists():
            raise FileExistsError(f"Output file already exists: {clean_source_path}")
        if clean_target_path.exists():
            raise FileExistsError(f"Output file already exists: {clean_target_path}")

    if verbose:
        print("Copying raw files from Google Drive to local Colab storage...")

    shutil.copy2(raw_source_path, local_raw_source)
    shutil.copy2(raw_target_path, local_raw_target)

    if verbose:
        print("Running preprocess-corpus.sh locally...")

    start = time.time()
    result = subprocess.run(
        [
            "bash",
            str(script_path),
            str(local_raw_source),
            str(local_raw_target),
            str(local_clean_source),
            str(local_clean_target),
        ],
        text=True,
        capture_output=True,
    )
    elapsed = time.time() - start

    if verbose:
        print(f"Finished in {elapsed:.2f} seconds.")
        print("Return code:", result.returncode)
        if result.stdout.strip():
            print("\n--- STDOUT ---")
            print(result.stdout)
        if result.stderr.strip():
            print("\n--- STDERR ---")
            print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(
            "preprocess-corpus.sh failed.\n"
            f"Return code: {result.returncode}\n"
            f"STDOUT:\n{result.stdout}\n"
            f"STDERR:\n{result.stderr}"
        )

    if verbose:
        print("Copying cleaned files back to Google Drive...")

    shutil.copy2(local_clean_source, clean_source_path)
    shutil.copy2(local_clean_target, clean_target_path)

    validation_summary = validate_clean_parallel_corpus(
        clean_source_path,
        clean_target_path,
        verbose=verbose,
    )

    return {
        "raw_source_lines": raw_source_lines,
        "raw_target_lines": raw_target_lines,
        "clean_pairs": validation_summary["num_pairs"],
        "clean_source_path": str(clean_source_path),
        "clean_target_path": str(clean_target_path),
        "num_suspicious_source_spacing_lines": validation_summary["num_suspicious_source_spacing_lines"],
        "num_suspicious_target_spacing_lines": validation_summary["num_suspicious_target_spacing_lines"],
    }

We perform the real preprocessing of the News Commentary corpus using the helpers defined previously. The implementation calls run_preprocess_script_locally with the raw input files and cleaned output paths, enables overwriting to support re-execution, and prints the returned summary so that we can inspect the number of raw lines, the final number of cleaned parallel pairs, and the output locations.

In [6]:
summary = run_preprocess_script_locally(
    script_path=script_path,
    raw_source_path=raw_source_path,
    raw_target_path=raw_target_path,
    clean_source_path=clean_source_path,
    clean_target_path=clean_target_path,
    overwrite=True,
    verbose=True,
)

print("\nPreprocessing summary:")
for key, value in summary.items():
    print(f"{key}: {value}")

Copying raw files from Google Drive to local Colab storage...
Running preprocess-corpus.sh locally...
Finished in 14.82 seconds.
Return code: 0
Copying cleaned files back to Google Drive...
Validation successful.
Number of cleaned parallel pairs: 238511
Clean source file: /content/drive/MyDrive/anlp/news_commentary_clean.en
Clean target file: /content/drive/MyDrive/anlp/news_commentary_clean.es
Structural validation passed. Some lines may still contain Unicode spacing characters not fully normalized by the provided shell script.
Example suspicious source indices: [128, 379, 465, 2167, 2594, 2791, 3503, 3514, 3634, 5489]
Example suspicious target indices: [230, 848, 1118, 1503, 1634, 1999, 2355, 2603, 2877, 4874]

Preprocessing summary:
raw_source_lines: 238872
raw_target_lines: 238872
clean_pairs: 238511
clean_source_path: /content/drive/MyDrive/anlp/news_commentary_clean.en
clean_target_path: /content/drive/MyDrive/anlp/news_commentary_clean.es
num_suspicious_source_spacing_lines: 581

This cell performs a lightweight manual inspection of the cleaned bilingual corpus. It reloads the cleaned English and Spanish files with read_lines, prints their lengths to verify that both sides still contain the same number of entries, and then displays the first five parallel pairs in aligned form.

In [7]:
clean_source_lines = read_lines(clean_source_path)
clean_target_lines = read_lines(clean_target_path)

print(f"Cleaned source lines: {len(clean_source_lines)}")
print(f"Cleaned target lines: {len(clean_target_lines)}")

print("\nFirst 5 cleaned parallel pairs:")
for i, (src, tgt) in enumerate(zip(clean_source_lines[:5], clean_target_lines[:5]), start=1):
    print(f"\nPair {i}")
    print("EN:", src)
    print("ES:", tgt)

Cleaned source lines: 238511
Cleaned target lines: 238511

First 5 cleaned parallel pairs:

Pair 1
EN: But caste-bound India’s record of exclusion is worse.
ES: Pero el récord de exclusión de la India ceñida a las castas es peor.

Pair 2
EN: The reasons are complex and not completely understood, but the facts are clear.
ES: Las razones son complejas y no se han llegado a explicar en su totalidad, pero los hechos son claros.

Pair 3
EN: Indeed, Turkey’s leaders are downgrading the importance of “hard power” security issues in favor of enhancing the country’s soft power while also grasping economic opportunities.
ES: De hecho, los líderes turcos están reduciendo la importancia de los problemas de seguridad de "poder duro" en favor de aumentar el poder blando del país, aprovechando al mismo tiempo las oportunidades económicas.

Pair 4
EN: But that does not mean that clashes are inevitable.
ES: Pero eso no significa que los enfrentamientos sean inevitables.

Pair 5
EN: On the ground, it ha

We implement the explicit post-preprocessing quality checks required to confirm that the cleaned corpus is structurally valid for later experimental splitting. The code reconstructs the parallel pairs, checks whether both files contain the same number of lines, verifies the absence of blank lines on either side, tests whether duplicate bilingual pairs remain.

In [8]:
clean_source_lines = read_lines(clean_source_path)
clean_target_lines = read_lines(clean_target_path)
clean_pairs = list(zip(clean_source_lines, clean_target_lines))

same_num_lines = len(clean_source_lines) == len(clean_target_lines)
no_blank_lines = all(line.strip() != "" for line in clean_source_lines) and \
                 all(line.strip() != "" for line in clean_target_lines)
no_duplicate_pairs = len(clean_pairs) == len(set(clean_pairs))

print("QUALITY CHECKS AFTER PREPROCESSING")
print("----------------------------------")
print(f"1. Same number of lines on both sides: {same_num_lines}")
print(f"2. No blank lines: {no_blank_lines}")
print(f"3. No duplicate parallel pairs: {no_duplicate_pairs}")
print(f"4. Structural alignment preserved: {same_num_lines and len(clean_pairs) == len(clean_source_lines)}")

assert same_num_lines, "The cleaned source and target files do not contain the same number of lines."
assert no_blank_lines, "Blank lines were found after preprocessing."
assert no_duplicate_pairs, "Duplicate parallel pairs remain after preprocessing."

print("\nAll required quality checks passed.")
print("Note: minor Unicode spacing characters may still exist in some lines.")

QUALITY CHECKS AFTER PREPROCESSING
----------------------------------
1. Same number of lines on both sides: True
2. No blank lines: True
3. No duplicate parallel pairs: True
4. Structural alignment preserved: True

All required quality checks passed.
Note: minor Unicode spacing characters may still exist in some lines,


## SECTION 2

The second section defines the experimental partition of the cleaned News Commentary English–Spanish corpus and operationalizes the strict non-overlap policy, which is essential to prevent data leakage and preserve valid development and test evaluation. The implementation first fixes the exact assignment sizes and output files for the five required subsets: 1,000 parallel training pairs, 200 development pairs, 200 test pairs, 1,000 English monolingual sentences, and 1,000 Spanish monolingual sentences. It then constructs the split exclusively from the preprocessed bilingual pool (`news_commentary_clean.en/.es`), applying an additional language-sanity filter to discard suspicious non-English/Spanish pairs and using disjoint sentence-pair slices for train, dev, and test. The monolingual corpora are not sampled independently from the whole corpus; instead, they are extracted from the remaining bilingual pairs excluded from all parallel subsets, taking only the English side for the English monolingual file and only the Spanish side for the Spanish monolingual file. This design is reinforced by explicit validation routines that check exact sizes, absence of blank lines, uniqueness of monolingual entries, absence of overlap between train/dev/test, and absence of overlap between monolingual sentences and the corresponding sides of any parallel subset.


The following cell code defines the formal structure of the final experimental partition by fixing both the exact corpus sizes and the names and paths of the output files that will store each subset. It creates separate variables for the bilingual training, development, and test sets, as well as for the English and Spanish monolingual corpora, and then maps each of them inside the /content/drive/MyDrive/anlp directory. Its main role is to centralize the split configuration so that all later operations use a explicit specification of the required datasets and their expected sizes.

In [9]:
parallel_training_size = 1000
development_parallel_size = 200
test_parallel_size = 200
english_monolingual_size = 1000
spanish_monolingual_size = 1000

parallel_training_source_filename = "news_commentary_train.en"
parallel_training_target_filename = "news_commentary_train.es"

development_source_filename = "news_commentary_dev.en"
development_target_filename = "news_commentary_dev.es"

test_source_filename = "news_commentary_test.en"
test_target_filename = "news_commentary_test.es"

english_monolingual_filename = "news_commentary_mono.en"
spanish_monolingual_filename = "news_commentary_mono.es"

parallel_training_source_path = mydrive / parallel_training_source_filename
parallel_training_target_path = mydrive / parallel_training_target_filename

development_source_path = mydrive / development_source_filename
development_target_path = mydrive / development_target_filename

test_source_path = mydrive / test_source_filename
test_target_path = mydrive / test_target_filename

english_monolingual_path = mydrive / english_monolingual_filename
spanish_monolingual_path = mydrive / spanish_monolingual_filename

print("Final split files that will be created:")
print("Parallel training EN:", parallel_training_source_path)
print("Parallel training ES:", parallel_training_target_path)
print("Development EN      :", development_source_path)
print("Development ES      :", development_target_path)
print("Test EN             :", test_source_path)
print("Test ES             :", test_target_path)
print("Monolingual EN      :", english_monolingual_path)
print("Monolingual ES      :", spanish_monolingual_path)

print("\nRequired sizes:")
print("Parallel training size :", parallel_training_size)
print("Development size       :", development_parallel_size)
print("Test size              :", test_parallel_size)
print("English monolingual    :", english_monolingual_size)
print("Spanish monolingual    :", spanish_monolingual_size)

Final split files that will be created:
Parallel training EN: /content/drive/MyDrive/anlp/news_commentary_train.en
Parallel training ES: /content/drive/MyDrive/anlp/news_commentary_train.es
Development EN      : /content/drive/MyDrive/anlp/news_commentary_dev.en
Development ES      : /content/drive/MyDrive/anlp/news_commentary_dev.es
Test EN             : /content/drive/MyDrive/anlp/news_commentary_test.en
Test ES             : /content/drive/MyDrive/anlp/news_commentary_test.es
Monolingual EN      : /content/drive/MyDrive/anlp/news_commentary_mono.en
Monolingual ES      : /content/drive/MyDrive/anlp/news_commentary_mono.es

Required sizes:
Parallel training size : 1000
Development size       : 200
Test size              : 200
English monolingual    : 1000
Spanish monolingual    : 1000


The following cell code defines the utilities used to build the final experimental split from the cleaned English–Spanish corpus. It first applies a simple language-sanity filter based on regular expressions to discard suspicious pairs that do not look like English–Spanish text. Then, it includes helper functions to write and read Moses-format files, manage output paths safely, and preserve alignment between both languages. The main function, `build_experimental_split_from_clean_corpus`, creates the required train, development, test, and monolingual subsets from the cleaned bilingual pool, ensuring that the parallel subsets are disjoint and that monolingual sentences do not overlap with the corresponding side of any parallel dataset. Finally, `validate_experimental_split_non_overlap` verifies the exact sizes, checks for blank lines and duplicate monolingual entries, and enforces the strict non-overlap policy required to avoid data leakage in the iterative back-translation experiment.


In [10]:
from pathlib import Path
import re

# The main objective of this cell is to filter for english-spanish pairs
# in order to keep only sentence pairs that look like English-Spanish text

NON_LATIN_SCRIPT_PATTERN = re.compile(
    "["
    "\u0370-\u03FF"  # Greek
    "\u0400-\u04FF"  # Cyrillic
    "\u0500-\u052F"  # Cyrillic Supplement
    "\u0590-\u05FF"  # Hebrew
    "\u0600-\u06FF"  # Arabic
    "\u0750-\u077F"  # Arabic Supplement
    "\u08A0-\u08FF"  # Arabic Extended
    "\u0900-\u097F"  # Devanagari
    "\u0980-\u09FF"  # Bengali
    "\u0A00-\u0A7F"  # Gurmukhi
    "\u0A80-\u0AFF"  # Gujarati
    "\u0B00-\u0B7F"  # Oriya
    "\u0B80-\u0BFF"  # Tamil
    "\u0C00-\u0C7F"  # Telugu
    "\u0C80-\u0CFF"  # Kannada
    "\u0D00-\u0D7F"  # Malayalam
    "\u0E00-\u0E7F"  # Thai
    "\u0E80-\u0EFF"  # Lao
    "\u10A0-\u10FF"  # Georgian
    "\u3040-\u309F"  # Hiragana
    "\u30A0-\u30FF"  # Katakana
    "\u3400-\u4DBF"  # CJK Extension A
    "\u4E00-\u9FFF"  # CJK Unified Ideographs
    "\uAC00-\uD7AF"  # Hangul
    "]"
)

LATIN_LETTER_PATTERN = re.compile(r"[A-Za-zÁÉÍÓÚÜÑáéíóúüñÀ-ÖØ-öø-ÿ]")

def looks_like_english_spanish_pair(source_sentence: str, target_sentence: str) -> bool:
    if source_sentence.strip() == "" or target_sentence.strip() == "":
        return False

    if not LATIN_LETTER_PATTERN.search(source_sentence):
        return False
    if not LATIN_LETTER_PATTERN.search(target_sentence):
        return False

    if NON_LATIN_SCRIPT_PATTERN.search(source_sentence):
        return False
    if NON_LATIN_SCRIPT_PATTERN.search(target_sentence):
        return False

    return True


def write_lines_to_file(output_path, lines):
    output_path = Path(output_path)
    with output_path.open("w", encoding="utf-8") as f:
        for line in lines:
            f.write(f"{line}\n")

def write_parallel_pairs_to_moses_files(source_output_path, target_output_path, parallel_pairs):
    source_lines = [source_sentence for source_sentence, _ in parallel_pairs]
    target_lines = [target_sentence for _, target_sentence in parallel_pairs]
    write_lines_to_file(source_output_path, source_lines)
    write_lines_to_file(target_output_path, target_lines)

def ensure_output_paths_are_writable(output_paths, overwrite=False):
    for output_path in output_paths:
        output_path = Path(output_path)
        if output_path.exists():
            if overwrite:
                output_path.unlink()
            else:
                raise FileExistsError(
                    f"Output file already exists: {output_path}. "
                    f"Set overwrite=True if you want to recreate it."
                )

def read_parallel_pairs_from_moses_files(source_path, target_path):
    source_lines = read_lines(source_path)
    target_lines = read_lines(target_path)

    if len(source_lines) != len(target_lines):
        raise ValueError(
            f"Moses files do not have the same number of lines: "
            f"{source_path} has {len(source_lines)}, "
            f"{target_path} has {len(target_lines)}."
        )

    return list(zip(source_lines, target_lines))

def build_experimental_split_from_clean_corpus(
    clean_source_corpus_path,
    clean_target_corpus_path,
    parallel_training_source_output_path,
    parallel_training_target_output_path,
    development_source_output_path,
    development_target_output_path,
    test_source_output_path,
    test_target_output_path,
    english_monolingual_output_path,
    spanish_monolingual_output_path,
    parallel_training_size,
    development_parallel_size,
    test_parallel_size,
    english_monolingual_size,
    spanish_monolingual_size,
    overwrite=False,
    verbose=True,
):

    clean_source_corpus_path = Path(clean_source_corpus_path)
    clean_target_corpus_path = Path(clean_target_corpus_path)

    cleaned_source_sentences = read_lines(clean_source_corpus_path)
    cleaned_target_sentences = read_lines(clean_target_corpus_path)

    if len(cleaned_source_sentences) != len(cleaned_target_sentences):
        raise ValueError(
            f"The cleaned corpus files do not have the same number of lines: "
            f"{clean_source_corpus_path} has {len(cleaned_source_sentences)}, "
            f"{clean_target_corpus_path} has {len(cleaned_target_sentences)}."
        )

    cleaned_parallel_pairs = list(zip(cleaned_source_sentences, cleaned_target_sentences))

    # Filter out suspicious pairs that do not look like EN-ES
    original_cleaned_pair_count = len(cleaned_parallel_pairs)

    cleaned_parallel_pairs = [
        (source_sentence, target_sentence)
        for source_sentence, target_sentence in cleaned_parallel_pairs
        if looks_like_english_spanish_pair(source_sentence, target_sentence)
    ]

    filtered_out_pair_count = original_cleaned_pair_count - len(cleaned_parallel_pairs)

    if verbose:
        print(
            f"Language sanity filter: kept {len(cleaned_parallel_pairs)} candidate EN-ES pairs "
            f"and removed {filtered_out_pair_count} suspicious pairs."
        )

    required_parallel_pairs = (
        parallel_training_size
        + development_parallel_size
        + test_parallel_size
    )

    if len(cleaned_parallel_pairs) < required_parallel_pairs:
        raise ValueError(
            f"The cleaned corpus is too small after EN-ES filtering. "
            f"At least {required_parallel_pairs} cleaned parallel pairs are required "
            f"for train/dev/test, but only {len(cleaned_parallel_pairs)} are available."
        )

    ensure_output_paths_are_writable(
        [
            parallel_training_source_output_path,
            parallel_training_target_output_path,
            development_source_output_path,
            development_target_output_path,
            test_source_output_path,
            test_target_output_path,
            english_monolingual_output_path,
            spanish_monolingual_output_path,
        ],
        overwrite=overwrite,
    )

    parallel_training_end = parallel_training_size
    development_end = parallel_training_end + development_parallel_size
    test_end = development_end + test_parallel_size

    parallel_training_pairs = cleaned_parallel_pairs[:parallel_training_end]
    development_pairs = cleaned_parallel_pairs[parallel_training_end:development_end]
    test_pairs = cleaned_parallel_pairs[development_end:test_end]

    remaining_pairs_for_monolingual_extraction = cleaned_parallel_pairs[test_end:]

    parallel_source_sentences = {
        source_sentence
        for source_sentence, _ in (parallel_training_pairs + development_pairs + test_pairs)
    }
    parallel_target_sentences = {
        target_sentence
        for _, target_sentence in (parallel_training_pairs + development_pairs + test_pairs)
    }

    english_monolingual_sentences = []
    seen_english_monolingual_sentences = set()

    for source_sentence, _ in remaining_pairs_for_monolingual_extraction:
        if source_sentence.strip() == "":
            continue
        if source_sentence in parallel_source_sentences:
            continue
        if source_sentence in seen_english_monolingual_sentences:
            continue

        english_monolingual_sentences.append(source_sentence)
        seen_english_monolingual_sentences.add(source_sentence)

        if len(english_monolingual_sentences) == english_monolingual_size:
            break

    spanish_monolingual_sentences = []
    seen_spanish_monolingual_sentences = set()

    for _, target_sentence in remaining_pairs_for_monolingual_extraction:
        if target_sentence.strip() == "":
            continue
        if target_sentence in parallel_target_sentences:
            continue
        if target_sentence in seen_spanish_monolingual_sentences:
            continue

        spanish_monolingual_sentences.append(target_sentence)
        seen_spanish_monolingual_sentences.add(target_sentence)

        if len(spanish_monolingual_sentences) == spanish_monolingual_size:
            break

    if len(english_monolingual_sentences) != english_monolingual_size:
        raise ValueError(
            f"Could not extract {english_monolingual_size} English monolingual sentences "
            f"without overlap. Only {len(english_monolingual_sentences)} were found."
        )

    if len(spanish_monolingual_sentences) != spanish_monolingual_size:
        raise ValueError(
            f"Could not extract {spanish_monolingual_size} Spanish monolingual sentences "
            f"without overlap. Only {len(spanish_monolingual_sentences)} were found."
        )

    write_parallel_pairs_to_moses_files(
        parallel_training_source_output_path,
        parallel_training_target_output_path,
        parallel_training_pairs,
    )
    write_parallel_pairs_to_moses_files(
        development_source_output_path,
        development_target_output_path,
        development_pairs,
    )
    write_parallel_pairs_to_moses_files(
        test_source_output_path,
        test_target_output_path,
        test_pairs,
    )

    write_lines_to_file(english_monolingual_output_path, english_monolingual_sentences)
    write_lines_to_file(spanish_monolingual_output_path, spanish_monolingual_sentences)

    split_summary = validate_experimental_split_non_overlap(
        parallel_training_source_output_path,
        parallel_training_target_output_path,
        development_source_output_path,
        development_target_output_path,
        test_source_output_path,
        test_target_output_path,
        english_monolingual_output_path,
        spanish_monolingual_output_path,
        expected_parallel_training_size=parallel_training_size,
        expected_development_size=development_parallel_size,
        expected_test_size=test_parallel_size,
        expected_english_monolingual_size=english_monolingual_size,
        expected_spanish_monolingual_size=spanish_monolingual_size,
        verbose=verbose,
    )

    return split_summary

def validate_experimental_split_non_overlap(
    parallel_training_source_path,
    parallel_training_target_path,
    development_source_path,
    development_target_path,
    test_source_path,
    test_target_path,
    english_monolingual_path,
    spanish_monolingual_path,
    expected_parallel_training_size,
    expected_development_size,
    expected_test_size,
    expected_english_monolingual_size,
    expected_spanish_monolingual_size,
    verbose=True,
):
    parallel_training_pairs = read_parallel_pairs_from_moses_files(
        parallel_training_source_path, parallel_training_target_path
    )
    development_pairs = read_parallel_pairs_from_moses_files(
        development_source_path, development_target_path
    )
    test_pairs = read_parallel_pairs_from_moses_files(
        test_source_path, test_target_path
    )

    english_monolingual_sentences = read_lines(english_monolingual_path)
    spanish_monolingual_sentences = read_lines(spanish_monolingual_path)

    if len(parallel_training_pairs) != expected_parallel_training_size:
        raise ValueError(
            f"Parallel training set has {len(parallel_training_pairs)} pairs, "
            f"but {expected_parallel_training_size} were expected."
        )
    if len(development_pairs) != expected_development_size:
        raise ValueError(
            f"Development set has {len(development_pairs)} pairs, "
            f"but {expected_development_size} were expected."
        )
    if len(test_pairs) != expected_test_size:
        raise ValueError(
            f"Test set has {len(test_pairs)} pairs, "
            f"but {expected_test_size} were expected."
        )
    if len(english_monolingual_sentences) != expected_english_monolingual_size:
        raise ValueError(
            f"English monolingual set has {len(english_monolingual_sentences)} sentences, "
            f"but {expected_english_monolingual_size} were expected."
        )
    if len(spanish_monolingual_sentences) != expected_spanish_monolingual_size:
        raise ValueError(
            f"Spanish monolingual set has {len(spanish_monolingual_sentences)} sentences, "
            f"but {expected_spanish_monolingual_size} were expected."
        )

    if any(sentence.strip() == "" for sentence in english_monolingual_sentences):
        raise ValueError("The English monolingual file contains blank lines.")
    if any(sentence.strip() == "" for sentence in spanish_monolingual_sentences):
        raise ValueError("The Spanish monolingual file contains blank lines.")

    if len(english_monolingual_sentences) != len(set(english_monolingual_sentences)):
        raise ValueError("The English monolingual file contains duplicate sentences.")
    if len(spanish_monolingual_sentences) != len(set(spanish_monolingual_sentences)):
        raise ValueError("The Spanish monolingual file contains duplicate sentences.")

    parallel_training_pair_set = set(parallel_training_pairs)
    development_pair_set = set(development_pairs)
    test_pair_set = set(test_pairs)

    training_development_pair_overlap = parallel_training_pair_set & development_pair_set
    training_test_pair_overlap = parallel_training_pair_set & test_pair_set
    development_test_pair_overlap = development_pair_set & test_pair_set

    if training_development_pair_overlap:
        raise ValueError("Parallel training and development sets overlap.")
    if training_test_pair_overlap:
        raise ValueError("Parallel training and test sets overlap.")
    if development_test_pair_overlap:
        raise ValueError("Development and test sets overlap.")

    all_parallel_source_sentences = {
        source_sentence
        for source_sentence, _ in (parallel_training_pairs + development_pairs + test_pairs)
    }
    all_parallel_target_sentences = {
        target_sentence
        for _, target_sentence in (parallel_training_pairs + development_pairs + test_pairs)
    }

    english_monolingual_parallel_overlap = set(english_monolingual_sentences) & all_parallel_source_sentences
    spanish_monolingual_parallel_overlap = set(spanish_monolingual_sentences) & all_parallel_target_sentences

    if english_monolingual_parallel_overlap:
        raise ValueError(
            "The English monolingual set overlaps with the English side of the parallel subsets."
        )
    if spanish_monolingual_parallel_overlap:
        raise ValueError(
            "The Spanish monolingual set overlaps with the Spanish side of the parallel subsets."
        )

    validation_summary = {
        "parallel_training_pairs": len(parallel_training_pairs),
        "development_pairs": len(development_pairs),
        "test_pairs": len(test_pairs),
        "english_monolingual_sentences": len(english_monolingual_sentences),
        "spanish_monolingual_sentences": len(spanish_monolingual_sentences),
        "training_vs_development_overlap": False,
        "training_vs_test_overlap": False,
        "development_vs_test_overlap": False,
        "english_monolingual_vs_parallel_overlap": False,
        "spanish_monolingual_vs_parallel_overlap": False,
        "no_blank_lines_in_monolingual_files": True,
        "no_duplicates_in_monolingual_files": True,
    }

    if verbose:
        print("Experimental split validation successful.")
        print("Parallel training pairs      :", validation_summary["parallel_training_pairs"])
        print("Development pairs            :", validation_summary["development_pairs"])
        print("Test pairs                   :", validation_summary["test_pairs"])
        print("English monolingual sentences:", validation_summary["english_monolingual_sentences"])
        print("Spanish monolingual sentences:", validation_summary["spanish_monolingual_sentences"])
        print("\nStrict non-overlap policy satisfied across all required subsets.")

    return validation_summary

The following code executes the split-generation pipeline defined in the previous cell and therefore materializes the final corpora used in the experiment.

In [11]:
experimental_split_summary = build_experimental_split_from_clean_corpus(
    clean_source_corpus_path=clean_source_path,
    clean_target_corpus_path=clean_target_path,
    parallel_training_source_output_path=parallel_training_source_path,
    parallel_training_target_output_path=parallel_training_target_path,
    development_source_output_path=development_source_path,
    development_target_output_path=development_target_path,
    test_source_output_path=test_source_path,
    test_target_output_path=test_target_path,
    english_monolingual_output_path=english_monolingual_path,
    spanish_monolingual_output_path=spanish_monolingual_path,
    parallel_training_size=parallel_training_size,
    development_parallel_size=development_parallel_size,
    test_parallel_size=test_parallel_size,
    english_monolingual_size=english_monolingual_size,
    spanish_monolingual_size=spanish_monolingual_size,
    overwrite=True,
    verbose=True,
)

print("\nExperimental split summary:")
for key, value in experimental_split_summary.items():
    print(f"{key}: {value}")

Language sanity filter: kept 237132 candidate EN-ES pairs and removed 1379 suspicious pairs.
Experimental split validation successful.
Parallel training pairs      : 1000
Development pairs            : 200
Test pairs                   : 200
English monolingual sentences: 1000
Spanish monolingual sentences: 1000

Strict non-overlap policy satisfied across all required subsets.

Experimental split summary:
parallel_training_pairs: 1000
development_pairs: 200
test_pairs: 200
english_monolingual_sentences: 1000
spanish_monolingual_sentences: 1000
training_vs_development_overlap: False
training_vs_test_overlap: False
development_vs_test_overlap: False
english_monolingual_vs_parallel_overlap: False
spanish_monolingual_vs_parallel_overlap: False
no_blank_lines_in_monolingual_files: True
no_duplicates_in_monolingual_files: True


In [12]:
print("Created files:")
print(" -", parallel_training_source_path.name)
print(" -", parallel_training_target_path.name)
print(" -", development_source_path.name)
print(" -", development_target_path.name)
print(" -", test_source_path.name)
print(" -", test_target_path.name)
print(" -", english_monolingual_path.name)
print(" -", spanish_monolingual_path.name)

parallel_training_pairs_preview = read_parallel_pairs_from_moses_files(
    parallel_training_source_path, parallel_training_target_path
)
development_pairs_preview = read_parallel_pairs_from_moses_files(
    development_source_path, development_target_path
)
test_pairs_preview = read_parallel_pairs_from_moses_files(
    test_source_path, test_target_path
)

english_monolingual_preview = read_lines(english_monolingual_path)
spanish_monolingual_preview = read_lines(spanish_monolingual_path)

print("\nFirst 2 parallel training pairs:")
for pair_index, (source_sentence, target_sentence) in enumerate(parallel_training_pairs_preview[:2], start=1):
    print(f"\nTraining pair {pair_index}")
    print("EN:", source_sentence)
    print("ES:", target_sentence)

print("\nFirst 2 development pairs:")
for pair_index, (source_sentence, target_sentence) in enumerate(development_pairs_preview[:2], start=1):
    print(f"\nDevelopment pair {pair_index}")
    print("EN:", source_sentence)
    print("ES:", target_sentence)

print("\nFirst 2 test pairs:")
for pair_index, (source_sentence, target_sentence) in enumerate(test_pairs_preview[:2], start=1):
    print(f"\nTest pair {pair_index}")
    print("EN:", source_sentence)
    print("ES:", target_sentence)

print("\nFirst 3 English monolingual sentences:")
for sentence_index, source_sentence in enumerate(english_monolingual_preview[:3], start=1):
    print(f"{sentence_index}. {source_sentence}")

print("\nFirst 3 Spanish monolingual sentences:")
for sentence_index, target_sentence in enumerate(spanish_monolingual_preview[:3], start=1):
    print(f"{sentence_index}. {target_sentence}")

Created files:
 - news_commentary_train.en
 - news_commentary_train.es
 - news_commentary_dev.en
 - news_commentary_dev.es
 - news_commentary_test.en
 - news_commentary_test.es
 - news_commentary_mono.en
 - news_commentary_mono.es

First 2 parallel training pairs:

Training pair 1
EN: But caste-bound India’s record of exclusion is worse.
ES: Pero el récord de exclusión de la India ceñida a las castas es peor.

Training pair 2
EN: The reasons are complex and not completely understood, but the facts are clear.
ES: Las razones son complejas y no se han llegado a explicar en su totalidad, pero los hechos son claros.

First 2 development pairs:

Development pair 1
EN: The US should do the same.
ES: Estados Unidos debería hacer lo mismo.

Development pair 2
EN: Corporate earnings in Brazil have gone up roughly as fast as stock prices.
ES: Las ganancias corporativas han subido en Brasil casi tan rápido como los precios de las acciones.

First 2 test pairs:

Test pair 1
EN: Deflecting mounting

## SECTION 3

The third section establishes the operational environment required to run the bidirectional English↔Spanish iterative back-translation. This is implemented by first installing only the dependencies needed by the final workflow, pinning their versions explicitly, and verifying compatibility between critical libraries such as `transformers` and `accelerate`, which prevents execution errors during later fine-tuning. The environment is then configured by mounting Google Drive at `/content/drive`, defining the fixed working directory `/content/drive/MyDrive/anlp/`, and declaring in advance the selected pretrained models, the input corpus files, the output directories, the required corpus sizes, and the main training and inference parameters. To reduce unnecessary run-to-run variation, the project sets reproducibility-oriented controls through a dedicated function that fixes the random seed for Python, NumPy, and PyTorch, also extending this configuration to CUDA and cuDNN when GPU is available. In addition, the implementation includes runtime-oriented safeguards, such as disabling tokenizer parallelism warnings, adapting batch size and mixed-precision settings to GPU availability, and validating that all required files exist and have the exact sizes and line alignment.


The following code configures the execution environment for the bidirectional machine translation experiment.

In [13]:
from google.colab import drive
import os
import random
import platform
from pathlib import Path

import numpy as np
import torch

drive.mount("/content/drive", force_remount=True)

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONHASHSEED"] = "42"

def set_global_reproducibility(random_seed: int) -> None:
    random.seed(random_seed)
    np.random.seed(random_seed)
    torch.manual_seed(random_seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(random_seed)
        torch.cuda.manual_seed_all(random_seed)

    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

random_seed = 42
set_global_reproducibility(random_seed)

corpus_root_directory = Path("/content/drive/MyDrive/anlp")
corpus_root_directory.mkdir(parents=True, exist_ok=True)

source_language_code = "en"
target_language_code = "es"

forward_translation_model_name = "Helsinki-NLP/opus-mt-en-es"
backward_translation_model_name = "Helsinki-NLP/opus-mt-es-en"

parallel_training_source_path = corpus_root_directory / "news_commentary_train.en"
parallel_training_target_path = corpus_root_directory / "news_commentary_train.es"

development_source_path = corpus_root_directory / "news_commentary_dev.en"
development_target_path = corpus_root_directory / "news_commentary_dev.es"

test_source_path = corpus_root_directory / "news_commentary_test.en"
test_target_path = corpus_root_directory / "news_commentary_test.es"

english_monolingual_path = corpus_root_directory / "news_commentary_mono.en"
spanish_monolingual_path = corpus_root_directory / "news_commentary_mono.es"

forward_model_output_directory = corpus_root_directory / "iterative_backtranslation_en_to_es"
backward_model_output_directory = corpus_root_directory / "iterative_backtranslation_es_to_en"

forward_model_output_directory.mkdir(parents=True, exist_ok=True)
backward_model_output_directory.mkdir(parents=True, exist_ok=True)

parallel_training_size = 1000
development_parallel_size = 200
test_parallel_size = 200
english_monolingual_size = 1000
spanish_monolingual_size = 1000

num_backtranslation_iterations = 3
early_stopping_patience = 3

max_source_sequence_length = 128
max_target_sequence_length = 128

learning_rate = 2e-5
training_batch_size = 16 if torch.cuda.is_available() else 8
evaluation_batch_size = training_batch_size

execution_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pipeline_device = 0 if torch.cuda.is_available() else -1
use_fp16_training = torch.cuda.is_available()

print("Environment configuration completed successfully.\n")
print(f"Python version               : {platform.python_version()}")
print(f"PyTorch version              : {torch.__version__}")
print(f"Execution device             : {execution_device}")
print(f"Pipeline device              : {pipeline_device}")
print(f"Mixed precision enabled      : {use_fp16_training}")
print(f"Corpus root directory        : {corpus_root_directory}")
print(f"Forward model                : {forward_translation_model_name}")
print(f"Backward model               : {backward_translation_model_name}")
print(f"Forward model output folder  : {forward_model_output_directory}")
print(f"Backward model output folder : {backward_model_output_directory}")

Mounted at /content/drive
Environment configuration completed successfully.

Python version               : 3.12.12
PyTorch version              : 2.10.0+cu128
Execution device             : cuda
Pipeline device              : 0
Mixed precision enabled      : True
Corpus root directory        : /content/drive/MyDrive/anlp
Forward model                : Helsinki-NLP/opus-mt-en-es
Backward model               : Helsinki-NLP/opus-mt-es-en
Forward model output folder  : /content/drive/MyDrive/anlp/iterative_backtranslation_en_to_es
Backward model output folder : /content/drive/MyDrive/anlp/iterative_backtranslation_es_to_en


This cell verifies that the environment and corpus files are correctly prepared before training starts. It checks that all required bilingual and monolingual files exist and counts their lines. It also validates the line-by-line alignment of the Moses-format parallel files.

In [14]:
from importlib.metadata import version

def count_lines_in_text_file(file_path: Path) -> int:
    with file_path.open("r", encoding="utf-8") as input_file:
        return sum(1 for _ in input_file)

def validate_environment_setup() -> dict:
    required_input_files = {
        "parallel_training_source": parallel_training_source_path,
        "parallel_training_target": parallel_training_target_path,
        "development_source": development_source_path,
        "development_target": development_target_path,
        "test_source": test_source_path,
        "test_target": test_target_path,
        "english_monolingual": english_monolingual_path,
        "spanish_monolingual": spanish_monolingual_path,
    }

    for file_description, file_path in required_input_files.items():
        if not file_path.exists():
            raise FileNotFoundError(
                f"Missing required file for Section 9: {file_description} -> {file_path}"
            )

    line_count_summary = {
        "parallel_training_source_lines": count_lines_in_text_file(parallel_training_source_path),
        "parallel_training_target_lines": count_lines_in_text_file(parallel_training_target_path),
        "development_source_lines": count_lines_in_text_file(development_source_path),
        "development_target_lines": count_lines_in_text_file(development_target_path),
        "test_source_lines": count_lines_in_text_file(test_source_path),
        "test_target_lines": count_lines_in_text_file(test_target_path),
        "english_monolingual_lines": count_lines_in_text_file(english_monolingual_path),
        "spanish_monolingual_lines": count_lines_in_text_file(spanish_monolingual_path),
    }

    assert line_count_summary["parallel_training_source_lines"] == parallel_training_size, \
        "The English training file does not contain the required 1000 lines."
    assert line_count_summary["parallel_training_target_lines"] == parallel_training_size, \
        "The Spanish training file does not contain the required 1000 lines."

    assert line_count_summary["development_source_lines"] == development_parallel_size, \
        "The English development file does not contain the required 200 lines."
    assert line_count_summary["development_target_lines"] == development_parallel_size, \
        "The Spanish development file does not contain the required 200 lines."

    assert line_count_summary["test_source_lines"] == test_parallel_size, \
        "The English test file does not contain the required 200 lines."
    assert line_count_summary["test_target_lines"] == test_parallel_size, \
        "The Spanish test file does not contain the required 200 lines."

    assert line_count_summary["english_monolingual_lines"] == english_monolingual_size, \
        "The English monolingual file does not contain the required 1000 lines."
    assert line_count_summary["spanish_monolingual_lines"] == spanish_monolingual_size, \
        "The Spanish monolingual file does not contain the required 1000 lines."

    assert line_count_summary["parallel_training_source_lines"] == line_count_summary["parallel_training_target_lines"], \
        "Training Moses files are misaligned."
    assert line_count_summary["development_source_lines"] == line_count_summary["development_target_lines"], \
        "Development Moses files are misaligned."
    assert line_count_summary["test_source_lines"] == line_count_summary["test_target_lines"], \
        "Test Moses files are misaligned."

    return {
        "required_files_exist": True,
        "expected_line_counts_match": True,
        "training_moses_alignment_ok": True,
        "development_moses_alignment_ok": True,
        "test_moses_alignment_ok": True,
        "forward_output_directory_exists": forward_model_output_directory.exists(),
        "backward_output_directory_exists": backward_model_output_directory.exists(),
        **line_count_summary,
    }

environment_validation_summary = validate_environment_setup()

print("Installed package versions:")
print(f" - transformers : {version('transformers')}")
print(f" - datasets     : {version('datasets')}")
print(f" - evaluate     : {version('evaluate')}")
print(f" - sacrebleu    : {version('sacrebleu')}")
print(f" - sacremoses   : {version('sacremoses')}")
print(f" - accelerate   : {version('accelerate')}")

print("\nEnvironment validation summary:")
for summary_key, summary_value in environment_validation_summary.items():
    print(f"{summary_key}: {summary_value}")

if not torch.cuda.is_available():
    print("\nWarning: GPU is not available.")

Installed package versions:
 - transformers : 4.46.3
 - datasets     : 3.1.0
 - evaluate     : 0.4.3
 - sacrebleu    : 2.4.3
 - sacremoses   : 0.1.1
 - accelerate   : 1.6.0

Environment validation summary:
required_files_exist: True
expected_line_counts_match: True
training_moses_alignment_ok: True
development_moses_alignment_ok: True
test_moses_alignment_ok: True
forward_output_directory_exists: True
backward_output_directory_exists: True
parallel_training_source_lines: 1000
parallel_training_target_lines: 1000
development_source_lines: 200
development_target_lines: 200
test_source_lines: 200
test_target_lines: 200
english_monolingual_lines: 1000
spanish_monolingual_lines: 1000


We load the main libraries and translation components that will be used in the following stages. In addition, we define small reusable helpers for loading seq2seq models and translation pipelines, and we run a simple tokenization smoke test to confirm that the environment is functioning correctly.

In [15]:
from datasets import Dataset
import evaluate

from transformers import (
    AutoConfig,
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    pipeline,
)

bleu_metric = evaluate.load("sacrebleu")

forward_tokenizer = AutoTokenizer.from_pretrained(forward_translation_model_name)
backward_tokenizer = AutoTokenizer.from_pretrained(backward_translation_model_name)

forward_model_configuration = AutoConfig.from_pretrained(forward_translation_model_name)
backward_model_configuration = AutoConfig.from_pretrained(backward_translation_model_name)

def load_seq2seq_model_for_training(pretrained_model_name: str) -> AutoModelForSeq2SeqLM:
    loaded_model = AutoModelForSeq2SeqLM.from_pretrained(pretrained_model_name)
    return loaded_model.to(execution_device)

def build_translation_pipeline(pretrained_model_name: str):
    return pipeline(
        task="translation",
        model=pretrained_model_name,
        tokenizer=pretrained_model_name,
        device=pipeline_device,
        batch_size=evaluation_batch_size,
    )

forward_tokenized_example = forward_tokenizer(
    "This is a short environment smoke test.",
    return_tensors="pt"
)
backward_tokenized_example = backward_tokenizer(
    "Esta es una prueba corta del entorno.",
    return_tensors="pt"
)

print("Main dependencies loaded successfully.")
print(f"Forward tokenizer vocabulary size  : {forward_tokenizer.vocab_size}")
print(f"Backward tokenizer vocabulary size : {backward_tokenizer.vocab_size}")
print(f"Forward sample token count         : {forward_tokenized_example['input_ids'].shape[1]}")
print(f"Backward sample token count        : {backward_tokenized_example['input_ids'].shape[1]}")
print("BLEU metric loaded correctly.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

source.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/826k [00:00<?, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

source.spm:   0%|          | 0.00/826k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

Main dependencies loaded successfully.
Forward tokenizer vocabulary size  : 65001
Backward tokenizer vocabulary size : 65001
Forward sample token count         : 9
Backward sample token count        : 9
BLEU metric loaded correctly.


## SECTION 4

Section 4 defines the transition from the final corpus files stored in `/content/drive/MyDrive/anlp/` to the internal sentence-level datasets that the project will use throughout the experiment. This is implemented through a set of loading and validation utilities that explicitly read the required files (`news_commentary_train.en/.es`, `news_commentary_dev.en/.es`, `news_commentary_test.en/.es`, `news_commentary_mono.en`, and `news_commentary_mono.es`), verify file existence, reject empty or malformed content, enforce alignment in the Moses-format bilingual files, and ensure uniqueness in the monolingual corpora. The bilingual train, development, and test subsets are converted into Hugging Face `Dataset` objects with a `translation` field structured by language code, while the English and Spanish monolingual corpora are loaded separately as `Dataset` objects with a `text` field, which makes their later use for synthetic generation explicit. A central function, `load_experimental_datasets`, then checks that the real loaded sizes exactly match the required configuration—1000 authentic training pairs, 200 development pairs, 200 test pairs, and 1000 monolingual sentences per language—so so we make sure to not to assume correctness blindly. The section also introduces a clean functional separation of the data pipeline by creating distinct containers for authentic bilingual data, monolingual generation data, future synthetic bilingual data, and held-out evaluation data.


This cell defines the utility functions that load and validate the experimental corpora before training. Its main purpose is to ensure that all bilingual and monolingual files exist, are correctly formatted, contain no blank or duplicated entries where inappropriate, and match the exact dataset sizes. It converts aligned parallel sentence pairs into Hugging Face `Dataset` objects with a `translation` field and monolingual files into datasets with a `text` field. Finally, the `load_experimental_datasets` function brings together the full loading process for the training, development, test, and monolingual corpora, verifies their integrity and sizes, and returns them in a structured form for the later evaluation.


In [16]:
from datasets import Dataset
from pathlib import Path
from typing import Dict, List, Tuple


def validate_text_file_exists(file_path: Path, file_description: str) -> None:
    file_path = Path(file_path)
    if not file_path.exists():
        raise FileNotFoundError(f"Missing required file for {file_description}: {file_path}")


def validate_non_empty_text_lines(text_lines: List[str], dataset_description: str) -> None:
    if len(text_lines) == 0:
        raise ValueError(f"{dataset_description} is empty.")

    blank_line_indices = [index for index, line in enumerate(text_lines) if line.strip() == ""]
    if blank_line_indices:
        raise ValueError(
            f"{dataset_description} contains blank lines at indices: {blank_line_indices[:10]}"
        )


def validate_unique_text_lines(text_lines: List[str], dataset_description: str) -> None:
    if len(text_lines) != len(set(text_lines)):
        raise ValueError(f"{dataset_description} contains duplicate text segments.")


def validate_parallel_sentence_pairs(
    parallel_sentence_pairs: List[Tuple[str, str]],
    dataset_description: str,
) -> None:
    if len(parallel_sentence_pairs) == 0:
        raise ValueError(f"{dataset_description} is empty.")

    blank_pair_indices = [
        index
        for index, (source_sentence, target_sentence) in enumerate(parallel_sentence_pairs)
        if source_sentence.strip() == "" or target_sentence.strip() == ""
    ]
    if blank_pair_indices:
        raise ValueError(
            f"{dataset_description} contains blank parallel segments at indices: {blank_pair_indices[:10]}"
        )

    if len(parallel_sentence_pairs) != len(set(parallel_sentence_pairs)):
        raise ValueError(f"{dataset_description} contains duplicate parallel sentence pairs.")


def validate_dataset_size(actual_size: int, expected_size: int, dataset_description: str) -> None:
    if actual_size != expected_size:
        raise ValueError(
            f"{dataset_description} has {actual_size} examples, but {expected_size} were expected."
        )


def build_translation_dataset_from_parallel_pairs(
    parallel_sentence_pairs: List[Tuple[str, str]],
    source_language_code: str,
    target_language_code: str,
) -> Dataset:
    validate_parallel_sentence_pairs(
        parallel_sentence_pairs,
        dataset_description="Parallel sentence pairs used to build a translation dataset",
    )

    return Dataset.from_dict({
        "translation": [
            {
                source_language_code: source_sentence,
                target_language_code: target_sentence,
            }
            for source_sentence, target_sentence in parallel_sentence_pairs
        ]
    })


def build_parallel_translation_dataset(
    source_file_path: Path,
    target_file_path: Path,
    source_language_code: str,
    target_language_code: str,
    dataset_description: str,
) -> Dataset:
    validate_text_file_exists(source_file_path, f"{dataset_description} source file")
    validate_text_file_exists(target_file_path, f"{dataset_description} target file")

    source_sentences = read_lines(source_file_path)
    target_sentences = read_lines(target_file_path)

    if len(source_sentences) != len(target_sentences):
        raise ValueError(
            f"{dataset_description} is misaligned: "
            f"{source_file_path} has {len(source_sentences)} lines, "
            f"but {target_file_path} has {len(target_sentences)} lines."
        )

    validate_non_empty_text_lines(source_sentences, f"{dataset_description} source side")
    validate_non_empty_text_lines(target_sentences, f"{dataset_description} target side")

    parallel_sentence_pairs = list(zip(source_sentences, target_sentences))
    validate_parallel_sentence_pairs(parallel_sentence_pairs, dataset_description)

    return build_translation_dataset_from_parallel_pairs(
        parallel_sentence_pairs=parallel_sentence_pairs,
        source_language_code=source_language_code,
        target_language_code=target_language_code,
    )


def build_monolingual_text_dataset(
    text_file_path: Path,
    dataset_description: str,
    require_unique_entries: bool = True,
) -> Dataset:
    validate_text_file_exists(text_file_path, dataset_description)

    text_lines = read_lines(text_file_path)
    validate_non_empty_text_lines(text_lines, dataset_description)

    if require_unique_entries:
        validate_unique_text_lines(text_lines, dataset_description)

    return Dataset.from_dict({"text": text_lines})


def load_experimental_datasets(
    parallel_training_source_path: Path,
    parallel_training_target_path: Path,
    development_source_path: Path,
    development_target_path: Path,
    test_source_path: Path,
    test_target_path: Path,
    english_monolingual_path: Path,
    spanish_monolingual_path: Path,
    source_language_code: str,
    target_language_code: str,
    expected_parallel_training_size: int,
    expected_development_size: int,
    expected_test_size: int,
    expected_english_monolingual_size: int,
    expected_spanish_monolingual_size: int,
    verbose: bool = True,
) -> Dict[str, object]:
    authentic_parallel_train_dataset = build_parallel_translation_dataset(
        source_file_path=parallel_training_source_path,
        target_file_path=parallel_training_target_path,
        source_language_code=source_language_code,
        target_language_code=target_language_code,
        dataset_description="Authentic bilingual training dataset",
    )

    development_parallel_dataset = build_parallel_translation_dataset(
        source_file_path=development_source_path,
        target_file_path=development_target_path,
        source_language_code=source_language_code,
        target_language_code=target_language_code,
        dataset_description="Development bilingual dataset",
    )

    test_parallel_dataset = build_parallel_translation_dataset(
        source_file_path=test_source_path,
        target_file_path=test_target_path,
        source_language_code=source_language_code,
        target_language_code=target_language_code,
        dataset_description="Test bilingual dataset",
    )

    english_monolingual_generation_dataset = build_monolingual_text_dataset(
        text_file_path=english_monolingual_path,
        dataset_description="English monolingual generation dataset",
        require_unique_entries=True,
    )

    spanish_monolingual_generation_dataset = build_monolingual_text_dataset(
        text_file_path=spanish_monolingual_path,
        dataset_description="Spanish monolingual generation dataset",
        require_unique_entries=True,
    )

    dataset_size_summary = {
        "authentic_parallel_train_pairs": len(authentic_parallel_train_dataset),
        "development_parallel_pairs": len(development_parallel_dataset),
        "test_parallel_pairs": len(test_parallel_dataset),
        "english_monolingual_sentences": len(english_monolingual_generation_dataset),
        "spanish_monolingual_sentences": len(spanish_monolingual_generation_dataset),
    }

    validate_dataset_size(
        actual_size=dataset_size_summary["authentic_parallel_train_pairs"],
        expected_size=expected_parallel_training_size,
        dataset_description="Authentic bilingual training dataset",
    )
    validate_dataset_size(
        actual_size=dataset_size_summary["development_parallel_pairs"],
        expected_size=expected_development_size,
        dataset_description="Development bilingual dataset",
    )
    validate_dataset_size(
        actual_size=dataset_size_summary["test_parallel_pairs"],
        expected_size=expected_test_size,
        dataset_description="Test bilingual dataset",
    )
    validate_dataset_size(
        actual_size=dataset_size_summary["english_monolingual_sentences"],
        expected_size=expected_english_monolingual_size,
        dataset_description="English monolingual generation dataset",
    )
    validate_dataset_size(
        actual_size=dataset_size_summary["spanish_monolingual_sentences"],
        expected_size=expected_spanish_monolingual_size,
        dataset_description="Spanish monolingual generation dataset",
    )

    if verbose:
        print("All datasets were loaded successfully.")
        print("Verified sizes:")
        for summary_key, summary_value in dataset_size_summary.items():
            print(f" - {summary_key}: {summary_value}")

    return {
        "authentic_parallel_train_dataset": authentic_parallel_train_dataset,
        "development_parallel_dataset": development_parallel_dataset,
        "test_parallel_dataset": test_parallel_dataset,
        "english_monolingual_generation_dataset": english_monolingual_generation_dataset,
        "spanish_monolingual_generation_dataset": spanish_monolingual_generation_dataset,
        "dataset_size_summary": dataset_size_summary,
    }

This cell executes the loading process defined previously and organizes the resulting datasets into clearly separated functional pipelines. Specifically, it calls **load_experimental_datasets** with the paths, language codes, and expected sizes already configured in the project, and then stores the validated outputs in explicit variables for the authentic bilingual training set, the development set, the test set, and the English and Spanish monolingual generation sets. Finally, it creates separate dictionaries for authentic bilingual data, monolingual data, synthetic bilingual data, and evaluation data, with the synthetic containers intentionally initialized as empty lists because no synthetic pairs have been generated yet.

In [17]:
loaded_experimental_data = load_experimental_datasets(
    parallel_training_source_path=parallel_training_source_path,
    parallel_training_target_path=parallel_training_target_path,
    development_source_path=development_source_path,
    development_target_path=development_target_path,
    test_source_path=test_source_path,
    test_target_path=test_target_path,
    english_monolingual_path=english_monolingual_path,
    spanish_monolingual_path=spanish_monolingual_path,
    source_language_code=source_language_code,
    target_language_code=target_language_code,
    expected_parallel_training_size=parallel_training_size,
    expected_development_size=development_parallel_size,
    expected_test_size=test_parallel_size,
    expected_english_monolingual_size=english_monolingual_size,
    expected_spanish_monolingual_size=spanish_monolingual_size,
    verbose=True,
)

authentic_parallel_train_dataset = loaded_experimental_data["authentic_parallel_train_dataset"]
development_parallel_dataset = loaded_experimental_data["development_parallel_dataset"]
test_parallel_dataset = loaded_experimental_data["test_parallel_dataset"]

english_monolingual_generation_dataset = loaded_experimental_data["english_monolingual_generation_dataset"]
spanish_monolingual_generation_dataset = loaded_experimental_data["spanish_monolingual_generation_dataset"]

dataset_size_summary = loaded_experimental_data["dataset_size_summary"]

train_dataset = authentic_parallel_train_dataset
dev_dataset = development_parallel_dataset
test_dataset = test_parallel_dataset

english_monolingual_dataset = english_monolingual_generation_dataset
spanish_monolingual_dataset = spanish_monolingual_generation_dataset

authentic_bilingual_data_pipeline = {
    "parallel_training_dataset": authentic_parallel_train_dataset,
}

monolingual_data_pipeline = {
    "english_monolingual_generation_dataset": english_monolingual_generation_dataset,
    "spanish_monolingual_generation_dataset": spanish_monolingual_generation_dataset,
}

synthetic_bilingual_data_pipeline = {
    "synthetic_parallel_pairs_from_english_monolingual": [],
    "synthetic_parallel_pairs_from_spanish_monolingual": [],
}

evaluation_data_pipeline = {
    "development_parallel_dataset": development_parallel_dataset,
    "test_parallel_dataset": test_parallel_dataset,
}

experimental_data_pipelines = {
    "authentic_bilingual_data_pipeline": authentic_bilingual_data_pipeline,
    "monolingual_data_pipeline": monolingual_data_pipeline,
    "synthetic_bilingual_data_pipeline": synthetic_bilingual_data_pipeline,
    "evaluation_data_pipeline": evaluation_data_pipeline,
}

All datasets were loaded successfully.
Verified sizes:
 - authentic_parallel_train_pairs: 1000
 - development_parallel_pairs: 200
 - test_parallel_pairs: 200
 - english_monolingual_sentences: 1000
 - spanish_monolingual_sentences: 1000


This cell performs an inspection and sanity check of the datasets loaded in Section 4 by printing their validated sizes and representative examples. It reports the counts stored in dataset_size_summary, shows the sizes of the authentic bilingual, monolingual, synthetic, and evaluation pipelines, and confirms that the synthetic containers are still empty at this stage, which is methodologically correct before iterative back-translation begins. It also prints one example from the training, development, and test bilingual datasets, together with one example sentence from each monolingual corpus.

In [18]:
print("DATA LOADING AND DATASET CONSTRUCTION SUMMARY")
print("---------------------------------------------")
for summary_key, summary_value in dataset_size_summary.items():
    print(f"{summary_key}: {summary_value}")

print("\nAUTHENTIC BILINGUAL DATA PIPELINE")
print("--------------------------------")
print("Training dataset size:", len(authentic_bilingual_data_pipeline["parallel_training_dataset"]))

print("\nMONOLINGUAL DATA PIPELINE")
print("-------------------------")
print("English monolingual generation dataset size:",
      len(monolingual_data_pipeline["english_monolingual_generation_dataset"]))
print("Spanish monolingual generation dataset size:",
      len(monolingual_data_pipeline["spanish_monolingual_generation_dataset"]))

print("\nSYNTHETIC BILINGUAL DATA PIPELINE")
print("---------------------------------")
print("Synthetic pairs from English monolingual:",
      len(synthetic_bilingual_data_pipeline["synthetic_parallel_pairs_from_english_monolingual"]))
print("Synthetic pairs from Spanish monolingual:",
      len(synthetic_bilingual_data_pipeline["synthetic_parallel_pairs_from_spanish_monolingual"]))

print("\nEVALUATION DATA PIPELINE")
print("------------------------")
print("Development dataset size:", len(evaluation_data_pipeline["development_parallel_dataset"]))
print("Test dataset size       :", len(evaluation_data_pipeline["test_parallel_dataset"]))

print("\nExample authentic bilingual training pair:")
print(authentic_parallel_train_dataset[0])

print("\nExample development pair:")
print(development_parallel_dataset[0])

print("\nExample test pair:")
print(test_parallel_dataset[0])

print("\nExample English monolingual sentence:")
print(english_monolingual_generation_dataset[0])

print("\nExample Spanish monolingual sentence:")
print(spanish_monolingual_generation_dataset[0])


DATA LOADING AND DATASET CONSTRUCTION SUMMARY
---------------------------------------------
authentic_parallel_train_pairs: 1000
development_parallel_pairs: 200
test_parallel_pairs: 200
english_monolingual_sentences: 1000
spanish_monolingual_sentences: 1000

AUTHENTIC BILINGUAL DATA PIPELINE
--------------------------------
Training dataset size: 1000

MONOLINGUAL DATA PIPELINE
-------------------------
English monolingual generation dataset size: 1000
Spanish monolingual generation dataset size: 1000

SYNTHETIC BILINGUAL DATA PIPELINE
---------------------------------
Synthetic pairs from English monolingual: 0
Synthetic pairs from Spanish monolingual: 0

EVALUATION DATA PIPELINE
------------------------
Development dataset size: 200
Test dataset size       : 200

Example authentic bilingual training pair:
{'translation': {'en': 'But caste-bound India’s record of exclusion is worse.', 'es': 'Pero el récord de exclusión de la India ceñida a las castas es peor.'}}

Example development p

## SECTION 5

From this section, the project becomes a genuinely bidirectional machine translation system by explicitly loading the two separate pretrained Helsinki-NLP bilingual models, the one for English to Spanish (`Helsinki-NLP/opus-mt-en-es`) and the one for Spanish to English (`Helsinki-NLP/opus-mt-es-en`), together with their corresponding tokenizers and configurations. The implementation is structured through reusable helper functions that parse and validate the translation direction encoded in each model name, ensure that each model is associated with the expected source and target languages, reuse previously loaded tokenizers and configurations when possible, and finally load the pretrained seq2seq weights. The resulting components are stored in two independent directional bundles and in a central registry, which preserves a strict separation between both translation flows throughout the project. This design is especially important because iterative back-translation requires the English to Spanish model to later generate synthetic Spanish from English monolingual data, while the Spanish to English model generates synthetic English from Spanish monolingual data.

This cell defines the helpers used to manage the pretrained translation models. It validates the translation direction encoded in each Helsinki-NLP model name, loads or reuses tokenizers and configurations, loads the seq2seq model itself, and also provides a small function to generate a translation.

In [19]:
import re
from typing import Any, Dict, Optional

from transformers import AutoConfig, AutoModelForSeq2SeqLM, AutoTokenizer

SUPPORTED_HELSINKI_TRANSLATION_MODEL_PATTERN = re.compile(
    r"^Helsinki-NLP/opus-mt-([a-z]{2,3})-([a-z]{2,3})$"
)

def parse_helsinki_translation_direction(pretrained_model_name: str) -> Dict[str, str]:
    model_name_match = SUPPORTED_HELSINKI_TRANSLATION_MODEL_PATTERN.match(pretrained_model_name)

    if model_name_match is None:
        raise ValueError(
            f"Unsupported Helsinki model name format: {pretrained_model_name}. "
            "Expected a name such as 'Helsinki-NLP/opus-mt-en-es'."
        )

    parsed_source_language_code, parsed_target_language_code = model_name_match.groups()

    return {
        "source_language_code": parsed_source_language_code,
        "target_language_code": parsed_target_language_code,
    }

def validate_pretrained_translation_direction(
    pretrained_model_name: str,
    expected_source_language_code: str,
    expected_target_language_code: str,
) -> Dict[str, str]:
    parsed_direction = parse_helsinki_translation_direction(pretrained_model_name)

    if parsed_direction["source_language_code"] != expected_source_language_code:
        raise ValueError(
            f"Model direction mismatch for {pretrained_model_name}: "
            f"expected source language '{expected_source_language_code}', "
            f"but found '{parsed_direction['source_language_code']}'."
        )

    if parsed_direction["target_language_code"] != expected_target_language_code:
        raise ValueError(
            f"Model direction mismatch for {pretrained_model_name}: "
            f"expected target language '{expected_target_language_code}', "
            f"but found '{parsed_direction['target_language_code']}'."
        )

    return parsed_direction

def reuse_or_load_tokenizer(
    pretrained_model_name: str,
    existing_tokenizer: Optional[Any] = None,
):
    if (
        existing_tokenizer is not None
        and getattr(existing_tokenizer, "name_or_path", None) == pretrained_model_name
    ):
        return existing_tokenizer

    return AutoTokenizer.from_pretrained(pretrained_model_name)

def reuse_or_load_model_configuration(
    pretrained_model_name: str,
    existing_configuration: Optional[Any] = None,
):
    if (
        existing_configuration is not None
        and getattr(existing_configuration, "name_or_path", None) == pretrained_model_name
    ):
        return existing_configuration

    return AutoConfig.from_pretrained(pretrained_model_name)

def load_pretrained_seq2seq_model_for_bidirectional_mt(
    pretrained_model_name: str,
    model_storage_device: str = "cpu",
    put_model_in_eval_mode: bool = True,
) -> AutoModelForSeq2SeqLM:
    loaded_model = AutoModelForSeq2SeqLM.from_pretrained(pretrained_model_name)
    loaded_model = loaded_model.to(model_storage_device)

    if put_model_in_eval_mode:
        loaded_model.eval()

    return loaded_model

def build_directional_pretrained_translation_bundle(
    pretrained_model_name: str,
    expected_source_language_code: str,
    expected_target_language_code: str,
    directional_role_name: str,
    existing_tokenizer: Optional[Any] = None,
    existing_configuration: Optional[Any] = None,
    model_storage_device: str = "cpu",
) -> Dict[str, Any]:
    validated_direction = validate_pretrained_translation_direction(
        pretrained_model_name=pretrained_model_name,
        expected_source_language_code=expected_source_language_code,
        expected_target_language_code=expected_target_language_code,
    )

    directional_tokenizer = reuse_or_load_tokenizer(
        pretrained_model_name=pretrained_model_name,
        existing_tokenizer=existing_tokenizer,
    )

    directional_configuration = reuse_or_load_model_configuration(
        pretrained_model_name=pretrained_model_name,
        existing_configuration=existing_configuration,
    )

    directional_model = load_pretrained_seq2seq_model_for_bidirectional_mt(
        pretrained_model_name=pretrained_model_name,
        model_storage_device=model_storage_device,
        put_model_in_eval_mode=True,
    )

    return {
        "directional_role_name": directional_role_name,
        "model_name": pretrained_model_name,
        "source_language_code": validated_direction["source_language_code"],
        "target_language_code": validated_direction["target_language_code"],
        "tokenizer": directional_tokenizer,
        "configuration": directional_configuration,
        "model": directional_model,
    }

def generate_translation_with_pretrained_components(
    input_text: str,
    pretrained_model,
    pretrained_tokenizer,
    max_input_length: int = 128,
    max_new_tokens: int = 64,
) -> str:
    tokenized_inputs = pretrained_tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_length,
    )

    with torch.no_grad():
        generated_token_ids = pretrained_model.generate(
            **tokenized_inputs,
            max_new_tokens=max_new_tokens,
        )

    decoded_translation = pretrained_tokenizer.decode(
        generated_token_ids[0],
        skip_special_tokens=True,
    )

    return decoded_translation

This code loads the two pretrained bilingual models, the one for English to Spanish and the one for Spanish to English. It builds a structured bundle for each direction, extracts the main components into explicit variables, creates compatibility aliases, and stores everything in a central registry.

In [20]:
existing_forward_tokenizer = globals().get("forward_tokenizer", None)
existing_backward_tokenizer = globals().get("backward_tokenizer", None)

existing_forward_configuration = globals().get("forward_model_configuration", None)
existing_backward_configuration = globals().get("backward_model_configuration", None)

english_to_spanish_pretrained_components = build_directional_pretrained_translation_bundle(
    pretrained_model_name=forward_translation_model_name,
    expected_source_language_code=source_language_code,
    expected_target_language_code=target_language_code,
    directional_role_name="english_to_spanish",
    existing_tokenizer=existing_forward_tokenizer,
    existing_configuration=existing_forward_configuration,
    model_storage_device="cpu",
)

spanish_to_english_pretrained_components = build_directional_pretrained_translation_bundle(
    pretrained_model_name=backward_translation_model_name,
    expected_source_language_code=target_language_code,
    expected_target_language_code=source_language_code,
    directional_role_name="spanish_to_english",
    existing_tokenizer=existing_backward_tokenizer,
    existing_configuration=existing_backward_configuration,
    model_storage_device="cpu",
)

english_to_spanish_pretrained_model = english_to_spanish_pretrained_components["model"]
english_to_spanish_pretrained_tokenizer = english_to_spanish_pretrained_components["tokenizer"]
english_to_spanish_pretrained_configuration = english_to_spanish_pretrained_components["configuration"]

spanish_to_english_pretrained_model = spanish_to_english_pretrained_components["model"]
spanish_to_english_pretrained_tokenizer = spanish_to_english_pretrained_components["tokenizer"]
spanish_to_english_pretrained_configuration = spanish_to_english_pretrained_components["configuration"]

forward_pretrained_translation_model = english_to_spanish_pretrained_model
forward_pretrained_translation_tokenizer = english_to_spanish_pretrained_tokenizer
forward_pretrained_translation_configuration = english_to_spanish_pretrained_configuration

backward_pretrained_translation_model = spanish_to_english_pretrained_model
backward_pretrained_translation_tokenizer = spanish_to_english_pretrained_tokenizer
backward_pretrained_translation_configuration = spanish_to_english_pretrained_configuration

bidirectional_pretrained_translation_component_registry = {
    "english_to_spanish": english_to_spanish_pretrained_components,
    "spanish_to_english": spanish_to_english_pretrained_components,
}

print("Pretrained translation systems loaded successfully.\n")

print("Loaded pretrained models:")
print(f" - English -> Spanish : {english_to_spanish_pretrained_components['model_name']}")
print(f" - Spanish -> English : {spanish_to_english_pretrained_components['model_name']}")

print("\nLoaded tokenizers:")
print(f" - English -> Spanish tokenizer : {english_to_spanish_pretrained_tokenizer.name_or_path}")
print(f" - Spanish -> English tokenizer : {spanish_to_english_pretrained_tokenizer.name_or_path}")

pytorch_model.bin:   0%|          | 0.00/312M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/312M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/312M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Pretrained translation systems loaded successfully.

Loaded pretrained models:
 - English -> Spanish : Helsinki-NLP/opus-mt-en-es
 - Spanish -> English : Helsinki-NLP/opus-mt-es-en

Loaded tokenizers:
 - English -> Spanish tokenizer : Helsinki-NLP/opus-mt-en-es
 - Spanish -> English tokenizer : Helsinki-NLP/opus-mt-es-en


This cell performs a simple inspection and smoke test of the loaded models. It checks where each model is stored and translates one short sentence in each direction.

In [21]:
english_to_spanish_pretrained_model_device = next(
    english_to_spanish_pretrained_model.parameters()
).device
spanish_to_english_pretrained_model_device = next(
    spanish_to_english_pretrained_model.parameters()
).device

pretrained_model_loading_smoke_test_summary = {
    "english_to_spanish_input": "This is a short bidirectional translation smoke test.",
    "spanish_to_english_input": "Esta es una prueba corta de traducción bidireccional.",
}

pretrained_model_loading_smoke_test_summary["english_to_spanish_output"] = (
    generate_translation_with_pretrained_components(
        input_text=pretrained_model_loading_smoke_test_summary["english_to_spanish_input"],
        pretrained_model=english_to_spanish_pretrained_model,
        pretrained_tokenizer=english_to_spanish_pretrained_tokenizer,
        max_input_length=max_source_sequence_length,
        max_new_tokens=48,
    )
)

pretrained_model_loading_smoke_test_summary["spanish_to_english_output"] = (
    generate_translation_with_pretrained_components(
        input_text=pretrained_model_loading_smoke_test_summary["spanish_to_english_input"],
        pretrained_model=spanish_to_english_pretrained_model,
        pretrained_tokenizer=spanish_to_english_pretrained_tokenizer,
        max_input_length=max_source_sequence_length,
        max_new_tokens=48,
    )
)

print("BIDIRECTIONAL PRETRAINED MODEL INSPECTION")
print("-----------------------------------------")
print(f"English -> Spanish model device : {english_to_spanish_pretrained_model_device}")
print(f"Spanish -> English model device : {spanish_to_english_pretrained_model_device}")

print("\nEnglish -> Spanish system")
print("-------------------------")
print("Model name       :", english_to_spanish_pretrained_components["model_name"])
print("Source language  :", english_to_spanish_pretrained_components["source_language_code"])
print("Target language  :", english_to_spanish_pretrained_components["target_language_code"])
print("Vocabulary size  :", english_to_spanish_pretrained_tokenizer.vocab_size)

print("\nSpanish -> English system")
print("-------------------------")
print("Model name       :", spanish_to_english_pretrained_components["model_name"])
print("Source language  :", spanish_to_english_pretrained_components["source_language_code"])
print("Target language  :", spanish_to_english_pretrained_components["target_language_code"])
print("Vocabulary size  :", spanish_to_english_pretrained_tokenizer.vocab_size)

print("\nSMOKE TEST OUTPUTS")
print("------------------")
print("English input :", pretrained_model_loading_smoke_test_summary["english_to_spanish_input"])
print("Spanish output:", pretrained_model_loading_smoke_test_summary["english_to_spanish_output"])

print("\nSpanish input :", pretrained_model_loading_smoke_test_summary["spanish_to_english_input"])
print("English output:", pretrained_model_loading_smoke_test_summary["spanish_to_english_output"])


BIDIRECTIONAL PRETRAINED MODEL INSPECTION
-----------------------------------------
English -> Spanish model device : cpu
Spanish -> English model device : cpu

English -> Spanish system
-------------------------
Model name       : Helsinki-NLP/opus-mt-en-es
Source language  : en
Target language  : es
Vocabulary size  : 65001

Spanish -> English system
-------------------------
Model name       : Helsinki-NLP/opus-mt-es-en
Source language  : es
Target language  : en
Vocabulary size  : 65001

SMOKE TEST OUTPUTS
------------------
English input : This is a short bidirectional translation smoke test.
Spanish output: Esta es una breve prueba de humo de traducción bidireccional.

Spanish input : Esta es una prueba corta de traducción bidireccional.
English output: This is a short two-way translation test.


## SECTION 6

Section 6 implements the baseline supervised fine-tuning stage, that is, 'Iteration 0', as the operational starting point of the bidirectional iterative back-translation pipeline. This stage is structured through the computation development BLEU, which will be reused across the project. The authentic 1,000-pair parallel training corpus is reused directly for English to Spanish and programmatically inverted for Spanish to English, while the 200-pair development set is handled in the same directional way; importantly, no monolingual corpora or synthetic sentence pairs are incorporated at this stage, and the test set remains completely excluded. The core training routine, `fine_tune_directional_baseline_model`, loads each Helsinki-NLP model, tokenizes the corresponding train/dev datasets, fine-tunes with `Seq2SeqTrainer`, applies early stopping and model selection based on development BLEU, and saves both the best checkpoint and a JSON summary of the run.


The following cell code defines the helper functions that support the baseline supervised fine-tuning stage of 'Iteration 0'. It includes functions to reverse the bilingual dataset for the Spanish to English direction, summarize the corpus used in the baseline, build direction-specific tokenization and BLEU computation functions, tokenize the train and development datasets, clean output folders, release memory, and execute the full fine-tuning workflow with **Seq2SeqTrainer**.

In [22]:
import gc
import json
import shutil
from pathlib import Path
from typing import Callable, Dict, Tuple

import numpy as np
import evaluate
from datasets import Dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

baseline_iteration_name = "iteration_0_baseline"
baseline_max_num_epochs = 10

baseline_forward_model_output_directory = forward_model_output_directory / baseline_iteration_name
baseline_backward_model_output_directory = backward_model_output_directory / baseline_iteration_name


def reverse_translation_dataset_direction(
    parallel_translation_dataset: Dataset,
    current_source_language_code: str,
    current_target_language_code: str,
) -> Dataset:
    reversed_parallel_sentence_pairs = [
        (
            example["translation"][current_target_language_code],
            example["translation"][current_source_language_code],
        )
        for example in parallel_translation_dataset
    ]

    return build_translation_dataset_from_parallel_pairs(
        parallel_sentence_pairs=reversed_parallel_sentence_pairs,
        source_language_code=current_target_language_code,
        target_language_code=current_source_language_code,
    )


def build_iteration_zero_corpus_summary(authentic_parallel_pair_count: int) -> Dict[str, int]:
    if authentic_parallel_pair_count <= 0:
        raise ValueError("The authentic parallel corpus size must be positive for Iteration 0.")

    return {
        "iteration_name": baseline_iteration_name,
        "authentic_parallel_pairs": authentic_parallel_pair_count,
        "synthetic_parallel_pairs_added": 0,
        "total_training_pairs_per_direction": authentic_parallel_pair_count,
    }


def print_iteration_zero_corpus_summary(iteration_zero_corpus_summary: Dict[str, int]) -> None:
    print("ITERATION 0 / BASELINE CORPUS SIZE")
    print("----------------------------------")
    print(f"Iteration name                  : {iteration_zero_corpus_summary['iteration_name']}")
    print(f"Authentic parallel pairs        : {iteration_zero_corpus_summary['authentic_parallel_pairs']}")
    print(f"Synthetic parallel pairs added  : {iteration_zero_corpus_summary['synthetic_parallel_pairs_added']}")
    print(f"Total training pairs/direction  : {iteration_zero_corpus_summary['total_training_pairs_per_direction']}")
    print("\nImportant:")
    print("- Only authentic bilingual data are used at this stage.")
    print("- English and Spanish monolingual corpora are NOT used yet.")
    print("- The test set remains untouched and reserved for the final evaluation.")


def build_directional_tokenization_function(
    directional_tokenizer,
    source_language_code: str,
    target_language_code: str,
    max_source_sequence_length: int,
    max_target_sequence_length: int,
) -> Callable:
    def tokenize_batch(batch_examples):
        source_texts = [
            example[source_language_code]
            for example in batch_examples["translation"]
        ]
        target_texts = [
            example[target_language_code]
            for example in batch_examples["translation"]
        ]

        model_inputs = directional_tokenizer(
            text=source_texts,
            max_length=max_source_sequence_length,
            truncation=True,
        )

        label_inputs = directional_tokenizer(
            text_target=target_texts,
            max_length=max_target_sequence_length,
            truncation=True,
        )

        model_inputs["labels"] = label_inputs["input_ids"]
        return model_inputs

    return tokenize_batch


def postprocess_bleu_texts(predictions, references):
    cleaned_predictions = [prediction.strip() for prediction in predictions]
    cleaned_references = [[reference.strip()] for reference in references]
    return cleaned_predictions, cleaned_references


def build_directional_bleu_metric_function(directional_tokenizer) -> Callable:
    directional_bleu_metric = evaluate.load("sacrebleu")

    def compute_directional_bleu(eval_predictions):
        predicted_token_ids, label_token_ids = eval_predictions

        if isinstance(predicted_token_ids, tuple):
            predicted_token_ids = predicted_token_ids[0]

        decoded_predictions = directional_tokenizer.batch_decode(
            predicted_token_ids,
            skip_special_tokens=True,
        )

        label_token_ids = np.where(
            label_token_ids != -100,
            label_token_ids,
            directional_tokenizer.pad_token_id,
        )

        decoded_references = directional_tokenizer.batch_decode(
            label_token_ids,
            skip_special_tokens=True,
        )

        decoded_predictions, decoded_references = postprocess_bleu_texts(
            decoded_predictions,
            decoded_references,
        )

        bleu_result = directional_bleu_metric.compute(
            predictions=decoded_predictions,
            references=decoded_references,
        )

        prediction_lengths = [
            np.count_nonzero(prediction != directional_tokenizer.pad_token_id)
            for prediction in predicted_token_ids
        ]

        return {
            "bleu": round(float(bleu_result["score"]), 4),
            "gen_len": round(float(np.mean(prediction_lengths)), 4),
        }

    return compute_directional_bleu


def tokenize_datasets_for_direction(
    train_parallel_dataset: Dataset,
    development_parallel_dataset: Dataset,
    directional_tokenization_function: Callable,
) -> Tuple[Dataset, Dataset]:
    tokenized_train_parallel_dataset = train_parallel_dataset.map(
        directional_tokenization_function,
        batched=True,
        remove_columns=train_parallel_dataset.column_names,
        desc="Tokenizing baseline training dataset",
    )

    tokenized_development_parallel_dataset = development_parallel_dataset.map(
        directional_tokenization_function,
        batched=True,
        remove_columns=development_parallel_dataset.column_names,
        desc="Tokenizing baseline development dataset",
    )

    return tokenized_train_parallel_dataset, tokenized_development_parallel_dataset


def prepare_clean_output_directory(output_directory: Path) -> None:
    output_directory = Path(output_directory)
    if output_directory.exists():
        shutil.rmtree(output_directory)
    output_directory.mkdir(parents=True, exist_ok=True)


def clear_torch_memory() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def fine_tune_directional_baseline_model(
    stage_name: str,
    pretrained_model_name: str,
    directional_tokenizer,
    train_parallel_dataset: Dataset,
    development_parallel_dataset: Dataset,
    source_language_code: str,
    target_language_code: str,
    output_directory: Path,
) -> Dict[str, object]:
    clear_torch_memory()
    prepare_clean_output_directory(output_directory)

    print(f"STARTING {stage_name}")
    print("-" * len(f"STARTING {stage_name}"))
    print(f"Model name              : {pretrained_model_name}")
    print(f"Direction               : {source_language_code} -> {target_language_code}")
    print(f"Training pairs          : {len(train_parallel_dataset)}")
    print(f"Development pairs       : {len(development_parallel_dataset)}")
    print("Synthetic data used     : No")
    print("Monolingual data used   : No")
    print("Evaluation split used   : Development only")
    print("Test split used         : No\n")

    directional_tokenization_function = build_directional_tokenization_function(
        directional_tokenizer=directional_tokenizer,
        source_language_code=source_language_code,
        target_language_code=target_language_code,
        max_source_sequence_length=max_source_sequence_length,
        max_target_sequence_length=max_target_sequence_length,
    )

    tokenized_train_parallel_dataset, tokenized_development_parallel_dataset = tokenize_datasets_for_direction(
        train_parallel_dataset=train_parallel_dataset,
        development_parallel_dataset=development_parallel_dataset,
        directional_tokenization_function=directional_tokenization_function,
    )

    directional_model = AutoModelForSeq2SeqLM.from_pretrained(pretrained_model_name).to(execution_device)
    directional_data_collator = DataCollatorForSeq2Seq(
        tokenizer=directional_tokenizer,
        model=directional_model,
    )

    directional_compute_metrics = build_directional_bleu_metric_function(
        directional_tokenizer=directional_tokenizer
    )

    directional_training_arguments = Seq2SeqTrainingArguments(
        output_dir=str(output_directory),
        overwrite_output_dir=True,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",
        learning_rate=learning_rate,
        per_device_train_batch_size=training_batch_size,
        per_device_eval_batch_size=evaluation_batch_size,
        weight_decay=0.01,
        save_total_limit=2,
        num_train_epochs=baseline_max_num_epochs,
        predict_with_generate=True,
        generation_max_length=max_target_sequence_length,
        fp16=use_fp16_training,
        metric_for_best_model="bleu",
        greater_is_better=True,
        load_best_model_at_end=True,
        report_to="none",
    )

    directional_trainer = Seq2SeqTrainer(
        model=directional_model,
        args=directional_training_arguments,
        train_dataset=tokenized_train_parallel_dataset,
        eval_dataset=tokenized_development_parallel_dataset,
        data_collator=directional_data_collator,
        processing_class=directional_tokenizer,
        compute_metrics=directional_compute_metrics,
        callbacks=[
            EarlyStoppingCallback(
                early_stopping_patience=early_stopping_patience
            )
        ],
    )

    directional_trainer.train()

    final_development_metrics = directional_trainer.evaluate()

    directional_trainer.save_model(str(output_directory))
    directional_tokenizer.save_pretrained(str(output_directory))

    directional_baseline_result = {
        "stage_name": stage_name,
        "model_name": pretrained_model_name,
        "translation_direction": f"{source_language_code}->{target_language_code}",
        "output_directory": str(output_directory),
        "train_pairs": len(train_parallel_dataset),
        "development_pairs": len(development_parallel_dataset),
        "synthetic_parallel_pairs_added": 0,
        "monolingual_data_used": False,
        "test_set_used": False,
        "dev_bleu": round(float(final_development_metrics["eval_bleu"]), 4),
        "dev_gen_len": round(float(final_development_metrics["eval_gen_len"]), 4),
    }

    with (output_directory / "iteration_0_baseline_summary.json").open("w", encoding="utf-8") as summary_file:
        json.dump(directional_baseline_result, summary_file, indent=2, ensure_ascii=False)

    print("\nCompleted baseline fine-tuning successfully.")
    print(f"Saved best model to : {output_directory}")
    print(f"Development BLEU    : {directional_baseline_result['dev_bleu']}")
    print(f"Development gen_len : {directional_baseline_result['dev_gen_len']}\n")

    del directional_trainer
    del directional_data_collator
    del directional_model
    del tokenized_train_parallel_dataset
    del tokenized_development_parallel_dataset
    clear_torch_memory()

    return directional_baseline_result

This code prepares the datasets used in 'Iteration 0' and reports the baseline corpus size. The English to Spanish direction directly reuses the authentic train and development datasets, whereas the Spanish to English direction is created by reversing those same parallel pairs. It also prints the formal corpus summary, confirms through assertions that both directions contain exactly 1,000 training pairs and 200 development pairs, and shows example pairs to clarify how directionality is enforced.

In [23]:
english_to_spanish_baseline_train_dataset = train_dataset
english_to_spanish_baseline_dev_dataset = dev_dataset

spanish_to_english_baseline_train_dataset = reverse_translation_dataset_direction(
    parallel_translation_dataset=train_dataset,
    current_source_language_code=source_language_code,
    current_target_language_code=target_language_code,
)

spanish_to_english_baseline_dev_dataset = reverse_translation_dataset_direction(
    parallel_translation_dataset=dev_dataset,
    current_source_language_code=source_language_code,
    current_target_language_code=target_language_code,
)

forward_baseline_train_dataset = english_to_spanish_baseline_train_dataset
forward_baseline_dev_dataset = english_to_spanish_baseline_dev_dataset

backward_baseline_train_dataset = spanish_to_english_baseline_train_dataset
backward_baseline_dev_dataset = spanish_to_english_baseline_dev_dataset

iteration_zero_corpus_summary = build_iteration_zero_corpus_summary(
    authentic_parallel_pair_count=len(english_to_spanish_baseline_train_dataset)
)

print_iteration_zero_corpus_summary(iteration_zero_corpus_summary)

print("\nBASELINE DATASET CHECKS")
print("-----------------------")
print(f"EN -> ES training pairs : {len(english_to_spanish_baseline_train_dataset)}")
print(f"EN -> ES dev pairs      : {len(english_to_spanish_baseline_dev_dataset)}")
print(f"ES -> EN training pairs : {len(spanish_to_english_baseline_train_dataset)}")
print(f"ES -> EN dev pairs      : {len(spanish_to_english_baseline_dev_dataset)}")

assert len(english_to_spanish_baseline_train_dataset) == parallel_training_size, \
    "The EN->ES baseline training dataset does not contain 1000 authentic pairs."
assert len(english_to_spanish_baseline_dev_dataset) == development_parallel_size, \
    "The EN->ES baseline development dataset does not contain 200 pairs."

assert len(spanish_to_english_baseline_train_dataset) == parallel_training_size, \
    "The ES->EN baseline training dataset does not contain 1000 authentic pairs."
assert len(spanish_to_english_baseline_dev_dataset) == development_parallel_size, \
    "The ES->EN baseline development dataset does not contain 200 pairs."

en_to_es_example = english_to_spanish_baseline_train_dataset[0]["translation"]
es_to_en_example = spanish_to_english_baseline_train_dataset[0]["translation"]

print("\nExample EN -> ES training example (explicit directional view):")
print("Source (EN):", en_to_es_example[source_language_code])
print("Target (ES):", en_to_es_example[target_language_code])

print("\nExample ES -> EN training example (explicit directional view):")
print("Source (ES):", es_to_en_example[target_language_code])
print("Target (EN):", es_to_en_example[source_language_code])

print("\nImportant note:")
print("The dataset keeps the language-labelled keys 'en' and 'es'.")
print("For the ES -> EN baseline, the training direction is enforced later by passing:")
print(f" - source_language_code = '{target_language_code}'")
print(f" - target_language_code = '{source_language_code}'")

ITERATION 0 / BASELINE CORPUS SIZE
----------------------------------
Iteration name                  : iteration_0_baseline
Authentic parallel pairs        : 1000
Synthetic parallel pairs added  : 0
Total training pairs/direction  : 1000

Important:
- Only authentic bilingual data are used at this stage.
- English and Spanish monolingual corpora are NOT used yet.
- The test set remains untouched and reserved for the final evaluation.

BASELINE DATASET CHECKS
-----------------------
EN -> ES training pairs : 1000
EN -> ES dev pairs      : 200
ES -> EN training pairs : 1000
ES -> EN dev pairs      : 200

Example EN -> ES training example (explicit directional view):
Source (EN): But caste-bound India’s record of exclusion is worse.
Target (ES): Pero el récord de exclusión de la India ceñida a las castas es peor.

Example ES -> EN training example (explicit directional view):
Source (ES): Pero el récord de exclusión de la India ceñida a las castas es peor.
Target (EN): But caste-bound In

This code fine-tunes the English to Spanish baseline model for Iteration 0. It calls the general training routine with the selected Helsinki-NLP English to Spanish model, its tokenizer, the authentic bilingual training set, the development set, the correct language direction, and the output directory for the checkpoint. The process includes tokenization, epoch-level evaluation, BLEU-based model selection, and saving the best model.

In [24]:
english_to_spanish_iteration_0_result = fine_tune_directional_baseline_model(
    stage_name="English -> Spanish baseline (Iteration 0)",
    pretrained_model_name=forward_translation_model_name,
    directional_tokenizer=english_to_spanish_pretrained_tokenizer,
    train_parallel_dataset=english_to_spanish_baseline_train_dataset,
    development_parallel_dataset=english_to_spanish_baseline_dev_dataset,
    source_language_code=source_language_code,
    target_language_code=target_language_code,
    output_directory=baseline_forward_model_output_directory,
)

STARTING English -> Spanish baseline (Iteration 0)
--------------------------------------------------
Model name              : Helsinki-NLP/opus-mt-en-es
Direction               : en -> es
Training pairs          : 1000
Development pairs       : 200
Synthetic data used     : No
Monolingual data used   : No
Evaluation split used   : Development only
Test split used         : No



Tokenizing baseline training dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing baseline development dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Bleu,Gen Len
1,0.986700,0.918409,45.744500,29.605000
2,0.827100,0.919377,45.733600,29.620000
3,0.723500,0.929686,45.918100,29.480000
4,0.637000,0.942937,45.938900,29.795000
5,0.569000,0.951391,45.577800,29.990000
6,0.519800,0.965690,45.849700,29.580000
7,0.479000,0.975876,45.419000,29.900000


Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Tr

Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Tr


Completed baseline fine-tuning successfully.
Saved best model to : /content/drive/MyDrive/anlp/iterative_backtranslation_en_to_es/iteration_0_baseline
Development BLEU    : 45.9389
Development gen_len : 29.795



The following code fine-tunes the Spanish to English baseline model, completing the bidirectional initialization. It uses the reversed bilingual datasets prepared earlier together with the Helsinki-NLP Spanish to English pretrained model and its tokenizer. As in the previous cell, the model is trained only on the authentic parallel corpus, evaluated on the development set, and saved after selecting the best checkpoint according to BLEU. Thus, the cell establishes the reverse-direction baseline.

In [25]:
spanish_to_english_iteration_0_result = fine_tune_directional_baseline_model(
    stage_name="Spanish -> English baseline (Iteration 0)",
    pretrained_model_name=backward_translation_model_name,
    directional_tokenizer=spanish_to_english_pretrained_tokenizer,
    train_parallel_dataset=spanish_to_english_baseline_train_dataset,
    development_parallel_dataset=spanish_to_english_baseline_dev_dataset,
    source_language_code=target_language_code,   # es
    target_language_code=source_language_code,   # en
    output_directory=baseline_backward_model_output_directory,
)

STARTING Spanish -> English baseline (Iteration 0)
--------------------------------------------------
Model name              : Helsinki-NLP/opus-mt-es-en
Direction               : es -> en
Training pairs          : 1000
Development pairs       : 200
Synthetic data used     : No
Monolingual data used   : No
Evaluation split used   : Development only
Test split used         : No



Tokenizing baseline training dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing baseline development dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Bleu,Gen Len
1,1.068200,1.000365,48.946600,26.220000
2,0.892000,0.995752,48.911000,26.275000
3,0.793300,0.999451,49.070900,26.135000
4,0.695300,1.008713,49.040800,26.185000
5,0.622200,1.015515,48.884100,26.145000
6,0.576800,1.025672,48.555200,26.140000


Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Tr

Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Tr


Completed baseline fine-tuning successfully.
Saved best model to : /content/drive/MyDrive/anlp/iterative_backtranslation_es_to_en/iteration_0_baseline
Development BLEU    : 49.0709
Development gen_len : 26.135



This cell summarizes and stores the final results of Iteration 0.

In [26]:
baseline_iteration_0_results = {
    "iteration_name": baseline_iteration_name,
    "corpus_summary": iteration_zero_corpus_summary,
    "english_to_spanish": english_to_spanish_iteration_0_result,
    "spanish_to_english": spanish_to_english_iteration_0_result,
}

assert english_to_spanish_iteration_0_result["synthetic_parallel_pairs_added"] == 0
assert spanish_to_english_iteration_0_result["synthetic_parallel_pairs_added"] == 0

assert english_to_spanish_iteration_0_result["monolingual_data_used"] is False
assert spanish_to_english_iteration_0_result["monolingual_data_used"] is False

assert english_to_spanish_iteration_0_result["test_set_used"] is False
assert spanish_to_english_iteration_0_result["test_set_used"] is False

baseline_overall_summary_path = corpus_root_directory / "iteration_0_baseline_overall_summary.json"

with baseline_overall_summary_path.open("w", encoding="utf-8") as summary_file:
    json.dump(baseline_iteration_0_results, summary_file, indent=2, ensure_ascii=False)

print("ITERATION 0 / BASELINE FINAL SUMMARY")
print("------------------------------------")
print(f"Iteration name                     : {baseline_iteration_0_results['iteration_name']}")
print(f"Authentic parallel pairs          : {baseline_iteration_0_results['corpus_summary']['authentic_parallel_pairs']}")
print(f"Synthetic parallel pairs added    : {baseline_iteration_0_results['corpus_summary']['synthetic_parallel_pairs_added']}")
print(f"Total training pairs / direction  : {baseline_iteration_0_results['corpus_summary']['total_training_pairs_per_direction']}")

print("\nDIRECTIONAL BASELINE RESULTS")
print("----------------------------")
print("English -> Spanish")
print(f" - Model name       : {english_to_spanish_iteration_0_result['model_name']}")
print(f" - Direction        : {english_to_spanish_iteration_0_result['translation_direction']}")
print(f" - Train pairs      : {english_to_spanish_iteration_0_result['train_pairs']}")
print(f" - Dev pairs        : {english_to_spanish_iteration_0_result['development_pairs']}")
print(f" - Dev BLEU         : {english_to_spanish_iteration_0_result['dev_bleu']}")
print(f" - Dev gen_len      : {english_to_spanish_iteration_0_result['dev_gen_len']}")

print("\nSpanish -> English")
print(f" - Model name       : {spanish_to_english_iteration_0_result['model_name']}")
print(f" - Direction        : {spanish_to_english_iteration_0_result['translation_direction']}")
print(f" - Train pairs      : {spanish_to_english_iteration_0_result['train_pairs']}")
print(f" - Dev pairs        : {spanish_to_english_iteration_0_result['development_pairs']}")
print(f" - Dev BLEU         : {spanish_to_english_iteration_0_result['dev_bleu']}")
print(f" - Dev gen_len      : {spanish_to_english_iteration_0_result['dev_gen_len']}")

print("\nMETHODOLOGICAL CHECKS")
print("---------------------")
print(" - Synthetic data used     : No")
print(" - Monolingual data used   : No")
print(" - Evaluation split used   : Development only")
print(" - Test split used         : No")

print("\nSaved combined baseline summary to:")
print(baseline_overall_summary_path)

print("Both baseline models have been fine-tuned and evaluated before iterative back-translation begins.")

ITERATION 0 / BASELINE FINAL SUMMARY
------------------------------------
Iteration name                     : iteration_0_baseline
Authentic parallel pairs          : 1000
Synthetic parallel pairs added    : 0
Total training pairs / direction  : 1000

DIRECTIONAL BASELINE RESULTS
----------------------------
English -> Spanish
 - Model name       : Helsinki-NLP/opus-mt-en-es
 - Direction        : en->es
 - Train pairs      : 1000
 - Dev pairs        : 200
 - Dev BLEU         : 45.9389
 - Dev gen_len      : 29.795

Spanish -> English
 - Model name       : Helsinki-NLP/opus-mt-es-en
 - Direction        : es->en
 - Train pairs      : 1000
 - Dev pairs        : 200
 - Dev BLEU         : 49.0709
 - Dev gen_len      : 26.135

METHODOLOGICAL CHECKS
---------------------
 - Synthetic data used     : No
 - Monolingual data used   : No
 - Evaluation split used   : Development only
 - Test split used         : No

Saved combined baseline summary to:
/content/drive/MyDrive/anlp/iteration_0_baseli

## SECTION 7

This section implements Iteration 1 as the first full operational realization of iterative back-translation, moving beyond the purely supervised baseline by allowing monolingual data to influence training through synthetic augmentation. This stage begins from the baseline checkpoints obtained in Iteration 0 rather than from the raw pretrained Helsinki-NLP models: the fine-tuned Spanish to English model is applied to the 1,000-sentence Spanish monolingual corpus to generate synthetic English sentences, which are paired with the original Spanish sentences to create new English–Spanish training pairs, while the fine-tuned English to Spanish baseline model is symmetrically used on the 1,000-sentence English monolingual corpus to generate synthetic Spanish–English pairs for the opposite direction. These synthetic pairs are then concatenated with the authentic 1,000 parallel training pairs to build augmented corpora of 2,000 training pairs per direction, and the code explicitly prints these corpus sizes. Afterward, both directional models are further fine-tuned from their baseline checkpoints on the corresponding mixed authentic-plus-synthetic datasets and evaluated again on the same 200-sentence development set, while the test set remains strictly unused.


In [27]:
import gc
import json
from pathlib import Path
from typing import Dict, List, Tuple

import torch
from datasets import Dataset
from tqdm.auto import tqdm
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

iteration_one_name = "iteration_1"
iteration_one_max_num_epochs = baseline_max_num_epochs

iteration_one_forward_model_output_directory = forward_model_output_directory / iteration_one_name
iteration_one_backward_model_output_directory = backward_model_output_directory / iteration_one_name


def extract_texts_from_monolingual_dataset(monolingual_text_dataset: Dataset) -> List[str]:
    if len(monolingual_text_dataset) == 0:
        raise ValueError("The monolingual dataset is empty.")

    extracted_texts = [example["text"].strip() for example in monolingual_text_dataset]

    blank_text_indices = [
        index for index, text in enumerate(extracted_texts)
        if text == ""
    ]

    if blank_text_indices:
        raise ValueError(
            f"The monolingual dataset contains blank texts at indices: {blank_text_indices[:10]}"
        )

    return extracted_texts


def pair_synthetic_source_sentences_with_original_target_sentences(
    synthetic_source_sentences: List[str],
    original_target_sentences: List[str],
    pairing_description: str,
) -> List[Tuple[str, str]]:
    if len(synthetic_source_sentences) != len(original_target_sentences):
        raise ValueError(
            f"{pairing_description}: the number of synthetic source sentences "
            f"({len(synthetic_source_sentences)}) does not match the number of "
            f"original target sentences ({len(original_target_sentences)})."
        )

    synthetic_parallel_pairs = []
    blank_pair_indices = []

    for pair_index, (synthetic_source_sentence, original_target_sentence) in enumerate(
        zip(synthetic_source_sentences, original_target_sentences)
    ):
        synthetic_source_sentence = synthetic_source_sentence.strip()
        original_target_sentence = original_target_sentence.strip()

        if synthetic_source_sentence == "" or original_target_sentence == "":
            blank_pair_indices.append(pair_index)

        synthetic_parallel_pairs.append(
            (synthetic_source_sentence, original_target_sentence)
        )

    if blank_pair_indices:
        raise ValueError(
            f"{pairing_description}: blank segments detected in the synthetic pairs "
            f"at indices: {blank_pair_indices[:10]}"
        )

    return synthetic_parallel_pairs


def generate_translations_from_checkpoint_in_batches(
    input_texts: List[str],
    model_checkpoint_directory: Path,
    batch_size: int,
    max_input_length: int,
    max_new_tokens: int,
    progress_description: str,
) -> List[str]:
    if len(input_texts) == 0:
        raise ValueError("The list of input texts for generation is empty.")

    model_checkpoint_directory = Path(model_checkpoint_directory)

    if not model_checkpoint_directory.exists():
        raise FileNotFoundError(
            f"Model checkpoint directory not found: {model_checkpoint_directory}"
        )

    clear_torch_memory()

    generation_tokenizer = AutoTokenizer.from_pretrained(str(model_checkpoint_directory))
    generation_model = AutoModelForSeq2SeqLM.from_pretrained(
        str(model_checkpoint_directory)
    ).to(execution_device)
    generation_model.eval()

    generated_texts: List[str] = []

    for batch_start_index in tqdm(
        range(0, len(input_texts), batch_size),
        desc=progress_description,
    ):
        batch_input_texts = input_texts[
            batch_start_index : batch_start_index + batch_size
        ]

        tokenized_batch = generation_tokenizer(
            batch_input_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_input_length,
        )

        tokenized_batch = {
            tensor_name: tensor_value.to(execution_device)
            for tensor_name, tensor_value in tokenized_batch.items()
        }

        with torch.no_grad():
            generated_token_ids = generation_model.generate(
                **tokenized_batch,
                max_new_tokens=max_new_tokens,
            )

        decoded_batch_texts = generation_tokenizer.batch_decode(
            generated_token_ids,
            skip_special_tokens=True,
        )

        generated_texts.extend([decoded_text.strip() for decoded_text in decoded_batch_texts])

    del generation_model
    del generation_tokenizer
    clear_torch_memory()

    return generated_texts


def build_synthetic_parallel_pairs_from_monolingual_dataset(
    monolingual_text_dataset: Dataset,
    generation_model_checkpoint_directory: Path,
    synthetic_source_language_name: str,
    original_target_language_name: str,
    progress_description: str,
) -> List[Tuple[str, str]]:
    original_target_sentences = extract_texts_from_monolingual_dataset(
        monolingual_text_dataset
    )

    synthetic_source_sentences = generate_translations_from_checkpoint_in_batches(
        input_texts=original_target_sentences,
        model_checkpoint_directory=generation_model_checkpoint_directory,
        batch_size=evaluation_batch_size,
        max_input_length=max_source_sequence_length,
        max_new_tokens=max_target_sequence_length,
        progress_description=progress_description,
    )

    synthetic_parallel_pairs = pair_synthetic_source_sentences_with_original_target_sentences(
        synthetic_source_sentences=synthetic_source_sentences,
        original_target_sentences=original_target_sentences,
        pairing_description=(
            f"Synthetic pair construction for {synthetic_source_language_name} -> "
            f"{original_target_language_name}"
        ),
    )

    return synthetic_parallel_pairs


def build_augmented_parallel_dataset_for_direction(
    authentic_parallel_dataset: Dataset,
    synthetic_parallel_pairs: List[Tuple[str, str]],
    source_language_code: str,
    target_language_code: str,
) -> Dataset:
    authentic_parallel_pairs = [
        (
            example["translation"][source_language_code],
            example["translation"][target_language_code],
        )
        for example in authentic_parallel_dataset
    ]

    augmented_parallel_pairs = authentic_parallel_pairs + synthetic_parallel_pairs

    return build_translation_dataset_from_parallel_pairs(
        parallel_sentence_pairs=augmented_parallel_pairs,
        source_language_code=source_language_code,
        target_language_code=target_language_code,
    )


def build_iteration_one_corpus_summary(
    authentic_parallel_pairs_per_direction: int,
    english_to_spanish_synthetic_pairs: int,
    spanish_to_english_synthetic_pairs: int,
) -> Dict[str, int]:
    if authentic_parallel_pairs_per_direction <= 0:
        raise ValueError("The authentic parallel corpus size must be positive.")
    if english_to_spanish_synthetic_pairs <= 0:
        raise ValueError("The EN->ES synthetic pair count must be positive.")
    if spanish_to_english_synthetic_pairs <= 0:
        raise ValueError("The ES->EN synthetic pair count must be positive.")

    return {
        "iteration_name": iteration_one_name,
        "authentic_parallel_pairs_per_direction": authentic_parallel_pairs_per_direction,
        "english_to_spanish_synthetic_pairs": english_to_spanish_synthetic_pairs,
        "spanish_to_english_synthetic_pairs": spanish_to_english_synthetic_pairs,
        "english_to_spanish_total_training_pairs": (
            authentic_parallel_pairs_per_direction + english_to_spanish_synthetic_pairs
        ),
        "spanish_to_english_total_training_pairs": (
            authentic_parallel_pairs_per_direction + spanish_to_english_synthetic_pairs
        ),
    }


def print_iteration_one_corpus_summary(iteration_one_corpus_summary: Dict[str, int]) -> None:
    print("ITERATION 1 CORPUS SIZE")
    print("-----------------------")
    print(f"Iteration name                           : {iteration_one_corpus_summary['iteration_name']}")
    print(f"Authentic parallel pairs / direction     : {iteration_one_corpus_summary['authentic_parallel_pairs_per_direction']}")
    print(f"Synthetic EN-ES pairs added              : {iteration_one_corpus_summary['english_to_spanish_synthetic_pairs']}")
    print(f"Synthetic ES-EN pairs added              : {iteration_one_corpus_summary['spanish_to_english_synthetic_pairs']}")
    print(f"Total EN->ES training pairs              : {iteration_one_corpus_summary['english_to_spanish_total_training_pairs']}")
    print(f"Total ES->EN training pairs              : {iteration_one_corpus_summary['spanish_to_english_total_training_pairs']}")
    print("\nImportant:")
    print("- Iteration 1 starts from the baseline models obtained in Iteration 0.")
    print("- Synthetic data are generated from the monolingual corpora.")
    print("- The test set remains untouched.")


def fine_tune_directional_model_from_checkpoint(
    stage_name: str,
    starting_model_checkpoint_directory: Path,
    directional_tokenizer,
    train_parallel_dataset: Dataset,
    development_parallel_dataset: Dataset,
    source_language_code: str,
    target_language_code: str,
    output_directory: Path,
    iteration_name: str,
    authentic_parallel_pairs_used: int,
    synthetic_parallel_pairs_added: int,
    max_num_epochs: int,
) -> Dict[str, object]:
    clear_torch_memory()
    prepare_clean_output_directory(output_directory)

    starting_model_checkpoint_directory = Path(starting_model_checkpoint_directory)

    if not starting_model_checkpoint_directory.exists():
        raise FileNotFoundError(
            f"Starting model checkpoint directory not found: {starting_model_checkpoint_directory}"
        )

    print(f"STARTING {stage_name}")
    print("-" * len(f"STARTING {stage_name}"))
    print(f"Starting checkpoint        : {starting_model_checkpoint_directory}")
    print(f"Direction                  : {source_language_code} -> {target_language_code}")
    print(f"Authentic training pairs   : {authentic_parallel_pairs_used}")
    print(f"Synthetic training pairs   : {synthetic_parallel_pairs_added}")
    print(f"Total training pairs       : {len(train_parallel_dataset)}")
    print(f"Development pairs          : {len(development_parallel_dataset)}")
    print("Monolingual data used      : Yes")
    print("Evaluation split used      : Development only")
    print("Test split used            : No\n")

    directional_tokenization_function = build_directional_tokenization_function(
        directional_tokenizer=directional_tokenizer,
        source_language_code=source_language_code,
        target_language_code=target_language_code,
        max_source_sequence_length=max_source_sequence_length,
        max_target_sequence_length=max_target_sequence_length,
    )

    tokenized_train_parallel_dataset, tokenized_development_parallel_dataset = tokenize_datasets_for_direction(
        train_parallel_dataset=train_parallel_dataset,
        development_parallel_dataset=development_parallel_dataset,
        directional_tokenization_function=directional_tokenization_function,
    )

    directional_model = AutoModelForSeq2SeqLM.from_pretrained(
        str(starting_model_checkpoint_directory)
    ).to(execution_device)

    directional_data_collator = DataCollatorForSeq2Seq(
        tokenizer=directional_tokenizer,
        model=directional_model,
    )

    directional_compute_metrics = build_directional_bleu_metric_function(
        directional_tokenizer=directional_tokenizer
    )

    directional_training_arguments = Seq2SeqTrainingArguments(
        output_dir=str(output_directory),
        overwrite_output_dir=True,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",
        learning_rate=learning_rate,
        per_device_train_batch_size=training_batch_size,
        per_device_eval_batch_size=evaluation_batch_size,
        weight_decay=0.01,
        save_total_limit=2,
        num_train_epochs=max_num_epochs,
        predict_with_generate=True,
        generation_max_length=max_target_sequence_length,
        fp16=use_fp16_training,
        metric_for_best_model="bleu",
        greater_is_better=True,
        load_best_model_at_end=True,
        report_to="none",
    )

    directional_trainer = Seq2SeqTrainer(
        model=directional_model,
        args=directional_training_arguments,
        train_dataset=tokenized_train_parallel_dataset,
        eval_dataset=tokenized_development_parallel_dataset,
        data_collator=directional_data_collator,
        processing_class=directional_tokenizer,
        compute_metrics=directional_compute_metrics,
        callbacks=[
            EarlyStoppingCallback(
                early_stopping_patience=early_stopping_patience
            )
        ],
    )

    directional_trainer.train()

    final_development_metrics = directional_trainer.evaluate()

    directional_trainer.save_model(str(output_directory))
    directional_tokenizer.save_pretrained(str(output_directory))

    directional_iteration_result = {
        "stage_name": stage_name,
        "iteration_name": iteration_name,
        "starting_model_checkpoint_directory": str(starting_model_checkpoint_directory),
        "output_directory": str(output_directory),
        "translation_direction": f"{source_language_code}->{target_language_code}",
        "authentic_parallel_pairs_used": authentic_parallel_pairs_used,
        "synthetic_parallel_pairs_added": synthetic_parallel_pairs_added,
        "total_training_pairs": len(train_parallel_dataset),
        "development_pairs": len(development_parallel_dataset),
        "monolingual_data_used": True,
        "test_set_used": False,
        "dev_bleu": round(float(final_development_metrics["eval_bleu"]), 4),
        "dev_gen_len": round(float(final_development_metrics["eval_gen_len"]), 4),
    }

    with (output_directory / f"{iteration_name}_summary.json").open("w", encoding="utf-8") as summary_file:
        json.dump(directional_iteration_result, summary_file, indent=2, ensure_ascii=False)

    print(f"\nCompleted {stage_name} successfully.")
    print(f"Saved best model to : {output_directory}")
    print(f"Development BLEU    : {directional_iteration_result['dev_bleu']}")
    print(f"Development gen_len : {directional_iteration_result['dev_gen_len']}\n")

    del directional_trainer
    del directional_data_collator
    del directional_model
    del tokenized_train_parallel_dataset
    del tokenized_development_parallel_dataset
    clear_torch_memory()

    return directional_iteration_result

The following code prepares the Iteration 1 data. It uses the baseline Spanish to English model to translate the Spanish monolingual corpus and create synthetic English–Spanish pairs, and it uses the baseline English to Spanish model to translate the English monolingual corpus and create synthetic Spanish–English pairs. These synthetic pairs are then merged with the 1,000 authentic bilingual pairs to build two augmented training datasets of 2,000 examples each, while the development sets remain unchanged.

## Synthetic data generation

I repeat this explanation here to make it clear that the synthetic data generation procedure starts from this point onward.

I've implemented a synthetic data generation methodology based on iterative **back-translation**. The main idea is to use monolingual corpora to create additional synthetic parallel sentence pairs that augment the small authentic bilingual training set. Concretely, the Spanish monolingual corpus is translated with the current Spanish to English model to generate synthetic English sentences, which are then paired with the original Spanish sentences for English to Spanish training; symmetrically, the English monolingual corpus is translated with the current English to Spanish model to generate synthetic Spanish sentences, which are paired with the original English sentences for Spanish to English training. This is implemented through functions that load the monolingual datasets, generate translations in batches from the saved checkpoints, build aligned synthetic sentence pairs, and concatenate them with the authentic parallel corpus before each new fine-tuning iteration. The same procedure is repeated across iterations so that synthetic data are regenerated with progressively updated models.

In greater detail, the synthetic data generation stage is organized as a controlled transformation pipeline that preserves sentence alignment and directional consistency at every step. The code first extracts and validates the monolingual texts from the Hugging Face datasets, rejecting empty inputs or blank segments, and then loads the opposite-direction fine-tuned checkpoint from the previous stage to produce translations in mini-batches under the same length constraints used elsewhere in the project. After generation, each synthetic sentence is paired strictly by position with its original monolingual counterpart, so the resulting pairs keep a one-to-one correspondence without altering the order of the source material. These validated synthetic pairs are then converted into translation datasets with the appropriate language-code structure and used to rebuild the augmented training corpus for the new round. It is important to highlight that, from Iteration 2 onward, the project **does not accumulate synthetic** data from earlier rounds; instead, it regenerates a **fresh synthetic corpus** with the most recent opposite-direction model, which makes the procedure **genuinely iterative** rather than a simple one-time augmentation.

In [28]:
english_to_spanish_iteration_1_synthetic_pairs = build_synthetic_parallel_pairs_from_monolingual_dataset(
    monolingual_text_dataset=spanish_monolingual_dataset,
    generation_model_checkpoint_directory=baseline_backward_model_output_directory,
    synthetic_source_language_name="English",
    original_target_language_name="Spanish",
    progress_description=(
        "Iteration 1: translating the Spanish monolingual corpus "
        "with the baseline ES->EN model"
    ),
)

spanish_to_english_iteration_1_synthetic_pairs = build_synthetic_parallel_pairs_from_monolingual_dataset(
    monolingual_text_dataset=english_monolingual_dataset,
    generation_model_checkpoint_directory=baseline_forward_model_output_directory,
    synthetic_source_language_name="Spanish",
    original_target_language_name="English",
    progress_description=(
        "Iteration 1: translating the English monolingual corpus "
        "with the baseline EN->ES model"
    ),
)

english_to_spanish_iteration_1_train_dataset = build_augmented_parallel_dataset_for_direction(
    authentic_parallel_dataset=english_to_spanish_baseline_train_dataset,
    synthetic_parallel_pairs=english_to_spanish_iteration_1_synthetic_pairs,
    source_language_code=source_language_code,  # en
    target_language_code=target_language_code,  # es
)

spanish_to_english_iteration_1_train_dataset = build_augmented_parallel_dataset_for_direction(
    authentic_parallel_dataset=spanish_to_english_baseline_train_dataset,
    synthetic_parallel_pairs=spanish_to_english_iteration_1_synthetic_pairs,
    source_language_code=target_language_code,  # es
    target_language_code=source_language_code,  # en
)

english_to_spanish_iteration_1_dev_dataset = english_to_spanish_baseline_dev_dataset
spanish_to_english_iteration_1_dev_dataset = spanish_to_english_baseline_dev_dataset

iteration_one_corpus_summary = build_iteration_one_corpus_summary(
    authentic_parallel_pairs_per_direction=parallel_training_size,
    english_to_spanish_synthetic_pairs=len(english_to_spanish_iteration_1_synthetic_pairs),
    spanish_to_english_synthetic_pairs=len(spanish_to_english_iteration_1_synthetic_pairs),
)

print_iteration_one_corpus_summary(iteration_one_corpus_summary)

print("\nITERATION 1 DATASET CHECKS")
print("--------------------------")
print(f"EN -> ES synthetic pairs         : {len(english_to_spanish_iteration_1_synthetic_pairs)}")
print(f"ES -> EN synthetic pairs         : {len(spanish_to_english_iteration_1_synthetic_pairs)}")
print(f"EN -> ES total training pairs    : {len(english_to_spanish_iteration_1_train_dataset)}")
print(f"ES -> EN total training pairs    : {len(spanish_to_english_iteration_1_train_dataset)}")
print(f"EN -> ES dev pairs               : {len(english_to_spanish_iteration_1_dev_dataset)}")
print(f"ES -> EN dev pairs               : {len(spanish_to_english_iteration_1_dev_dataset)}")

assert len(english_to_spanish_iteration_1_synthetic_pairs) == spanish_monolingual_size, \
    "The EN->ES synthetic pair count must match the Spanish monolingual corpus size."
assert len(spanish_to_english_iteration_1_synthetic_pairs) == english_monolingual_size, \
    "The ES->EN synthetic pair count must match the English monolingual corpus size."

assert len(english_to_spanish_iteration_1_train_dataset) == parallel_training_size + spanish_monolingual_size, \
    "The EN->ES Iteration 1 training dataset must contain 2000 pairs."
assert len(spanish_to_english_iteration_1_train_dataset) == parallel_training_size + english_monolingual_size, \
    "The ES->EN Iteration 1 training dataset must contain 2000 pairs."

assert len(english_to_spanish_iteration_1_dev_dataset) == development_parallel_size, \
    "The EN->ES Iteration 1 development dataset must contain 200 pairs."
assert len(spanish_to_english_iteration_1_dev_dataset) == development_parallel_size, \
    "The ES->EN Iteration 1 development dataset must contain 200 pairs."

forward_iteration_1_train_dataset = english_to_spanish_iteration_1_train_dataset
forward_iteration_1_dev_dataset = english_to_spanish_iteration_1_dev_dataset

backward_iteration_1_train_dataset = spanish_to_english_iteration_1_train_dataset
backward_iteration_1_dev_dataset = spanish_to_english_iteration_1_dev_dataset

print("\nStarting checkpoints used for Iteration 1:")
print(" - EN -> ES starts from:", baseline_forward_model_output_directory)
print(" - ES -> EN starts from:", baseline_backward_model_output_directory)

print("\nExample synthetic EN -> ES pair:")
example_en_es_synthetic_pair = english_to_spanish_iteration_1_synthetic_pairs[0]
print("Synthetic EN:", example_en_es_synthetic_pair[0])
print("Original ES :", example_en_es_synthetic_pair[1])

print("\nExample synthetic ES -> EN pair:")
example_es_en_synthetic_pair = spanish_to_english_iteration_1_synthetic_pairs[0]
print("Synthetic ES:", example_es_en_synthetic_pair[0])
print("Original EN :", example_es_en_synthetic_pair[1])

print("Both augmented training corpora are ready.")

Iteration 1: translating the Spanish monolingual corpus with the baseline ES->EN model:   0%|          | 0/63 …

Iteration 1: translating the English monolingual corpus with the baseline EN->ES model:   0%|          | 0/63 …

ITERATION 1 CORPUS SIZE
-----------------------
Iteration name                           : iteration_1
Authentic parallel pairs / direction     : 1000
Synthetic EN-ES pairs added              : 1000
Synthetic ES-EN pairs added              : 1000
Total EN->ES training pairs              : 2000
Total ES->EN training pairs              : 2000

Important:
- Iteration 1 starts from the baseline models obtained in Iteration 0.
- Synthetic data are generated from the monolingual corpora.
- The test set remains untouched.

ITERATION 1 DATASET CHECKS
--------------------------
EN -> ES synthetic pairs         : 1000
ES -> EN synthetic pairs         : 1000
EN -> ES total training pairs    : 2000
ES -> EN total training pairs    : 2000
EN -> ES dev pairs               : 200
ES -> EN dev pairs               : 200

Starting checkpoints used for Iteration 1:
 - EN -> ES starts from: /content/drive/MyDrive/anlp/iterative_backtranslation_en_to_es/iteration_0_baseline
 - ES -> EN starts from: /content

This code fine-tunes the English to Spanish model for Iteration 1 using the augmented dataset created in the previous cell. The model starts from the baseline checkpoint rather than from the original pretrained model, which is consistent with iterative back-translation. Training is performed on 2,000 total pairs and evaluated on the 200-sentence development set only.

In [29]:
english_to_spanish_iteration_1_result = fine_tune_directional_model_from_checkpoint(
    stage_name="English -> Spanish Iteration 1",
    starting_model_checkpoint_directory=baseline_forward_model_output_directory,
    directional_tokenizer=english_to_spanish_pretrained_tokenizer,
    train_parallel_dataset=english_to_spanish_iteration_1_train_dataset,
    development_parallel_dataset=english_to_spanish_iteration_1_dev_dataset,
    source_language_code=source_language_code,   # en
    target_language_code=target_language_code,   # es
    output_directory=iteration_one_forward_model_output_directory,
    iteration_name=iteration_one_name,
    authentic_parallel_pairs_used=parallel_training_size,
    synthetic_parallel_pairs_added=len(english_to_spanish_iteration_1_synthetic_pairs),
    max_num_epochs=iteration_one_max_num_epochs,
)

STARTING English -> Spanish Iteration 1
---------------------------------------
Starting checkpoint        : /content/drive/MyDrive/anlp/iterative_backtranslation_en_to_es/iteration_0_baseline
Direction                  : en -> es
Authentic training pairs   : 1000
Synthetic training pairs   : 1000
Total training pairs       : 2000
Development pairs          : 200
Monolingual data used      : Yes
Evaluation split used      : Development only
Test split used            : No



Tokenizing baseline training dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Tokenizing baseline development dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Bleu,Gen Len
1,0.595200,0.946101,45.660500,29.640000
2,0.499600,0.962621,45.740400,29.715000
3,0.427600,0.986513,44.828500,29.800000
4,0.369800,1.011240,45.189100,29.510000
5,0.321100,1.037810,45.059100,29.620000


Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Tr

Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Tr


Completed English -> Spanish Iteration 1 successfully.
Saved best model to : /content/drive/MyDrive/anlp/iterative_backtranslation_en_to_es/iteration_1
Development BLEU    : 45.7404
Development gen_len : 29.715



We then perform the same process for the Spanish to English direction. This code starts from the baseline Spanish to English checkpoint and fine-tunes the model on the augmented dataset that combines authentic pairs with synthetic pairs generated from the English monolingual corpus. The development set is again used for validation, while the test set remains unused. In this way, the cell completes the first bidirectional iterative training round.

In [30]:
spanish_to_english_iteration_1_result = fine_tune_directional_model_from_checkpoint(
    stage_name="Spanish -> English Iteration 1",
    starting_model_checkpoint_directory=baseline_backward_model_output_directory,
    directional_tokenizer=spanish_to_english_pretrained_tokenizer,
    train_parallel_dataset=spanish_to_english_iteration_1_train_dataset,
    development_parallel_dataset=spanish_to_english_iteration_1_dev_dataset,
    source_language_code=target_language_code,   # es
    target_language_code=source_language_code,   # en
    output_directory=iteration_one_backward_model_output_directory,
    iteration_name=iteration_one_name,
    authentic_parallel_pairs_used=parallel_training_size,
    synthetic_parallel_pairs_added=len(spanish_to_english_iteration_1_synthetic_pairs),
    max_num_epochs=iteration_one_max_num_epochs,
)

STARTING Spanish -> English Iteration 1
---------------------------------------
Starting checkpoint        : /content/drive/MyDrive/anlp/iterative_backtranslation_es_to_en/iteration_0_baseline
Direction                  : es -> en
Authentic training pairs   : 1000
Synthetic training pairs   : 1000
Total training pairs       : 2000
Development pairs          : 200
Monolingual data used      : Yes
Evaluation split used      : Development only
Test split used            : No



Tokenizing baseline training dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Tokenizing baseline development dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Bleu,Gen Len
1,0.650500,1.012407,49.701400,26.320000
2,0.542200,1.029103,48.996000,26.215000
3,0.467600,1.039278,48.178800,26.275000
4,0.400700,1.065324,48.154600,26.330000


Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Tr

Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Tr


Completed Spanish -> English Iteration 1 successfully.
Saved best model to : /content/drive/MyDrive/anlp/iterative_backtranslation_es_to_en/iteration_1
Development BLEU    : 49.7014
Development gen_len : 26.32



The following code summarizes the final results of Iteration 1. It stores the corpus statistics and both directional results in a single structure, checks that the expected amount of synthetic data was used, confirms that monolingual data were incorporated, and verifies that the test set was not touched. It also saves the full summary to a JSON file and updates the experiment history. Its role is to formally close and document the first back-translation iteration.

In [31]:
iteration_one_results = {
    "iteration_name": iteration_one_name,
    "corpus_summary": iteration_one_corpus_summary,
    "english_to_spanish": english_to_spanish_iteration_1_result,
    "spanish_to_english": spanish_to_english_iteration_1_result,
}

assert english_to_spanish_iteration_1_result["synthetic_parallel_pairs_added"] == spanish_monolingual_size
assert spanish_to_english_iteration_1_result["synthetic_parallel_pairs_added"] == english_monolingual_size

assert english_to_spanish_iteration_1_result["monolingual_data_used"] is True
assert spanish_to_english_iteration_1_result["monolingual_data_used"] is True

assert english_to_spanish_iteration_1_result["test_set_used"] is False
assert spanish_to_english_iteration_1_result["test_set_used"] is False

iteration_one_overall_summary_path = corpus_root_directory / "iteration_1_overall_summary.json"

with iteration_one_overall_summary_path.open("w", encoding="utf-8") as summary_file:
    json.dump(iteration_one_results, summary_file, indent=2, ensure_ascii=False)

iterative_backtranslation_history = {
    "iteration_0_baseline": baseline_iteration_0_results,
    "iteration_1": iteration_one_results,
}

print("ITERATION 1 FINAL SUMMARY")
print("-------------------------")
print(f"Iteration name                         : {iteration_one_results['iteration_name']}")
print(f"Authentic pairs / direction            : {iteration_one_results['corpus_summary']['authentic_parallel_pairs_per_direction']}")
print(f"EN -> ES synthetic pairs               : {iteration_one_results['corpus_summary']['english_to_spanish_synthetic_pairs']}")
print(f"ES -> EN synthetic pairs               : {iteration_one_results['corpus_summary']['spanish_to_english_synthetic_pairs']}")
print(f"EN -> ES total training pairs          : {iteration_one_results['corpus_summary']['english_to_spanish_total_training_pairs']}")
print(f"ES -> EN total training pairs          : {iteration_one_results['corpus_summary']['spanish_to_english_total_training_pairs']}")

print("\nDIRECTIONAL ITERATION 1 RESULTS")
print("-------------------------------")
print("English -> Spanish")
print(f" - Starting checkpoint : {english_to_spanish_iteration_1_result['starting_model_checkpoint_directory']}")
print(f" - Output directory    : {english_to_spanish_iteration_1_result['output_directory']}")
print(f" - Direction           : {english_to_spanish_iteration_1_result['translation_direction']}")
print(f" - Authentic pairs     : {english_to_spanish_iteration_1_result['authentic_parallel_pairs_used']}")
print(f" - Synthetic pairs     : {english_to_spanish_iteration_1_result['synthetic_parallel_pairs_added']}")
print(f" - Total training pairs: {english_to_spanish_iteration_1_result['total_training_pairs']}")
print(f" - Dev pairs           : {english_to_spanish_iteration_1_result['development_pairs']}")
print(f" - Dev BLEU            : {english_to_spanish_iteration_1_result['dev_bleu']}")
print(f" - Dev gen_len         : {english_to_spanish_iteration_1_result['dev_gen_len']}")

print("\nSpanish -> English")
print(f" - Starting checkpoint : {spanish_to_english_iteration_1_result['starting_model_checkpoint_directory']}")
print(f" - Output directory    : {spanish_to_english_iteration_1_result['output_directory']}")
print(f" - Direction           : {spanish_to_english_iteration_1_result['translation_direction']}")
print(f" - Authentic pairs     : {spanish_to_english_iteration_1_result['authentic_parallel_pairs_used']}")
print(f" - Synthetic pairs     : {spanish_to_english_iteration_1_result['synthetic_parallel_pairs_added']}")
print(f" - Total training pairs: {spanish_to_english_iteration_1_result['total_training_pairs']}")
print(f" - Dev pairs           : {spanish_to_english_iteration_1_result['development_pairs']}")
print(f" - Dev BLEU            : {spanish_to_english_iteration_1_result['dev_bleu']}")
print(f" - Dev gen_len         : {spanish_to_english_iteration_1_result['dev_gen_len']}")

print("\nSaved combined Iteration 1 summary to:")
print(iteration_one_overall_summary_path)

ITERATION 1 FINAL SUMMARY
-------------------------
Iteration name                         : iteration_1
Authentic pairs / direction            : 1000
EN -> ES synthetic pairs               : 1000
ES -> EN synthetic pairs               : 1000
EN -> ES total training pairs          : 2000
ES -> EN total training pairs          : 2000

DIRECTIONAL ITERATION 1 RESULTS
-------------------------------
English -> Spanish
 - Starting checkpoint : /content/drive/MyDrive/anlp/iterative_backtranslation_en_to_es/iteration_0_baseline
 - Output directory    : /content/drive/MyDrive/anlp/iterative_backtranslation_en_to_es/iteration_1
 - Direction           : en->es
 - Authentic pairs     : 1000
 - Synthetic pairs     : 1000
 - Total training pairs: 2000
 - Dev pairs           : 200
 - Dev BLEU            : 45.7404
 - Dev gen_len         : 29.715

Spanish -> English
 - Starting checkpoint : /content/drive/MyDrive/anlp/iterative_backtranslation_es_to_en/iteration_0_baseline
 - Output directory    : /c

## SECTION 8

This section implements Iteration 2 as the second full round of iterative back-translation and shows that the method is truly iterative rather than a single synthetic-data step. It regenerates the synthetic corpora using the improved models obtained after Iteration 1: the updated Spanish to English model translates the Spanish monolingual corpus to create new synthetic English to Spanish pairs, while the updated English to Spanish model translates the English monolingual corpus to create new synthetic Spanish to English pairs. These regenerated pairs are then combined again with the same 1,000 authentic parallel training pairs, so each directional training corpus still contains 2,000 pairs, although the synthetic content is now refreshed rather than reused. After rebuilding these augmented datasets, both models are fine-tuned again from the Iteration 1 checkpoints and evaluated on the same 200-sentence development set, while the test set remains untouched.


This cell defines the auxiliary structure for Iteration 2. It makes explicit that Iteration 2 keeps the same corpus size as Iteration 1, but refreshes the synthetic data with improved models.

In [32]:
from pathlib import Path
from typing import Dict

iteration_two_name = "iteration_2"
iteration_two_max_num_epochs = iteration_one_max_num_epochs

iteration_two_forward_model_output_directory = forward_model_output_directory / iteration_two_name
iteration_two_backward_model_output_directory = backward_model_output_directory / iteration_two_name


def build_iteration_two_corpus_summary(
    authentic_parallel_pairs_per_direction: int,
    english_to_spanish_synthetic_pairs: int,
    spanish_to_english_synthetic_pairs: int,
) -> Dict[str, int]:
    if authentic_parallel_pairs_per_direction <= 0:
        raise ValueError("The authentic parallel corpus size must be positive.")
    if english_to_spanish_synthetic_pairs <= 0:
        raise ValueError("The EN->ES synthetic pair count must be positive.")
    if spanish_to_english_synthetic_pairs <= 0:
        raise ValueError("The ES->EN synthetic pair count must be positive.")

    return {
        "iteration_name": iteration_two_name,
        "authentic_parallel_pairs_per_direction": authentic_parallel_pairs_per_direction,
        "english_to_spanish_synthetic_pairs": english_to_spanish_synthetic_pairs,
        "spanish_to_english_synthetic_pairs": spanish_to_english_synthetic_pairs,
        "english_to_spanish_total_training_pairs": (
            authentic_parallel_pairs_per_direction + english_to_spanish_synthetic_pairs
        ),
        "spanish_to_english_total_training_pairs": (
            authentic_parallel_pairs_per_direction + spanish_to_english_synthetic_pairs
        ),
    }


def print_iteration_two_corpus_summary(iteration_two_corpus_summary: Dict[str, int]) -> None:
    print("ITERATION 2 CORPUS SIZE")
    print("-----------------------")
    print(f"Iteration name                           : {iteration_two_corpus_summary['iteration_name']}")
    print(f"Authentic parallel pairs / direction     : {iteration_two_corpus_summary['authentic_parallel_pairs_per_direction']}")
    print(f"Synthetic EN-ES pairs added              : {iteration_two_corpus_summary['english_to_spanish_synthetic_pairs']}")
    print(f"Synthetic ES-EN pairs added              : {iteration_two_corpus_summary['spanish_to_english_synthetic_pairs']}")
    print(f"Total EN->ES training pairs              : {iteration_two_corpus_summary['english_to_spanish_total_training_pairs']}")
    print(f"Total ES->EN training pairs              : {iteration_two_corpus_summary['spanish_to_english_total_training_pairs']}")


def compute_development_bleu_delta(
    previous_iteration_directional_result: Dict[str, object],
    current_iteration_directional_result: Dict[str, object],
) -> float:
    """
    Compute the change in development BLEU between two consecutive iterations.
    """
    previous_bleu = float(previous_iteration_directional_result["dev_bleu"])
    current_bleu = float(current_iteration_directional_result["dev_bleu"])
    return round(current_bleu - previous_bleu, 4)


This code prepares the data for Iteration 2. It regenerates synthetic sentence pairs using the Iteration 1 checkpoints, rebuilds the augmented English to Spanish and Spanish to English training datasets, and reuses the same development sets. The most important detail here is that synthetic data from Iteration 1 are not accumulated; instead, they are replaced by newly generated pairs, which correctly reflects the logic of iterative back-translation.

In [33]:
iteration_two_en_to_es_generation_checkpoint_directory = iteration_one_backward_model_output_directory
iteration_two_en_to_es_starting_checkpoint_directory = iteration_one_forward_model_output_directory

iteration_two_es_to_en_generation_checkpoint_directory = iteration_one_forward_model_output_directory
iteration_two_es_to_en_starting_checkpoint_directory = iteration_one_backward_model_output_directory


english_to_spanish_iteration_2_synthetic_pairs = build_synthetic_parallel_pairs_from_monolingual_dataset(
    monolingual_text_dataset=spanish_monolingual_dataset,
    generation_model_checkpoint_directory=iteration_two_en_to_es_generation_checkpoint_directory,
    synthetic_source_language_name="English",
    original_target_language_name="Spanish",
    progress_description=(
        "Iteration 2: translating the Spanish monolingual corpus "
        "with the Iteration 1 ES->EN model"
    ),
)

spanish_to_english_iteration_2_synthetic_pairs = build_synthetic_parallel_pairs_from_monolingual_dataset(
    monolingual_text_dataset=english_monolingual_dataset,
    generation_model_checkpoint_directory=iteration_two_es_to_en_generation_checkpoint_directory,
    synthetic_source_language_name="Spanish",
    original_target_language_name="English",
    progress_description=(
        "Iteration 2: translating the English monolingual corpus "
        "with the Iteration 1 EN->ES model"
    ),
)

english_to_spanish_iteration_2_train_dataset = build_augmented_parallel_dataset_for_direction(
    authentic_parallel_dataset=english_to_spanish_baseline_train_dataset,
    synthetic_parallel_pairs=english_to_spanish_iteration_2_synthetic_pairs,
    source_language_code=source_language_code,  # en
    target_language_code=target_language_code,  # es
)

spanish_to_english_iteration_2_train_dataset = build_augmented_parallel_dataset_for_direction(
    authentic_parallel_dataset=spanish_to_english_baseline_train_dataset,
    synthetic_parallel_pairs=spanish_to_english_iteration_2_synthetic_pairs,
    source_language_code=target_language_code,  # es
    target_language_code=source_language_code,  # en
)

english_to_spanish_iteration_2_dev_dataset = english_to_spanish_baseline_dev_dataset
spanish_to_english_iteration_2_dev_dataset = spanish_to_english_baseline_dev_dataset

iteration_two_corpus_summary = build_iteration_two_corpus_summary(
    authentic_parallel_pairs_per_direction=parallel_training_size,
    english_to_spanish_synthetic_pairs=len(english_to_spanish_iteration_2_synthetic_pairs),
    spanish_to_english_synthetic_pairs=len(spanish_to_english_iteration_2_synthetic_pairs),
)

print_iteration_two_corpus_summary(iteration_two_corpus_summary)

print("\nITERATION 2 DATASET CHECKS")
print("--------------------------")
print(f"EN -> ES synthetic pairs         : {len(english_to_spanish_iteration_2_synthetic_pairs)}")
print(f"ES -> EN synthetic pairs         : {len(spanish_to_english_iteration_2_synthetic_pairs)}")
print(f"EN -> ES total training pairs    : {len(english_to_spanish_iteration_2_train_dataset)}")
print(f"ES -> EN total training pairs    : {len(spanish_to_english_iteration_2_train_dataset)}")
print(f"EN -> ES dev pairs               : {len(english_to_spanish_iteration_2_dev_dataset)}")
print(f"ES -> EN dev pairs               : {len(spanish_to_english_iteration_2_dev_dataset)}")

assert len(english_to_spanish_iteration_2_synthetic_pairs) == spanish_monolingual_size, \
    "The EN->ES Iteration 2 synthetic pair count must match the Spanish monolingual corpus size."
assert len(spanish_to_english_iteration_2_synthetic_pairs) == english_monolingual_size, \
    "The ES->EN Iteration 2 synthetic pair count must match the English monolingual corpus size."

assert len(english_to_spanish_iteration_2_train_dataset) == parallel_training_size + spanish_monolingual_size, \
    "The EN->ES Iteration 2 training dataset must contain 2000 pairs."
assert len(spanish_to_english_iteration_2_train_dataset) == parallel_training_size + english_monolingual_size, \
    "The ES->EN Iteration 2 training dataset must contain 2000 pairs."

assert len(english_to_spanish_iteration_2_dev_dataset) == development_parallel_size, \
    "The EN->ES Iteration 2 development dataset must contain 200 pairs."
assert len(spanish_to_english_iteration_2_dev_dataset) == development_parallel_size, \
    "The ES->EN Iteration 2 development dataset must contain 200 pairs."

forward_iteration_2_train_dataset = english_to_spanish_iteration_2_train_dataset
forward_iteration_2_dev_dataset = english_to_spanish_iteration_2_dev_dataset

backward_iteration_2_train_dataset = spanish_to_english_iteration_2_train_dataset
backward_iteration_2_dev_dataset = spanish_to_english_iteration_2_dev_dataset

print("\nIteration 2 checkpoint policy:")
print(" - EN -> ES synthetic generation uses :", iteration_two_en_to_es_generation_checkpoint_directory)
print(" - EN -> ES fine-tuning starts from   :", iteration_two_en_to_es_starting_checkpoint_directory)
print(" - ES -> EN synthetic generation uses :", iteration_two_es_to_en_generation_checkpoint_directory)
print(" - ES -> EN fine-tuning starts from   :", iteration_two_es_to_en_starting_checkpoint_directory)

print("\nRebuilding policy:")
print(" - The authentic 1000 parallel pairs are kept.")
print(" - The synthetic pairs from Iteration 1 are NOT accumulated.")
print(" - New synthetic pairs are regenerated with the improved Iteration 1 models.")
print(" - Therefore, the total size may still be 2000, but the synthetic content is updated.")

print("\nExample synthetic EN -> ES pair from Iteration 2:")
example_en_es_iteration_2_synthetic_pair = english_to_spanish_iteration_2_synthetic_pairs[0]
print("Synthetic EN:", example_en_es_iteration_2_synthetic_pair[0])
print("Original ES :", example_en_es_iteration_2_synthetic_pair[1])

print("\nExample synthetic ES -> EN pair from Iteration 2:")
example_es_en_iteration_2_synthetic_pair = spanish_to_english_iteration_2_synthetic_pairs[0]
print("Synthetic ES:", example_es_en_iteration_2_synthetic_pair[0])
print("Original EN :", example_es_en_iteration_2_synthetic_pair[1])

print("Both Iteration 2 augmented training corpora are ready.")


Iteration 2: translating the Spanish monolingual corpus with the Iteration 1 ES->EN model:   0%|          | 0/…

Iteration 2: translating the English monolingual corpus with the Iteration 1 EN->ES model:   0%|          | 0/…

ITERATION 2 CORPUS SIZE
-----------------------
Iteration name                           : iteration_2
Authentic parallel pairs / direction     : 1000
Synthetic EN-ES pairs added              : 1000
Synthetic ES-EN pairs added              : 1000
Total EN->ES training pairs              : 2000
Total ES->EN training pairs              : 2000

ITERATION 2 DATASET CHECKS
--------------------------
EN -> ES synthetic pairs         : 1000
ES -> EN synthetic pairs         : 1000
EN -> ES total training pairs    : 2000
ES -> EN total training pairs    : 2000
EN -> ES dev pairs               : 200
ES -> EN dev pairs               : 200

Iteration 2 checkpoint policy:
 - EN -> ES synthetic generation uses : /content/drive/MyDrive/anlp/iterative_backtranslation_es_to_en/iteration_1
 - EN -> ES fine-tuning starts from   : /content/drive/MyDrive/anlp/iterative_backtranslation_en_to_es/iteration_1
 - ES -> EN synthetic generation uses : /content/drive/MyDrive/anlp/iterative_backtranslation_en_to_es

The following code fine-tunes the English to Spanish model for Iteration 2. It starts from the best English to Spanish checkpoint obtained in Iteration 1 and trains the model on the new augmented dataset containing authentic and regenerated synthetic pairs. The same training strategy as in previous sections is reused.

In [34]:
english_to_spanish_iteration_2_result = fine_tune_directional_model_from_checkpoint(
    stage_name="English -> Spanish Iteration 2",
    starting_model_checkpoint_directory=iteration_two_en_to_es_starting_checkpoint_directory,
    directional_tokenizer=english_to_spanish_pretrained_tokenizer,
    train_parallel_dataset=english_to_spanish_iteration_2_train_dataset,
    development_parallel_dataset=english_to_spanish_iteration_2_dev_dataset,
    source_language_code=source_language_code,   # en
    target_language_code=target_language_code,   # es
    output_directory=iteration_two_forward_model_output_directory,
    iteration_name=iteration_two_name,
    authentic_parallel_pairs_used=parallel_training_size,
    synthetic_parallel_pairs_added=len(english_to_spanish_iteration_2_synthetic_pairs),
    max_num_epochs=iteration_two_max_num_epochs,
)


STARTING English -> Spanish Iteration 2
---------------------------------------
Starting checkpoint        : /content/drive/MyDrive/anlp/iterative_backtranslation_en_to_es/iteration_1
Direction                  : en -> es
Authentic training pairs   : 1000
Synthetic training pairs   : 1000
Total training pairs       : 2000
Development pairs          : 200
Monolingual data used      : Yes
Evaluation split used      : Development only
Test split used            : No



Tokenizing baseline training dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Tokenizing baseline development dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Bleu,Gen Len
1,0.424300,0.989436,45.408600,29.570000
2,0.355000,1.019803,44.962000,29.730000
3,0.301500,1.050315,43.990200,29.600000
4,0.257100,1.087293,44.086400,30.055000


Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Tr

Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Tr


Completed English -> Spanish Iteration 2 successfully.
Saved best model to : /content/drive/MyDrive/anlp/iterative_backtranslation_en_to_es/iteration_2
Development BLEU    : 45.4086
Development gen_len : 29.57



This cell performs the symmetric process for the Spanish to English direction. It starts from the Iteration 1 Spanish to English checkpoint and fine-tunes the model on the rebuilt Iteration 2 augmented corpus.

In [35]:
spanish_to_english_iteration_2_result = fine_tune_directional_model_from_checkpoint(
    stage_name="Spanish -> English Iteration 2",
    starting_model_checkpoint_directory=iteration_two_es_to_en_starting_checkpoint_directory,
    directional_tokenizer=spanish_to_english_pretrained_tokenizer,
    train_parallel_dataset=spanish_to_english_iteration_2_train_dataset,
    development_parallel_dataset=spanish_to_english_iteration_2_dev_dataset,
    source_language_code=target_language_code,   # es
    target_language_code=source_language_code,   # en
    output_directory=iteration_two_backward_model_output_directory,
    iteration_name=iteration_two_name,
    authentic_parallel_pairs_used=parallel_training_size,
    synthetic_parallel_pairs_added=len(spanish_to_english_iteration_2_synthetic_pairs),
    max_num_epochs=iteration_two_max_num_epochs,
)


STARTING Spanish -> English Iteration 2
---------------------------------------
Starting checkpoint        : /content/drive/MyDrive/anlp/iterative_backtranslation_es_to_en/iteration_1
Direction                  : es -> en
Authentic training pairs   : 1000
Synthetic training pairs   : 1000
Total training pairs       : 2000
Development pairs          : 200
Monolingual data used      : Yes
Evaluation split used      : Development only
Test split used            : No



Tokenizing baseline training dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Tokenizing baseline development dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Bleu,Gen Len
1,0.497800,1.034558,48.870800,26.435000
2,0.442200,1.052661,48.397600,26.450000
3,0.378900,1.070502,47.576500,26.410000
4,0.325200,1.096701,47.359400,26.475000


Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Tr

Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Tr


Completed Spanish -> English Iteration 2 successfully.
Saved best model to : /content/drive/MyDrive/anlp/iterative_backtranslation_es_to_en/iteration_2
Development BLEU    : 48.8708
Development gen_len : 26.435



This cell summarizes the final results of Iteration 2.

In [36]:
import json

iteration_two_results = {
    "iteration_name": iteration_two_name,
    "corpus_summary": iteration_two_corpus_summary,
    "english_to_spanish": english_to_spanish_iteration_2_result,
    "spanish_to_english": spanish_to_english_iteration_2_result,
}

assert english_to_spanish_iteration_2_result["synthetic_parallel_pairs_added"] == spanish_monolingual_size
assert spanish_to_english_iteration_2_result["synthetic_parallel_pairs_added"] == english_monolingual_size

assert english_to_spanish_iteration_2_result["monolingual_data_used"] is True
assert spanish_to_english_iteration_2_result["monolingual_data_used"] is True

assert english_to_spanish_iteration_2_result["test_set_used"] is False
assert spanish_to_english_iteration_2_result["test_set_used"] is False

iteration_two_overall_summary_path = corpus_root_directory / "iteration_2_overall_summary.json"

with iteration_two_overall_summary_path.open("w", encoding="utf-8") as summary_file:
    json.dump(iteration_two_results, summary_file, indent=2, ensure_ascii=False)

english_to_spanish_iteration_2_bleu_delta_vs_iteration_1 = compute_development_bleu_delta(
    previous_iteration_directional_result=english_to_spanish_iteration_1_result,
    current_iteration_directional_result=english_to_spanish_iteration_2_result,
)

spanish_to_english_iteration_2_bleu_delta_vs_iteration_1 = compute_development_bleu_delta(
    previous_iteration_directional_result=spanish_to_english_iteration_1_result,
    current_iteration_directional_result=spanish_to_english_iteration_2_result,
)

existing_iterative_history = globals().get("iterative_backtranslation_history", None)

if existing_iterative_history is None:
    iterative_backtranslation_history = {
        "iteration_0_baseline": baseline_iteration_0_results,
        "iteration_1": iteration_one_results,
    }
else:
    iterative_backtranslation_history = dict(existing_iterative_history)

iterative_backtranslation_history[iteration_two_name] = iteration_two_results

print("ITERATION 2 FINAL SUMMARY")
print("-------------------------")
print(f"Iteration name                         : {iteration_two_results['iteration_name']}")
print(f"Authentic pairs / direction            : {iteration_two_results['corpus_summary']['authentic_parallel_pairs_per_direction']}")
print(f"EN -> ES synthetic pairs               : {iteration_two_results['corpus_summary']['english_to_spanish_synthetic_pairs']}")
print(f"ES -> EN synthetic pairs               : {iteration_two_results['corpus_summary']['spanish_to_english_synthetic_pairs']}")
print(f"EN -> ES total training pairs          : {iteration_two_results['corpus_summary']['english_to_spanish_total_training_pairs']}")
print(f"ES -> EN total training pairs          : {iteration_two_results['corpus_summary']['spanish_to_english_total_training_pairs']}")

print("\nDIRECTIONAL ITERATION 2 RESULTS")
print("-------------------------------")
print("English -> Spanish")
print(f" - Starting checkpoint : {english_to_spanish_iteration_2_result['starting_model_checkpoint_directory']}")
print(f" - Output directory    : {english_to_spanish_iteration_2_result['output_directory']}")
print(f" - Direction           : {english_to_spanish_iteration_2_result['translation_direction']}")
print(f" - Authentic pairs     : {english_to_spanish_iteration_2_result['authentic_parallel_pairs_used']}")
print(f" - Synthetic pairs     : {english_to_spanish_iteration_2_result['synthetic_parallel_pairs_added']}")
print(f" - Total training pairs: {english_to_spanish_iteration_2_result['total_training_pairs']}")
print(f" - Dev pairs           : {english_to_spanish_iteration_2_result['development_pairs']}")
print(f" - Dev BLEU            : {english_to_spanish_iteration_2_result['dev_bleu']}")
print(f" - Dev gen_len         : {english_to_spanish_iteration_2_result['dev_gen_len']}")
print(f" - BLEU delta vs It. 1 : {english_to_spanish_iteration_2_bleu_delta_vs_iteration_1:+.4f}")

print("\nSpanish -> English")
print(f" - Starting checkpoint : {spanish_to_english_iteration_2_result['starting_model_checkpoint_directory']}")
print(f" - Output directory    : {spanish_to_english_iteration_2_result['output_directory']}")
print(f" - Direction           : {spanish_to_english_iteration_2_result['translation_direction']}")
print(f" - Authentic pairs     : {spanish_to_english_iteration_2_result['authentic_parallel_pairs_used']}")
print(f" - Synthetic pairs     : {spanish_to_english_iteration_2_result['synthetic_parallel_pairs_added']}")
print(f" - Total training pairs: {spanish_to_english_iteration_2_result['total_training_pairs']}")
print(f" - Dev pairs           : {spanish_to_english_iteration_2_result['development_pairs']}")
print(f" - Dev BLEU            : {spanish_to_english_iteration_2_result['dev_bleu']}")
print(f" - Dev gen_len         : {spanish_to_english_iteration_2_result['dev_gen_len']}")
print(f" - BLEU delta vs It. 1 : {spanish_to_english_iteration_2_bleu_delta_vs_iteration_1:+.4f}")

print("\nMETHODOLOGICAL CHECKS")
print("---------------------")
print(" - Iteration 2 starts from the improved Iteration 1 checkpoints")
print(" - Synthetic data are regenerated using the improved Iteration 1 models")
print(" - The augmented corpora are rebuilt with the new synthetic sentence pairs")
print(" - Development set used for evaluation")
print(" - Test set not used")
print(" - This is iterative back-translation, not one-shot back-translation")

print("\nInterpretation note:")
print("A new iteration does not guarantee a BLEU improvement in every direction.")
print("The important methodological point is that synthetic data are regenerated")
print("with stronger models and the systems are fine-tuned again in a new cycle.")

print("\nSaved combined Iteration 2 summary to:")
print(iteration_two_overall_summary_path)

ITERATION 2 FINAL SUMMARY
-------------------------
Iteration name                         : iteration_2
Authentic pairs / direction            : 1000
EN -> ES synthetic pairs               : 1000
ES -> EN synthetic pairs               : 1000
EN -> ES total training pairs          : 2000
ES -> EN total training pairs          : 2000

DIRECTIONAL ITERATION 2 RESULTS
-------------------------------
English -> Spanish
 - Starting checkpoint : /content/drive/MyDrive/anlp/iterative_backtranslation_en_to_es/iteration_1
 - Output directory    : /content/drive/MyDrive/anlp/iterative_backtranslation_en_to_es/iteration_2
 - Direction           : en->es
 - Authentic pairs     : 1000
 - Synthetic pairs     : 1000
 - Total training pairs: 2000
 - Dev pairs           : 200
 - Dev BLEU            : 45.4086
 - Dev gen_len         : 29.57
 - BLEU delta vs It. 1 : -0.3318

Spanish -> English
 - Starting checkpoint : /content/drive/MyDrive/anlp/iterative_backtranslation_es_to_en/iteration_1
 - Output dir

## SECTION 9

This section implements Iteration 3 as the final round of the iterative back-translation process. We first define the specific directories and helper functions for this stage, then regenerate the synthetic bilingual data using the improved models obtained after Iteration 2 instead of reusing previous synthetic corpora. More specifically, the Iteration 2 Spanish to English model is used to translate the Spanish monolingual corpus and create new synthetic English–Spanish pairs, while the Iteration 2 English to Spanish model is applied to the English monolingual corpus to create new synthetic Spanish–English pairs. These new synthetic pairs are then combined with the same 1,000 authentic parallel training pairs, so each directional training corpus again contains 2,000 pairs in total. After that, both directional models are fine-tuned once more from their Iteration 2 checkpoints and evaluated on the same 200-sentence development set, while the test set remains unused.


This cell defines the configuration for Iteration 3.

In [37]:
from pathlib import Path
from typing import Dict

iteration_three_name = "iteration_3"
iteration_three_max_num_epochs = iteration_two_max_num_epochs

iteration_three_forward_model_output_directory = forward_model_output_directory / iteration_three_name
iteration_three_backward_model_output_directory = backward_model_output_directory / iteration_three_name


def build_iteration_three_corpus_summary(
    authentic_parallel_pairs_per_direction: int,
    english_to_spanish_synthetic_pairs: int,
    spanish_to_english_synthetic_pairs: int,
) -> Dict[str, int]:
    if authentic_parallel_pairs_per_direction <= 0:
        raise ValueError("The authentic parallel corpus size must be positive.")
    if english_to_spanish_synthetic_pairs <= 0:
        raise ValueError("The EN->ES synthetic pair count must be positive.")
    if spanish_to_english_synthetic_pairs <= 0:
        raise ValueError("The ES->EN synthetic pair count must be positive.")

    return {
        "iteration_name": iteration_three_name,
        "authentic_parallel_pairs_per_direction": authentic_parallel_pairs_per_direction,
        "english_to_spanish_synthetic_pairs": english_to_spanish_synthetic_pairs,
        "spanish_to_english_synthetic_pairs": spanish_to_english_synthetic_pairs,
        "english_to_spanish_total_training_pairs": (
            authentic_parallel_pairs_per_direction + english_to_spanish_synthetic_pairs
        ),
        "spanish_to_english_total_training_pairs": (
            authentic_parallel_pairs_per_direction + spanish_to_english_synthetic_pairs
        ),
    }


def print_iteration_three_corpus_summary(iteration_three_corpus_summary: Dict[str, int]) -> None:
    print("ITERATION 3 CORPUS SIZE")
    print("-----------------------")
    print(f"Iteration name                           : {iteration_three_corpus_summary['iteration_name']}")
    print(f"Authentic parallel pairs / direction     : {iteration_three_corpus_summary['authentic_parallel_pairs_per_direction']}")
    print(f"Synthetic EN-ES pairs added              : {iteration_three_corpus_summary['english_to_spanish_synthetic_pairs']}")
    print(f"Synthetic ES-EN pairs added              : {iteration_three_corpus_summary['spanish_to_english_synthetic_pairs']}")
    print(f"Total EN->ES training pairs              : {iteration_three_corpus_summary['english_to_spanish_total_training_pairs']}")
    print(f"Total ES->EN training pairs              : {iteration_three_corpus_summary['spanish_to_english_total_training_pairs']}")
    print("\nImportant:")
    print("- Iteration 3 starts from the improved models obtained in Iteration 2.")
    print("- Synthetic data are regenerated again with the latest opposite-direction models.")


This code prepares the Iteration 3 training data.

In [38]:
iteration_three_en_to_es_generation_checkpoint_directory = iteration_two_backward_model_output_directory
iteration_three_en_to_es_starting_checkpoint_directory = iteration_two_forward_model_output_directory

iteration_three_es_to_en_generation_checkpoint_directory = iteration_two_forward_model_output_directory
iteration_three_es_to_en_starting_checkpoint_directory = iteration_two_backward_model_output_directory


english_to_spanish_iteration_3_synthetic_pairs = build_synthetic_parallel_pairs_from_monolingual_dataset(
    monolingual_text_dataset=spanish_monolingual_dataset,
    generation_model_checkpoint_directory=iteration_three_en_to_es_generation_checkpoint_directory,
    synthetic_source_language_name="English",
    original_target_language_name="Spanish",
    progress_description=(
        "Iteration 3: translating the Spanish monolingual corpus "
        "with the Iteration 2 ES->EN model"
    ),
)

spanish_to_english_iteration_3_synthetic_pairs = build_synthetic_parallel_pairs_from_monolingual_dataset(
    monolingual_text_dataset=english_monolingual_dataset,
    generation_model_checkpoint_directory=iteration_three_es_to_en_generation_checkpoint_directory,
    synthetic_source_language_name="Spanish",
    original_target_language_name="English",
    progress_description=(
        "Iteration 3: translating the English monolingual corpus "
        "with the Iteration 2 EN->ES model"
    ),
)

english_to_spanish_iteration_3_train_dataset = build_augmented_parallel_dataset_for_direction(
    authentic_parallel_dataset=english_to_spanish_baseline_train_dataset,
    synthetic_parallel_pairs=english_to_spanish_iteration_3_synthetic_pairs,
    source_language_code=source_language_code,  # en
    target_language_code=target_language_code,  # es
)

spanish_to_english_iteration_3_train_dataset = build_augmented_parallel_dataset_for_direction(
    authentic_parallel_dataset=spanish_to_english_baseline_train_dataset,
    synthetic_parallel_pairs=spanish_to_english_iteration_3_synthetic_pairs,
    source_language_code=target_language_code,  # es
    target_language_code=source_language_code,  # en
)

english_to_spanish_iteration_3_dev_dataset = english_to_spanish_baseline_dev_dataset
spanish_to_english_iteration_3_dev_dataset = spanish_to_english_baseline_dev_dataset

iteration_three_corpus_summary = build_iteration_three_corpus_summary(
    authentic_parallel_pairs_per_direction=parallel_training_size,
    english_to_spanish_synthetic_pairs=len(english_to_spanish_iteration_3_synthetic_pairs),
    spanish_to_english_synthetic_pairs=len(spanish_to_english_iteration_3_synthetic_pairs),
)

print_iteration_three_corpus_summary(iteration_three_corpus_summary)

print("\nITERATION 3 DATASET CHECKS")
print("--------------------------")
print(f"EN -> ES synthetic pairs         : {len(english_to_spanish_iteration_3_synthetic_pairs)}")
print(f"ES -> EN synthetic pairs         : {len(spanish_to_english_iteration_3_synthetic_pairs)}")
print(f"EN -> ES total training pairs    : {len(english_to_spanish_iteration_3_train_dataset)}")
print(f"ES -> EN total training pairs    : {len(spanish_to_english_iteration_3_train_dataset)}")
print(f"EN -> ES dev pairs               : {len(english_to_spanish_iteration_3_dev_dataset)}")
print(f"ES -> EN dev pairs               : {len(spanish_to_english_iteration_3_dev_dataset)}")

assert len(english_to_spanish_iteration_3_synthetic_pairs) == spanish_monolingual_size, \
    "The EN->ES Iteration 3 synthetic pair count must match the Spanish monolingual corpus size."
assert len(spanish_to_english_iteration_3_synthetic_pairs) == english_monolingual_size, \
    "The ES->EN Iteration 3 synthetic pair count must match the English monolingual corpus size."

assert len(english_to_spanish_iteration_3_train_dataset) == parallel_training_size + spanish_monolingual_size, \
    "The EN->ES Iteration 3 training dataset must contain 2000 pairs."
assert len(spanish_to_english_iteration_3_train_dataset) == parallel_training_size + english_monolingual_size, \
    "The ES->EN Iteration 3 training dataset must contain 2000 pairs."

assert len(english_to_spanish_iteration_3_dev_dataset) == development_parallel_size, \
    "The EN->ES Iteration 3 development dataset must contain 200 pairs."
assert len(spanish_to_english_iteration_3_dev_dataset) == development_parallel_size, \
    "The ES->EN Iteration 3 development dataset must contain 200 pairs."

forward_iteration_3_train_dataset = english_to_spanish_iteration_3_train_dataset
forward_iteration_3_dev_dataset = english_to_spanish_iteration_3_dev_dataset

backward_iteration_3_train_dataset = spanish_to_english_iteration_3_train_dataset
backward_iteration_3_dev_dataset = spanish_to_english_iteration_3_dev_dataset

print("\nIteration 3 checkpoint policy:")
print(" - EN -> ES synthetic generation uses :", iteration_three_en_to_es_generation_checkpoint_directory)
print(" - EN -> ES fine-tuning starts from   :", iteration_three_en_to_es_starting_checkpoint_directory)
print(" - ES -> EN synthetic generation uses :", iteration_three_es_to_en_generation_checkpoint_directory)
print(" - ES -> EN fine-tuning starts from   :", iteration_three_es_to_en_starting_checkpoint_directory)

print("\nRebuilding policy:")
print(" - The authentic 1000 parallel pairs are kept.")
print(" - The synthetic pairs from previous iterations are NOT accumulated.")
print(" - New synthetic pairs are regenerated with the improved Iteration 2 models.")
print(" - Therefore, the total size remains 2000, but the synthetic content is refreshed.")

print("\nExample synthetic EN -> ES pair from Iteration 3:")
example_en_es_iteration_3_synthetic_pair = english_to_spanish_iteration_3_synthetic_pairs[0]
print("Synthetic EN:", example_en_es_iteration_3_synthetic_pair[0])
print("Original ES :", example_en_es_iteration_3_synthetic_pair[1])

print("\nExample synthetic ES -> EN pair from Iteration 3:")
example_es_en_iteration_3_synthetic_pair = spanish_to_english_iteration_3_synthetic_pairs[0]
print("Synthetic ES:", example_es_en_iteration_3_synthetic_pair[0])
print("Original EN :", example_es_en_iteration_3_synthetic_pair[1])

print("Both Iteration 3 augmented training corpora are ready.")


Iteration 3: translating the Spanish monolingual corpus with the Iteration 2 ES->EN model:   0%|          | 0/…

Iteration 3: translating the English monolingual corpus with the Iteration 2 EN->ES model:   0%|          | 0/…

ITERATION 3 CORPUS SIZE
-----------------------
Iteration name                           : iteration_3
Authentic parallel pairs / direction     : 1000
Synthetic EN-ES pairs added              : 1000
Synthetic ES-EN pairs added              : 1000
Total EN->ES training pairs              : 2000
Total ES->EN training pairs              : 2000

Important:
- Iteration 3 starts from the improved models obtained in Iteration 2.
- Synthetic data are regenerated again with the latest opposite-direction models.

ITERATION 3 DATASET CHECKS
--------------------------
EN -> ES synthetic pairs         : 1000
ES -> EN synthetic pairs         : 1000
EN -> ES total training pairs    : 2000
ES -> EN total training pairs    : 2000
EN -> ES dev pairs               : 200
ES -> EN dev pairs               : 200

Iteration 3 checkpoint policy:
 - EN -> ES synthetic generation uses : /content/drive/MyDrive/anlp/iterative_backtranslation_es_to_en/iteration_2
 - EN -> ES fine-tuning starts from   : /content/dri

This cell fine-tunes the English to Spanish model for Iteration 3. It starts from the Iteration 2 checkpoint, uses the newly rebuilt augmented training corpus, evaluates on the development set, and saves the updated model in the corresponding Iteration 3 output directory.

In [39]:
english_to_spanish_iteration_3_result = fine_tune_directional_model_from_checkpoint(
    stage_name="English -> Spanish Iteration 3",
    starting_model_checkpoint_directory=iteration_three_en_to_es_starting_checkpoint_directory,
    directional_tokenizer=english_to_spanish_pretrained_tokenizer,
    train_parallel_dataset=english_to_spanish_iteration_3_train_dataset,
    development_parallel_dataset=english_to_spanish_iteration_3_dev_dataset,
    source_language_code=source_language_code,   # en
    target_language_code=target_language_code,   # es
    output_directory=iteration_three_forward_model_output_directory,
    iteration_name=iteration_three_name,
    authentic_parallel_pairs_used=parallel_training_size,
    synthetic_parallel_pairs_added=len(english_to_spanish_iteration_3_synthetic_pairs),
    max_num_epochs=iteration_three_max_num_epochs,
)


STARTING English -> Spanish Iteration 3
---------------------------------------
Starting checkpoint        : /content/drive/MyDrive/anlp/iterative_backtranslation_en_to_es/iteration_2
Direction                  : en -> es
Authentic training pairs   : 1000
Synthetic training pairs   : 1000
Total training pairs       : 2000
Development pairs          : 200
Monolingual data used      : Yes
Evaluation split used      : Development only
Test split used            : No



Tokenizing baseline training dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Tokenizing baseline development dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Bleu,Gen Len
1,0.343000,1.025943,44.485200,29.965000
2,0.305200,1.055801,44.551000,29.800000
3,0.259000,1.080230,43.428900,29.990000
4,0.220800,1.126335,43.525500,29.940000
5,0.188900,1.148860,43.745400,29.830000


Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Tr

Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Tr


Completed English -> Spanish Iteration 3 successfully.
Saved best model to : /content/drive/MyDrive/anlp/iterative_backtranslation_en_to_es/iteration_3
Development BLEU    : 44.551
Development gen_len : 29.8



This code performs the same process for the Spanish to English model. It loads the Iteration 2 checkpoint for that direction, trains it with the new Iteration 3 augmented dataset, evaluates it on the development set, and stores the resulting checkpoint.

In [43]:
spanish_to_english_iteration_3_result = fine_tune_directional_model_from_checkpoint(
    stage_name="Spanish -> English Iteration 3",
    starting_model_checkpoint_directory=iteration_three_es_to_en_starting_checkpoint_directory,
    directional_tokenizer=spanish_to_english_pretrained_tokenizer,
    train_parallel_dataset=spanish_to_english_iteration_3_train_dataset,
    development_parallel_dataset=spanish_to_english_iteration_3_dev_dataset,
    source_language_code=target_language_code,   # es
    target_language_code=source_language_code,   # en
    output_directory=iteration_three_backward_model_output_directory,
    iteration_name=iteration_three_name,
    authentic_parallel_pairs_used=parallel_training_size,
    synthetic_parallel_pairs_added=len(spanish_to_english_iteration_3_synthetic_pairs),
    max_num_epochs=iteration_three_max_num_epochs,
)


STARTING Spanish -> English Iteration 3
---------------------------------------
Starting checkpoint        : /content/drive/MyDrive/anlp/iterative_backtranslation_es_to_en/iteration_2
Direction                  : es -> en
Authentic training pairs   : 1000
Synthetic training pairs   : 1000
Total training pairs       : 2000
Development pairs          : 200
Monolingual data used      : Yes
Evaluation split used      : Development only
Test split used            : No



Tokenizing baseline training dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Tokenizing baseline development dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Bleu,Gen Len
1,0.455700,1.053459,47.830300,26.365000
2,0.390900,1.078924,47.939500,26.380000
3,0.335400,1.099314,47.414100,26.415000
4,0.282400,1.127183,47.382000,26.535000
5,0.245500,1.139918,47.289000,26.380000


Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Tr

Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Tr


Completed Spanish -> English Iteration 3 successfully.
Saved best model to : /content/drive/MyDrive/anlp/iterative_backtranslation_es_to_en/iteration_3
Development BLEU    : 47.9395
Development gen_len : 26.38



This cell summarizes the final results of Iteration 3.

In [44]:
import json

iteration_three_results = {
    "iteration_name": iteration_three_name,
    "corpus_summary": iteration_three_corpus_summary,
    "english_to_spanish": english_to_spanish_iteration_3_result,
    "spanish_to_english": spanish_to_english_iteration_3_result,
}

assert english_to_spanish_iteration_3_result["synthetic_parallel_pairs_added"] == spanish_monolingual_size
assert spanish_to_english_iteration_3_result["synthetic_parallel_pairs_added"] == english_monolingual_size

assert english_to_spanish_iteration_3_result["monolingual_data_used"] is True
assert spanish_to_english_iteration_3_result["monolingual_data_used"] is True

assert english_to_spanish_iteration_3_result["test_set_used"] is False
assert spanish_to_english_iteration_3_result["test_set_used"] is False

iteration_three_overall_summary_path = corpus_root_directory / "iteration_3_overall_summary.json"

with iteration_three_overall_summary_path.open("w", encoding="utf-8") as summary_file:
    json.dump(iteration_three_results, summary_file, indent=2, ensure_ascii=False)

english_to_spanish_iteration_3_bleu_delta_vs_iteration_2 = compute_development_bleu_delta(
    previous_iteration_directional_result=english_to_spanish_iteration_2_result,
    current_iteration_directional_result=english_to_spanish_iteration_3_result,
)

spanish_to_english_iteration_3_bleu_delta_vs_iteration_2 = compute_development_bleu_delta(
    previous_iteration_directional_result=spanish_to_english_iteration_2_result,
    current_iteration_directional_result=spanish_to_english_iteration_3_result,
)

existing_iterative_history = globals().get("iterative_backtranslation_history", None)

if existing_iterative_history is None:
    iterative_backtranslation_history = {
        "iteration_0_baseline": baseline_iteration_0_results,
        "iteration_1": iteration_one_results,
        "iteration_2": iteration_two_results,
    }
else:
    iterative_backtranslation_history = dict(existing_iterative_history)

iterative_backtranslation_history[iteration_three_name] = iteration_three_results

print("ITERATION 3 FINAL SUMMARY")
print("-------------------------")
print(f"Iteration name                         : {iteration_three_results['iteration_name']}")
print(f"Authentic pairs / direction            : {iteration_three_results['corpus_summary']['authentic_parallel_pairs_per_direction']}")
print(f"EN -> ES synthetic pairs               : {iteration_three_results['corpus_summary']['english_to_spanish_synthetic_pairs']}")
print(f"ES -> EN synthetic pairs               : {iteration_three_results['corpus_summary']['spanish_to_english_synthetic_pairs']}")
print(f"EN -> ES total training pairs          : {iteration_three_results['corpus_summary']['english_to_spanish_total_training_pairs']}")
print(f"ES -> EN total training pairs          : {iteration_three_results['corpus_summary']['spanish_to_english_total_training_pairs']}")

print("\nDIRECTIONAL ITERATION 3 RESULTS")
print("-------------------------------")
print("English -> Spanish")
print(f" - Starting checkpoint : {english_to_spanish_iteration_3_result['starting_model_checkpoint_directory']}")
print(f" - Output directory    : {english_to_spanish_iteration_3_result['output_directory']}")
print(f" - Direction           : {english_to_spanish_iteration_3_result['translation_direction']}")
print(f" - Authentic pairs     : {english_to_spanish_iteration_3_result['authentic_parallel_pairs_used']}")
print(f" - Synthetic pairs     : {english_to_spanish_iteration_3_result['synthetic_parallel_pairs_added']}")
print(f" - Total training pairs: {english_to_spanish_iteration_3_result['total_training_pairs']}")
print(f" - Dev pairs           : {english_to_spanish_iteration_3_result['development_pairs']}")
print(f" - Dev BLEU            : {english_to_spanish_iteration_3_result['dev_bleu']}")
print(f" - Dev gen_len         : {english_to_spanish_iteration_3_result['dev_gen_len']}")
print(f" - BLEU delta vs It. 2 : {english_to_spanish_iteration_3_bleu_delta_vs_iteration_2:+.4f}")

print("\nSpanish -> English")
print(f" - Starting checkpoint : {spanish_to_english_iteration_3_result['starting_model_checkpoint_directory']}")
print(f" - Output directory    : {spanish_to_english_iteration_3_result['output_directory']}")
print(f" - Direction           : {spanish_to_english_iteration_3_result['translation_direction']}")
print(f" - Authentic pairs     : {spanish_to_english_iteration_3_result['authentic_parallel_pairs_used']}")
print(f" - Synthetic pairs     : {spanish_to_english_iteration_3_result['synthetic_parallel_pairs_added']}")
print(f" - Total training pairs: {spanish_to_english_iteration_3_result['total_training_pairs']}")
print(f" - Dev pairs           : {spanish_to_english_iteration_3_result['development_pairs']}")
print(f" - Dev BLEU            : {spanish_to_english_iteration_3_result['dev_bleu']}")
print(f" - Dev gen_len         : {spanish_to_english_iteration_3_result['dev_gen_len']}")
print(f" - BLEU delta vs It. 2 : {spanish_to_english_iteration_3_bleu_delta_vs_iteration_2:+.4f}")

print("\nSaved combined Iteration 3 summary to:")
print(iteration_three_overall_summary_path)

print("The required three-iteration iterative back-translation workflow is now complete.")


ITERATION 3 FINAL SUMMARY
-------------------------
Iteration name                         : iteration_3
Authentic pairs / direction            : 1000
EN -> ES synthetic pairs               : 1000
ES -> EN synthetic pairs               : 1000
EN -> ES total training pairs          : 2000
ES -> EN total training pairs          : 2000

DIRECTIONAL ITERATION 3 RESULTS
-------------------------------
English -> Spanish
 - Starting checkpoint : /content/drive/MyDrive/anlp/iterative_backtranslation_en_to_es/iteration_2
 - Output directory    : /content/drive/MyDrive/anlp/iterative_backtranslation_en_to_es/iteration_3
 - Direction           : en->es
 - Authentic pairs     : 1000
 - Synthetic pairs     : 1000
 - Total training pairs: 2000
 - Dev pairs           : 200
 - Dev BLEU            : 44.551
 - Dev gen_len         : 29.8
 - BLEU delta vs It. 2 : -0.8576

Spanish -> English
 - Starting checkpoint : /content/drive/MyDrive/anlp/iterative_backtranslation_es_to_en/iteration_2
 - Output direc

## SECTION 10

This section uses development-set BLEU as the formal criterion to select the final model for each translation direction. It loads the saved summary file for every checkpoint—baseline, Iteration 1, Iteration 2, and Iteration 3—validates that all stages are present and checks that each file corresponds to the correct translation direction. It then organizes the results separately for English to Spanish and Spanish to English, compares their dev BLEU scores across all stages, and identifies the checkpoint with the highest value in each direction.


This cell defines the main infrastructure for development-set tracking and model selection. It loads the JSON summaries saved after each stage, validates that they are consistent, confirms that the translation direction is correct, and checks that the test set has not been used. It also normalizes the baseline and iterative summaries into a common format, extracts key variables such as development BLEU and training size, and prepares the logic to compare all stages properly.

In [45]:
import json
from pathlib import Path
from typing import Any, Dict, List


development_tracking_section_name = "development_set_tracking_and_model_selection"
development_tracking_summary_output_path = (
    corpus_root_directory / "development_set_tracking_and_model_selection_summary.json"
)

development_tracking_stage_registry = {
    "iteration_0_baseline": {
        "stage_order": 0,
        "display_name": "Iteration 0 (baseline)",
    },
    "iteration_1": {
        "stage_order": 1,
        "display_name": "Iteration 1",
    },
    "iteration_2": {
        "stage_order": 2,
        "display_name": "Iteration 2",
    },
    "iteration_3": {
        "stage_order": 3,
        "display_name": "Iteration 3",
    },
}

development_tracking_direction_registry = {
    "english_to_spanish": {
        "display_name": "English -> Spanish",
        "expected_translation_direction": f"{source_language_code}->{target_language_code}",
        "model_name": forward_translation_model_name,
        "summary_paths_by_stage": {
            "iteration_0_baseline": baseline_forward_model_output_directory / "iteration_0_baseline_summary.json",
            "iteration_1": iteration_one_forward_model_output_directory / "iteration_1_summary.json",
            "iteration_2": iteration_two_forward_model_output_directory / "iteration_2_summary.json",
            "iteration_3": iteration_three_forward_model_output_directory / "iteration_3_summary.json",
        },
    },
    "spanish_to_english": {
        "display_name": "Spanish -> English",
        "expected_translation_direction": f"{target_language_code}->{source_language_code}",
        "model_name": backward_translation_model_name,
        "summary_paths_by_stage": {
            "iteration_0_baseline": baseline_backward_model_output_directory / "iteration_0_baseline_summary.json",
            "iteration_1": iteration_one_backward_model_output_directory / "iteration_1_summary.json",
            "iteration_2": iteration_two_backward_model_output_directory / "iteration_2_summary.json",
            "iteration_3": iteration_three_backward_model_output_directory / "iteration_3_summary.json",
        },
    },
}


def load_json_dictionary(json_file_path: Path) -> Dict[str, Any]:
    json_file_path = Path(json_file_path)

    if not json_file_path.exists():
        raise FileNotFoundError(
            f"Required JSON summary file not found: {json_file_path}. "
            "Make sure the previous training/evaluation cells have been executed."
        )

    with json_file_path.open("r", encoding="utf-8") as input_file:
        loaded_object = json.load(input_file)

    if not isinstance(loaded_object, dict):
        raise ValueError(f"The JSON file does not contain a dictionary: {json_file_path}")

    return loaded_object


def save_json_dictionary(json_file_path: Path, dictionary_to_save: Dict[str, Any]) -> None:
    json_file_path = Path(json_file_path)
    json_file_path.parent.mkdir(parents=True, exist_ok=True)

    with json_file_path.open("w", encoding="utf-8") as output_file:
        json.dump(dictionary_to_save, output_file, indent=2, ensure_ascii=False)


def extract_total_training_pairs_from_directional_summary(
    directional_summary: Dict[str, Any]
) -> int:
    if "total_training_pairs" in directional_summary:
        return int(directional_summary["total_training_pairs"])

    if "train_pairs" in directional_summary:
        return int(directional_summary["train_pairs"])

    raise KeyError(
        "Could not determine the total number of training pairs from the directional summary."
    )


def extract_authentic_parallel_pairs_from_directional_summary(
    directional_summary: Dict[str, Any]
) -> int:
    if "authentic_parallel_pairs_used" in directional_summary:
        return int(directional_summary["authentic_parallel_pairs_used"])

    if "train_pairs" in directional_summary:
        return int(directional_summary["train_pairs"])

    raise KeyError(
        "Could not determine the authentic parallel pair count from the directional summary."
    )


def validate_loaded_directional_summary(
    directional_summary: Dict[str, Any],
    expected_translation_direction: str,
    summary_file_path: Path,
) -> None:
    required_keys = [
        "translation_direction",
        "output_directory",
        "dev_bleu",
        "dev_gen_len",
        "test_set_used",
    ]

    missing_keys = [key for key in required_keys if key not in directional_summary]
    if missing_keys:
        raise KeyError(
            f"Directional summary file {summary_file_path} is missing keys: {missing_keys}"
        )

    if directional_summary["translation_direction"] != expected_translation_direction:
        raise ValueError(
            f"Directional summary {summary_file_path} has translation direction "
            f"{directional_summary['translation_direction']}, but {expected_translation_direction} was expected."
        )

    if bool(directional_summary["test_set_used"]):
        raise ValueError(
            f"Invalid methodological state in {summary_file_path}: "
            "the test set must not have been used before final evaluation."
        )

    checkpoint_directory = Path(directional_summary["output_directory"])
    if not checkpoint_directory.exists():
        raise FileNotFoundError(
            f"The checkpoint directory recorded in {summary_file_path} does not exist: "
            f"{checkpoint_directory}"
        )


def build_development_tracking_record_from_saved_summary(
    stage_key: str,
    direction_key: str,
) -> Dict[str, Any]:
    stage_metadata = development_tracking_stage_registry[stage_key]
    direction_metadata = development_tracking_direction_registry[direction_key]

    summary_file_path = direction_metadata["summary_paths_by_stage"][stage_key]
    directional_summary = load_json_dictionary(summary_file_path)

    validate_loaded_directional_summary(
        directional_summary=directional_summary,
        expected_translation_direction=direction_metadata["expected_translation_direction"],
        summary_file_path=summary_file_path,
    )

    starting_checkpoint_directory = directional_summary.get(
        "starting_model_checkpoint_directory", None
    )

    development_tracking_record = {
        "stage_key": stage_key,
        "stage_order": int(stage_metadata["stage_order"]),
        "stage_display_name": stage_metadata["display_name"],
        "direction_key": direction_key,
        "direction_display_name": direction_metadata["display_name"],
        "model_name": direction_metadata["model_name"],
        "translation_direction": directional_summary["translation_direction"],
        "checkpoint_directory": str(Path(directional_summary["output_directory"])),
        "summary_file_path": str(summary_file_path),
        "starting_checkpoint_directory": (
            str(Path(starting_checkpoint_directory))
            if starting_checkpoint_directory is not None
            else None
        ),
        "dev_bleu": round(float(directional_summary["dev_bleu"]), 4),
        "dev_gen_len": round(float(directional_summary["dev_gen_len"]), 4),
        "total_training_pairs": extract_total_training_pairs_from_directional_summary(
            directional_summary
        ),
        "authentic_parallel_pairs_used": extract_authentic_parallel_pairs_from_directional_summary(
            directional_summary
        ),
        "synthetic_parallel_pairs_added": int(
            directional_summary.get("synthetic_parallel_pairs_added", 0)
        ),
        "monolingual_data_used": bool(
            directional_summary.get("monolingual_data_used", False)
        ),
        "test_set_used": bool(directional_summary["test_set_used"]),
    }

    return development_tracking_record


def validate_directional_stage_coverage_for_selection(
    development_tracking_records: List[Dict[str, Any]],
    direction_key: str,
) -> None:
    direction_records = [
        record
        for record in development_tracking_records
        if record["direction_key"] == direction_key
    ]

    expected_stage_keys = set(development_tracking_stage_registry.keys())
    observed_stage_keys = {record["stage_key"] for record in direction_records}

    if observed_stage_keys != expected_stage_keys:
        raise ValueError(
            f"Direction {direction_key} does not contain the full required stage set. "
            f"Observed: {sorted(observed_stage_keys)} | Expected: {sorted(expected_stage_keys)}"
        )

    if len(direction_records) != len(expected_stage_keys):
        raise ValueError(
            f"Direction {direction_key} contains repeated or missing stage records."
        )


def group_development_tracking_records_by_direction(
    development_tracking_records: List[Dict[str, Any]]
) -> Dict[str, List[Dict[str, Any]]]:
    grouped_records: Dict[str, List[Dict[str, Any]]] = {}

    for record in development_tracking_records:
        grouped_records.setdefault(record["direction_key"], []).append(record)

    for direction_key in grouped_records:
        grouped_records[direction_key] = sorted(
            grouped_records[direction_key],
            key=lambda record: record["stage_order"],
        )

    return grouped_records


def validate_development_tracking_records(
    development_tracking_records: List[Dict[str, Any]]
) -> None:
    if len(development_tracking_records) == 0:
        raise ValueError("The development tracking record list is empty.")

    grouped_records = group_development_tracking_records_by_direction(
        development_tracking_records
    )

    expected_direction_keys = set(development_tracking_direction_registry.keys())
    observed_direction_keys = set(grouped_records.keys())

    if observed_direction_keys != expected_direction_keys:
        raise ValueError(
            f"The development tracking records do not cover the expected directions. "
            f"Observed: {sorted(observed_direction_keys)} | Expected: {sorted(expected_direction_keys)}"
        )

    for direction_key in expected_direction_keys:
        validate_directional_stage_coverage_for_selection(
            development_tracking_records=development_tracking_records,
            direction_key=direction_key,
        )

        direction_records = grouped_records[direction_key]

        if any(record["test_set_used"] for record in direction_records):
            raise ValueError(
                f"Direction {direction_key} contains records that incorrectly report test-set usage."
            )


def load_all_development_tracking_records_from_saved_summaries(
    verbose: bool = True
) -> List[Dict[str, Any]]:
    all_development_tracking_records: List[Dict[str, Any]] = []

    for direction_key in development_tracking_direction_registry.keys():
        for stage_key in development_tracking_stage_registry.keys():
            all_development_tracking_records.append(
                build_development_tracking_record_from_saved_summary(
                    stage_key=stage_key,
                    direction_key=direction_key,
                )
            )

    validate_development_tracking_records(all_development_tracking_records)

    if verbose:
        print("Tracking records were loaded successfully.")
        print(f"Loaded directional stage records: {len(all_development_tracking_records)}")
        print("Source of truth: per-checkpoint summary JSON files saved in each model directory.")
        print("This ensures that model selection is based on saved evaluation results,")
        print("not on transient in-memory notebook state.")

    return all_development_tracking_records


def build_directional_development_tracking_rows(
    development_tracking_records: List[Dict[str, Any]],
    direction_key: str,
) -> List[Dict[str, Any]]:
    validate_directional_stage_coverage_for_selection(
        development_tracking_records=development_tracking_records,
        direction_key=direction_key,
    )

    direction_records = sorted(
        [
            record
            for record in development_tracking_records
            if record["direction_key"] == direction_key
        ],
        key=lambda record: record["stage_order"],
    )

    baseline_dev_bleu = float(direction_records[0]["dev_bleu"])
    previous_stage_dev_bleu = None

    directional_rows: List[Dict[str, Any]] = []

    for record in direction_records:
        row = dict(record)
        row["dev_bleu_delta_vs_baseline"] = round(
            float(record["dev_bleu"]) - baseline_dev_bleu,
            4,
        )

        if previous_stage_dev_bleu is None:
            row["dev_bleu_delta_vs_previous_stage"] = None
        else:
            row["dev_bleu_delta_vs_previous_stage"] = round(
                float(record["dev_bleu"]) - previous_stage_dev_bleu,
                4,
            )

        directional_rows.append(row)
        previous_stage_dev_bleu = float(record["dev_bleu"])

    return directional_rows


def select_best_dev_checkpoint_for_direction(
    development_tracking_records: List[Dict[str, Any]],
    direction_key: str,
) -> Dict[str, Any]:
    directional_rows = build_directional_development_tracking_rows(
        development_tracking_records=development_tracking_records,
        direction_key=direction_key,
    )

    baseline_row = directional_rows[0]

    best_row = sorted(
        directional_rows,
        key=lambda row: (-float(row["dev_bleu"]), int(row["stage_order"])),
    )[0]

    return {
        "direction_key": best_row["direction_key"],
        "direction_display_name": best_row["direction_display_name"],
        "translation_direction": best_row["translation_direction"],
        "model_name": best_row["model_name"],
        "selected_stage_key": best_row["stage_key"],
        "selected_stage_display_name": best_row["stage_display_name"],
        "selected_checkpoint_directory": best_row["checkpoint_directory"],
        "selected_summary_file_path": best_row["summary_file_path"],
        "selected_dev_bleu": best_row["dev_bleu"],
        "selected_dev_gen_len": best_row["dev_gen_len"],
        "baseline_stage_key": baseline_row["stage_key"],
        "baseline_dev_bleu": baseline_row["dev_bleu"],
        "dev_bleu_gain_vs_baseline": round(
            float(best_row["dev_bleu"]) - float(baseline_row["dev_bleu"]),
            4,
        ),
        "iterative_backtranslation_improved_over_baseline": bool(
            float(best_row["dev_bleu"]) > float(baseline_row["dev_bleu"])
            and best_row["stage_key"] != "iteration_0_baseline"
        ),
        "baseline_remains_best_checkpoint": bool(
            best_row["stage_key"] == "iteration_0_baseline"
        ),
        "tie_break_policy": "earliest_stage_if_dev_bleu_is_identical",
        "all_stage_dev_bleu": {
            row["stage_key"]: row["dev_bleu"]
            for row in directional_rows
        },
    }


def build_development_model_selection_summary(
    development_tracking_records: List[Dict[str, Any]]
) -> Dict[str, Any]:
    validate_development_tracking_records(development_tracking_records)

    english_to_spanish_rows = build_directional_development_tracking_rows(
        development_tracking_records=development_tracking_records,
        direction_key="english_to_spanish",
    )
    spanish_to_english_rows = build_directional_development_tracking_rows(
        development_tracking_records=development_tracking_records,
        direction_key="spanish_to_english",
    )

    english_to_spanish_selection = select_best_dev_checkpoint_for_direction(
        development_tracking_records=development_tracking_records,
        direction_key="english_to_spanish",
    )
    spanish_to_english_selection = select_best_dev_checkpoint_for_direction(
        development_tracking_records=development_tracking_records,
        direction_key="spanish_to_english",
    )

    return {
        "section_name": development_tracking_section_name,
        "selection_criterion": (
            "highest development-set BLEU independently for each translation direction"
        ),
        "test_set_used_for_selection": False,
        "tie_break_policy": "earliest stage if dev BLEU is identical",
        "directional_tracking": {
            "english_to_spanish": english_to_spanish_rows,
            "spanish_to_english": spanish_to_english_rows,
        },
        "final_selected_checkpoints": {
            "english_to_spanish": english_to_spanish_selection,
            "spanish_to_english": spanish_to_english_selection,
        },
    }


def format_optional_delta(delta_value: Any) -> str:
    if delta_value is None:
        return "n/a"
    return f"{float(delta_value):+.4f}"


def print_directional_development_tracking_report(
    development_tracking_records: List[Dict[str, Any]],
    direction_key: str,
) -> None:
    directional_rows = build_directional_development_tracking_rows(
        development_tracking_records=development_tracking_records,
        direction_key=direction_key,
    )
    selected_checkpoint_summary = select_best_dev_checkpoint_for_direction(
        development_tracking_records=development_tracking_records,
        direction_key=direction_key,
    )

    direction_display_name = directional_rows[0]["direction_display_name"]
    report_title = f"DEVELOPMENT-SET TRACKING: {direction_display_name}"

    print(report_title)
    print("-" * len(report_title))
    print(
        f"{'Stage':<24}"
        f"{'BLEU':>10}"
        f"{'Δ vs base':>14}"
        f"{'Δ vs prev':>14}"
        f"{'Total pairs':>14}"
        f"{'Synthetic':>12}"
    )

    for row in directional_rows:
        print(
            f"{row['stage_display_name']:<24}"
            f"{row['dev_bleu']:>10.4f}"
            f"{format_optional_delta(row['dev_bleu_delta_vs_baseline']):>14}"
            f"{format_optional_delta(row['dev_bleu_delta_vs_previous_stage']):>14}"
            f"{row['total_training_pairs']:>14}"
            f"{row['synthetic_parallel_pairs_added']:>12}"
        )

    print("\nSelected checkpoint for final test evaluation:")
    print(f" - Stage               : {selected_checkpoint_summary['selected_stage_display_name']}")
    print(f" - Model name          : {selected_checkpoint_summary['model_name']}")
    print(f" - Translation         : {selected_checkpoint_summary['translation_direction']}")
    print(f" - Dev BLEU            : {selected_checkpoint_summary['selected_dev_bleu']}")
    print(f" - Checkpoint directory: {selected_checkpoint_summary['selected_checkpoint_directory']}")

    if selected_checkpoint_summary["iterative_backtranslation_improved_over_baseline"]:
        print(" - Interpretation      : Iterative back-translation improved this direction over the baseline.")
    elif selected_checkpoint_summary["baseline_remains_best_checkpoint"]:
        print(" - Interpretation      : The baseline remains the best dev-set checkpoint for this direction.")
    else:
        print(" - Interpretation      : No dev-set gain over the baseline was obtained.")

This code applies the tracking framework to all saved stages and selects the final checkpoint for each translation direction using only development BLEU. It loads all records, prints the stage-by-stage comparison for English to Spanish and Spanish to English, computes the final selection summary, and saves this information as a JSON file. In addition, it stores the chosen checkpoint paths in explicit variables for later use in the final evaluation.

In [46]:
development_tracking_records = load_all_development_tracking_records_from_saved_summaries(
    verbose=True
)

print()
print_directional_development_tracking_report(
    development_tracking_records=development_tracking_records,
    direction_key="english_to_spanish",
)

print()
print_directional_development_tracking_report(
    development_tracking_records=development_tracking_records,
    direction_key="spanish_to_english",
)

development_set_model_selection_summary = build_development_model_selection_summary(
    development_tracking_records=development_tracking_records
)

save_json_dictionary(
    json_file_path=development_tracking_summary_output_path,
    dictionary_to_save=development_set_model_selection_summary,
)

final_selected_english_to_spanish_checkpoint_directory = Path(
    development_set_model_selection_summary["final_selected_checkpoints"]["english_to_spanish"][
        "selected_checkpoint_directory"
    ]
)

final_selected_spanish_to_english_checkpoint_directory = Path(
    development_set_model_selection_summary["final_selected_checkpoints"]["spanish_to_english"][
        "selected_checkpoint_directory"
    ]
)

final_selected_forward_model_checkpoint_directory = final_selected_english_to_spanish_checkpoint_directory
final_selected_backward_model_checkpoint_directory = final_selected_spanish_to_english_checkpoint_directory

final_selected_model_registry = {
    "english_to_spanish": development_set_model_selection_summary["final_selected_checkpoints"][
        "english_to_spanish"
    ],
    "spanish_to_english": development_set_model_selection_summary["final_selected_checkpoints"][
        "spanish_to_english"
    ],
}

assert final_selected_english_to_spanish_checkpoint_directory.exists(), \
    "The selected English->Spanish checkpoint directory does not exist."

assert final_selected_spanish_to_english_checkpoint_directory.exists(), \
    "The selected Spanish->English checkpoint directory does not exist."

assert development_set_model_selection_summary["test_set_used_for_selection"] is False, \
    "The test set must not be used for model selection."

print("\nFINAL MODEL SELECTION FOR TEST EVALUATION")
print("-----------------------------------------")
print("English -> Spanish")
print(" - Selected stage       :",
      final_selected_model_registry["english_to_spanish"]["selected_stage_display_name"])
print(" - Selected dev BLEU    :",
      final_selected_model_registry["english_to_spanish"]["selected_dev_bleu"])
print(" - Selected checkpoint  :",
      final_selected_english_to_spanish_checkpoint_directory)

print("\nSpanish -> English")
print(" - Selected stage       :",
      final_selected_model_registry["spanish_to_english"]["selected_stage_display_name"])
print(" - Selected dev BLEU    :",
      final_selected_model_registry["spanish_to_english"]["selected_dev_bleu"])
print(" - Selected checkpoint  :",
      final_selected_spanish_to_english_checkpoint_directory)

print("\nMETHODOLOGICAL CHECKS")
print("---------------------")
print(" - Selection criterion  : highest dev BLEU per direction")
print(" - Test set used        : No")
print(" - Selection is independent for EN->ES and ES->EN")
print(" - This section closes the development-monitoring phase")

print(development_tracking_summary_output_path)

Tracking records were loaded successfully.
Loaded directional stage records: 8
Source of truth: per-checkpoint summary JSON files saved in each model directory.
This ensures that model selection is based on saved evaluation results,
not on transient in-memory notebook state.

DEVELOPMENT-SET TRACKING: English -> Spanish
--------------------------------------------
Stage                         BLEU     Δ vs base     Δ vs prev   Total pairs   Synthetic
Iteration 0 (baseline)     45.9389       +0.0000           n/a          1000           0
Iteration 1                45.7404       -0.1985       -0.1985          2000        1000
Iteration 2                45.4086       -0.5303       -0.3318          2000        1000
Iteration 3                44.5510       -1.3879       -0.8576          2000        1000

Selected checkpoint for final test evaluation:
 - Stage               : Iteration 0 (baseline)
 - Model name          : Helsinki-NLP/opus-mt-en-es
 - Translation         : en->es
 - Dev B

## SECTION 11

This is the final section, which is intended to close the experimental pipeline by carrying out the only unbiased final evaluation.

In [47]:
import json
from pathlib import Path
from typing import Any, Dict, List, Tuple

import evaluate
from datasets import Dataset

final_evaluation_section_name = "final_evaluation_on_test_set"
final_evaluation_output_directory = corpus_root_directory / final_evaluation_section_name
final_evaluation_output_directory.mkdir(parents=True, exist_ok=True)

final_test_bleu_metric = evaluate.load("sacrebleu")


def load_and_validate_development_based_selection_summary(
    selection_summary_path: Path,
) -> Dict[str, Any]:
    selection_summary_path = Path(selection_summary_path)

    if not selection_summary_path.exists():
        raise FileNotFoundError(
            f"Selection summary not found: {selection_summary_path}. "
            "."
        )

    selection_summary = load_json_dictionary(selection_summary_path)

    required_top_level_keys = [
        "selection_criterion",
        "test_set_used_for_selection",
        "final_selected_checkpoints",
    ]
    missing_top_level_keys = [
        key for key in required_top_level_keys if key not in selection_summary
    ]
    if missing_top_level_keys:
        raise KeyError(
            "The selection summary is missing required keys: "
            f"{missing_top_level_keys}"
        )

    if bool(selection_summary["test_set_used_for_selection"]):
        raise ValueError(
            "Invalid methodological state: the test set must not be used for model selection."
        )

    expected_direction_keys = set(development_tracking_direction_registry.keys())
    observed_direction_keys = set(selection_summary["final_selected_checkpoints"].keys())

    if observed_direction_keys != expected_direction_keys:
        raise ValueError(
            f"Unexpected direction coverage in the selection summary. "
            f"Observed: {sorted(observed_direction_keys)} | "
            f"Expected: {sorted(expected_direction_keys)}"
        )

    for direction_key, selected_checkpoint_summary in selection_summary["final_selected_checkpoints"].items():
        required_direction_keys = [
            "direction_display_name",
            "translation_direction",
            "model_name",
            "selected_stage_key",
            "selected_stage_display_name",
            "selected_checkpoint_directory",
            "selected_dev_bleu",
        ]
        missing_direction_keys = [
            key for key in required_direction_keys
            if key not in selected_checkpoint_summary
        ]
        if missing_direction_keys:
            raise KeyError(
                f"The selected-checkpoint summary for {direction_key} is missing keys: "
                f"{missing_direction_keys}"
            )

        expected_translation_direction = development_tracking_direction_registry[
            direction_key
        ]["expected_translation_direction"]

        if selected_checkpoint_summary["translation_direction"] != expected_translation_direction:
            raise ValueError(
                f"Direction mismatch for {direction_key}: "
                f"expected {expected_translation_direction}, but found "
                f"{selected_checkpoint_summary['translation_direction']}."
            )

        selected_stage_key = selected_checkpoint_summary["selected_stage_key"]
        if selected_stage_key not in development_tracking_stage_registry:
            raise ValueError(
                f"Unknown selected stage key for {direction_key}: {selected_stage_key}"
            )

        selected_checkpoint_directory = Path(
            selected_checkpoint_summary["selected_checkpoint_directory"]
        )
        if not selected_checkpoint_directory.exists():
            raise FileNotFoundError(
                f"Selected checkpoint directory does not exist for {direction_key}: "
                f"{selected_checkpoint_directory}"
            )

    return selection_summary


def extract_ordered_source_and_reference_texts_from_translation_dataset(
    parallel_translation_dataset: Dataset,
    source_language_code: str,
    target_language_code: str,
) -> Tuple[List[str], List[str]]:
    if len(parallel_translation_dataset) == 0:
        raise ValueError("The evaluation dataset is empty.")

    source_texts: List[str] = []
    reference_texts: List[str] = []

    for example_index, example in enumerate(parallel_translation_dataset):
        if "translation" not in example:
            raise KeyError(
                f"Missing 'translation' field at example index {example_index}."
            )

        translation_dictionary = example["translation"]

        if source_language_code not in translation_dictionary:
            raise KeyError(
                f"Missing source language '{source_language_code}' at example index {example_index}."
            )
        if target_language_code not in translation_dictionary:
            raise KeyError(
                f"Missing target language '{target_language_code}' at example index {example_index}."
            )

        source_text = translation_dictionary[source_language_code].strip()
        reference_text = translation_dictionary[target_language_code].strip()

        if source_text == "" or reference_text == "":
            raise ValueError(
                f"Blank source or reference segment detected at example index {example_index}."
            )

        source_texts.append(source_text)
        reference_texts.append(reference_text)

    return source_texts, reference_texts


def compute_final_bleu_from_predictions_and_references(
    predicted_texts: List[str],
    reference_texts: List[str],
) -> Dict[str, Any]:
    if len(predicted_texts) == 0:
        raise ValueError("The prediction list is empty.")
    if len(reference_texts) == 0:
        raise ValueError("The reference list is empty.")
    if len(predicted_texts) != len(reference_texts):
        raise ValueError(
            f"Prediction/reference length mismatch: "
            f"{len(predicted_texts)} predictions vs {len(reference_texts)} references."
        )

    cleaned_predictions, cleaned_references = postprocess_bleu_texts(
        predicted_texts,
        reference_texts,
    )

    bleu_result = final_test_bleu_metric.compute(
        predictions=cleaned_predictions,
        references=cleaned_references,
    )

    return {
        "bleu": round(float(bleu_result["score"]), 4),
        "precisions": [round(float(value), 4) for value in bleu_result["precisions"]],
        "bp": round(float(bleu_result["bp"]), 4),
        "sys_len": int(bleu_result["sys_len"]),
        "ref_len": int(bleu_result["ref_len"]),
    }


def evaluate_selected_checkpoint_on_held_out_test_set(
    selected_checkpoint_summary: Dict[str, Any],
    evaluation_parallel_dataset: Dataset,
    source_language_code: str,
    target_language_code: str,
    expected_test_size: int,
    direction_output_subdirectory_name: str,
    progress_description: str,
) -> Dict[str, Any]:
    if len(evaluation_parallel_dataset) != expected_test_size:
        raise ValueError(
            f"The held-out test dataset has {len(evaluation_parallel_dataset)} examples, "
            f"but {expected_test_size} were expected."
        )

    selected_checkpoint_directory = Path(
        selected_checkpoint_summary["selected_checkpoint_directory"]
    )

    if not selected_checkpoint_directory.exists():
        raise FileNotFoundError(
            f"Selected checkpoint directory not found: {selected_checkpoint_directory}"
        )

    direction_output_directory = (
        final_evaluation_output_directory / direction_output_subdirectory_name
    )
    prepare_clean_output_directory(direction_output_directory)

    test_source_texts, gold_reference_texts = (
        extract_ordered_source_and_reference_texts_from_translation_dataset(
            parallel_translation_dataset=evaluation_parallel_dataset,
            source_language_code=source_language_code,
            target_language_code=target_language_code,
        )
    )

    generated_hypotheses = generate_translations_from_checkpoint_in_batches(
        input_texts=test_source_texts,
        model_checkpoint_directory=selected_checkpoint_directory,
        batch_size=evaluation_batch_size,
        max_input_length=max_source_sequence_length,
        max_new_tokens=max_target_sequence_length,
        progress_description=progress_description,
    )

    final_bleu_summary = compute_final_bleu_from_predictions_and_references(
        predicted_texts=generated_hypotheses,
        reference_texts=gold_reference_texts,
    )

    source_output_path = direction_output_directory / f"test_sources.{source_language_code}"
    reference_output_path = direction_output_directory / f"test_references.{target_language_code}"
    hypothesis_output_path = direction_output_directory / f"test_hypotheses.{target_language_code}"
    directional_summary_output_path = (
        direction_output_directory / "final_test_directional_summary.json"
    )

    write_lines_to_file(source_output_path, test_source_texts)
    write_lines_to_file(reference_output_path, gold_reference_texts)
    write_lines_to_file(hypothesis_output_path, generated_hypotheses)

    directional_test_result = {
        "direction_display_name": selected_checkpoint_summary["direction_display_name"],
        "translation_direction": selected_checkpoint_summary["translation_direction"],
        "model_name": selected_checkpoint_summary["model_name"],
        "selected_stage_key": selected_checkpoint_summary["selected_stage_key"],
        "selected_stage_display_name": selected_checkpoint_summary["selected_stage_display_name"],
        "selected_checkpoint_directory": str(selected_checkpoint_directory),
        "selected_dev_bleu": round(float(selected_checkpoint_summary["selected_dev_bleu"]), 4),
        "test_set_size": len(evaluation_parallel_dataset),
        "test_set_used_for_selection": False,
        "test_set_role": "confirmatory final evaluation only",
        "test_bleu": final_bleu_summary["bleu"],
        "test_bleu_details": final_bleu_summary,
        "source_output_path": str(source_output_path),
        "reference_output_path": str(reference_output_path),
        "hypothesis_output_path": str(hypothesis_output_path),
    }

    save_json_dictionary(
        json_file_path=directional_summary_output_path,
        dictionary_to_save=directional_test_result,
    )

    return directional_test_result


def build_final_test_evaluation_summary(
    development_based_selection_summary: Dict[str, Any],
    english_to_spanish_test_result: Dict[str, Any],
    spanish_to_english_test_result: Dict[str, Any],
) -> Dict[str, Any]:
    return {
        "section_name": final_evaluation_section_name,
        "selection_criterion": development_based_selection_summary["selection_criterion"],
        "test_set_used_for_selection": False,
        "test_set_role": "final confirmatory evaluation only",
        "held_out_test_set_size": test_parallel_size,
        "final_test_results": {
            "english_to_spanish": english_to_spanish_test_result,
            "spanish_to_english": spanish_to_english_test_result,
        },
    }

In [48]:
development_based_selection_summary = load_and_validate_development_based_selection_summary(
    selection_summary_path=development_tracking_summary_output_path
)

print("FINAL TEST-EVALUATION PRECONDITIONS")
print("-----------------------------------")
print("Selection summary loaded from:", development_tracking_summary_output_path)
print("Test set used for model selection:",
      development_based_selection_summary["test_set_used_for_selection"])
print("Held-out test set size:", len(test_dataset))

assert development_based_selection_summary["test_set_used_for_selection"] is False, \
    "The test set must not be used for model selection."
assert len(test_dataset) == test_parallel_size, \
    f"The held-out test dataset must contain exactly {test_parallel_size} examples."

print("\nSelected checkpoints coming from Section 17")
print("-------------------------------------------")
print("English -> Spanish:")
print(" - Selected stage      :",
      development_based_selection_summary["final_selected_checkpoints"]["english_to_spanish"]["selected_stage_display_name"])
print(" - Selected checkpoint :",
      development_based_selection_summary["final_selected_checkpoints"]["english_to_spanish"]["selected_checkpoint_directory"])
print(" - Selected dev BLEU   :",
      development_based_selection_summary["final_selected_checkpoints"]["english_to_spanish"]["selected_dev_bleu"])

print("\nSpanish -> English:")
print(" - Selected stage      :",
      development_based_selection_summary["final_selected_checkpoints"]["spanish_to_english"]["selected_stage_display_name"])
print(" - Selected checkpoint :",
      development_based_selection_summary["final_selected_checkpoints"]["spanish_to_english"]["selected_checkpoint_directory"])
print(" - Selected dev BLEU   :",
      development_based_selection_summary["final_selected_checkpoints"]["spanish_to_english"]["selected_dev_bleu"])


english_to_spanish_final_test_result = evaluate_selected_checkpoint_on_held_out_test_set(
    selected_checkpoint_summary=development_based_selection_summary["final_selected_checkpoints"]["english_to_spanish"],
    evaluation_parallel_dataset=test_dataset,
    source_language_code=source_language_code,   # en
    target_language_code=target_language_code,   # es
    expected_test_size=test_parallel_size,
    direction_output_subdirectory_name="english_to_spanish",
    progress_description="Final test evaluation: translating EN test sentences with the selected EN->ES checkpoint",
)

spanish_to_english_final_test_result = evaluate_selected_checkpoint_on_held_out_test_set(
    selected_checkpoint_summary=development_based_selection_summary["final_selected_checkpoints"]["spanish_to_english"],
    evaluation_parallel_dataset=test_dataset,
    source_language_code=target_language_code,   # es
    target_language_code=source_language_code,   # en
    expected_test_size=test_parallel_size,
    direction_output_subdirectory_name="spanish_to_english",
    progress_description="Final test evaluation: translating ES test sentences with the selected ES->EN checkpoint",
)

print("\nDirectional final test evaluation finished successfully.")
print("English -> Spanish final test BLEU :", english_to_spanish_final_test_result["test_bleu"])
print("Spanish -> English final test BLEU :", spanish_to_english_final_test_result["test_bleu"])

FINAL TEST-EVALUATION PRECONDITIONS
-----------------------------------
Selection summary loaded from: /content/drive/MyDrive/anlp/development_set_tracking_and_model_selection_summary.json
Test set used for model selection: False
Held-out test set size: 200

Selected checkpoints coming from Section 17
-------------------------------------------
English -> Spanish:
 - Selected stage      : Iteration 0 (baseline)
 - Selected checkpoint : /content/drive/MyDrive/anlp/iterative_backtranslation_en_to_es/iteration_0_baseline
 - Selected dev BLEU   : 45.9389

Spanish -> English:
 - Selected stage      : Iteration 1
 - Selected checkpoint : /content/drive/MyDrive/anlp/iterative_backtranslation_es_to_en/iteration_1
 - Selected dev BLEU   : 49.7014


Final test evaluation: translating EN test sentences with the selected EN->ES checkpoint:   0%|          | 0/1…

Final test evaluation: translating ES test sentences with the selected ES->EN checkpoint:   0%|          | 0/1…


Directional final test evaluation finished successfully.
English -> Spanish final test BLEU : 43.2386
Spanish -> English final test BLEU : 47.3002


In [50]:
final_test_evaluation_summary = build_final_test_evaluation_summary(
    development_based_selection_summary=development_based_selection_summary,
    english_to_spanish_test_result=english_to_spanish_final_test_result,
    spanish_to_english_test_result=spanish_to_english_final_test_result,
)

final_test_evaluation_summary_output_path = (
    final_evaluation_output_directory / "final_test_evaluation_summary.json"
)

save_json_dictionary(
    json_file_path=final_test_evaluation_summary_output_path,
    dictionary_to_save=final_test_evaluation_summary,
)

final_english_to_spanish_test_bleu = english_to_spanish_final_test_result["test_bleu"]
final_spanish_to_english_test_bleu = spanish_to_english_final_test_result["test_bleu"]

print("FINAL EVALUATION ON THE TEST SET")
print("--------------------------------")

print("ENGLISH -> SPANISH")
print("------------------")
print(f"Selected stage                  : {english_to_spanish_final_test_result['selected_stage_display_name']}")
print(f"Selected checkpoint directory   : {english_to_spanish_final_test_result['selected_checkpoint_directory']}")
print(f"Development BLEU used for choice: {english_to_spanish_final_test_result['selected_dev_bleu']}")
print(f"Final test BLEU                 : {final_english_to_spanish_test_bleu}")
print(f"Hypotheses file                 : {english_to_spanish_final_test_result['hypothesis_output_path']}")

print("\nSPANISH -> ENGLISH")
print("------------------")
print(f"Selected stage                  : {spanish_to_english_final_test_result['selected_stage_display_name']}")
print(f"Selected checkpoint directory   : {spanish_to_english_final_test_result['selected_checkpoint_directory']}")
print(f"Development BLEU used for choice: {spanish_to_english_final_test_result['selected_dev_bleu']}")
print(f"Final test BLEU                 : {final_spanish_to_english_test_bleu}")
print(f"Hypotheses file                 : {spanish_to_english_final_test_result['hypothesis_output_path']}")

print("\nMETHODOLOGICAL CHECKS")
print("---------------------")
print(" - Model selection criterion : highest development BLEU per direction")
print(" - Test set used for choice  : No")
print(" - Test set role             : confirmatory final evaluation only")
print(" - EN->ES and ES->EN selected independently")

print(final_test_evaluation_summary_output_path)


FINAL EVALUATION ON THE TEST SET
--------------------------------
ENGLISH -> SPANISH
------------------
Selected stage                  : Iteration 0 (baseline)
Selected checkpoint directory   : /content/drive/MyDrive/anlp/iterative_backtranslation_en_to_es/iteration_0_baseline
Development BLEU used for choice: 45.9389
Final test BLEU                 : 43.2386
Hypotheses file                 : /content/drive/MyDrive/anlp/final_evaluation_on_test_set/english_to_spanish/test_hypotheses.es

SPANISH -> ENGLISH
------------------
Selected stage                  : Iteration 1
Selected checkpoint directory   : /content/drive/MyDrive/anlp/iterative_backtranslation_es_to_en/iteration_1
Development BLEU used for choice: 49.7014
Final test BLEU                 : 47.3002
Hypotheses file                 : /content/drive/MyDrive/anlp/final_evaluation_on_test_set/spanish_to_english/test_hypotheses.en

METHODOLOGICAL CHECKS
---------------------
 - Model selection criterion : highest development BLEU 

# **FINAL CONCLUSION**

 Under the followed methodology, the best English→Spanish system has been the baseline model from Iteration 0, with a development BLEU of 45.9389 and a final test BLEU of 43.2386, whereas the best Spanish→English system has been the Iteration 1 model, with a development BLEU of 49.7014 and a final test BLEU of 47.3002. These results indicate that iterative back-translation was not uniformly beneficial across all stages: it produced a limited improvement in Spanish→English at the first synthetic augmentation step, but later iterations degraded performance in both directions. This behavior **is not contradictory**, but expected in **a quite small data** setting such as this one, where only 1,000 authentic parallel pairs were available and the synthetic data added in each round were as numerous as the authentic data. In such conditions, the first iteration may help because it introduces useful additional bilingual signal from monolingual corpora, especially when the starting opposite-direction model is already reasonably strong; however, later iterations can propagate translation errors, amplify noise in the synthetic pairs, and progressively move the training signal away from the highest-quality authentic data. In addition, because the pretrained Helsinki-NLP models already provide a strong starting point and the corpus is relatively homogeneous, the margin for improvement is limited, so extra synthetic rounds do not necessarily yield better generalization. Therefore, the main conclusion here is that iterative back-translation can be beneficial, but its usefulness is **selective** rather than **monotonic**.
